# Setup

In [ ]:
from __future__ import annotations

# Standard library
import gc, hashlib, html, json, math, os, platform, random, re
import shutil, sys, tempfile, time, traceback, warnings
from collections.abc import Callable, Generator, Iterable
from contextlib import contextmanager
from datetime import datetime, timezone
from functools import partial
from importlib.metadata import version
from numbers import Real
from pathlib import Path
from typing import Any, Literal
from urllib.request import Request, urlopen

# Data and statistics
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.special import expit
from scipy.stats import norm, t as student_t

# Visualization and notebook display
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, Markdown, display
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont

# Classical machine learning
import sklearn
from econml.dml import CausalForestDML
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

# Foundation models and deep learning
import einops
import networkx as nx
import torch
import torch.nn.functional as F
from causalpfn import CATEEstimator
from tabpfn.model.encoders import LinearInputEncoderStep, NanHandlingEncoderStep, SequentialEncoder
from tabpfn.model.layer import PerFeatureEncoderLayer
from torch import nn
from torch.utils.checkpoint import checkpoint

This collection of lines imports numerous Python libraries commonly used in data science, machine learning, and statistical modeling. 

The first section brings in core functionalities from the Python standard library. These include modules for file system operations (`os`, `shutil`, `tempfile`), handling dates and times (`datetime`), working with text (`html`), managing processes (`gc`), and more. The inclusion of `typing` allows for more precise type hinting, improving code readability and maintainability.

Next, the script imports libraries essential for data manipulation and statistical analysis: `numpy` for numerical operations, `pandas` for dataframes, `scipy` for scientific computing, and `statsmodels` for statistical modeling. Specific functions from `scipy`, like optimization tools (`Bounds`, `LinearConstraint`, `milp`) and special functions (`expit`), are also imported.

Visualization libraries follow: `matplotlib` and `seaborn` provide plotting capabilities, while modules from `IPython.display` enable rich output in notebook environments. The `PIL` (Pillow) library is included for image processing. 

The code then imports several classical machine learning tools from the `sklearn` library, covering tasks like regression (`LinearRegression`, `GradientBoostingRegressor`), classification (`LogisticRegression`), model selection (`train_test_split`, `StratifiedKFold`), and preprocessing (`StandardScaler`, `OneHotEncoder`). It also includes `econml` for causal inference.

Finally, the script imports libraries related to foundation models and deep learning. These include `torch` (PyTorch) for tensor operations and neural networks, `einops` for tensor manipulation, `networkx` for graph analysis, and specialized modules from `causalpfn` and `tabpfn` designed for causal inference with tabular data using deep learning techniques. The inclusion of `torch.utils.checkpoint` suggests the code may deal with large models where memory efficiency is important. 



The overall purpose of this import block is to assemble a comprehensive toolkit for building, training, evaluating, and interpreting machine learning models, particularly those focused on causal inference and potentially involving deep learning architectures applied to tabular data.

In [ ]:
class CFG:
    # Paths, execution mode, and hardware
    ROOT = Path(os.environ.get('CAUSAL_NOTEBOOK_ROOT', Path.cwd())).expanduser().resolve()
    if 'CAUSAL_NOTEBOOK_ROOT' not in os.environ and (ROOT / 'causalfm' / 'causal_foundation_models.ipynb').is_file():
        ROOT = ROOT / 'causalfm'
    MODE = os.environ.get('CAUSAL_DEMO_MODE', 'demo')
    REUSE_RESULTS = os.environ.get('CAUSAL_REUSE_RESULTS', '0') == '1'
    DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
    OUT = ROOT / 'artifacts' / MODE
    GRAPH_DIR = ROOT / 'graphs'
    EXPORT_PREFIX = 'cfm_'
    RAW = ROOT / 'data' / 'raw'
    SEED = 42
    THREADS = 4
    CAUSALFM_BATCH_SIZE = 128
    CAUSALPFN_BATCH_SIZE = 256

    # Plotting and display
    PLOT_STYLE = 'seaborn-v0_8'
    FIGSIZE = (10, 4.8)
    FIGURE_DPI = 110
    DISPLAY_MAX_COLUMNS = 20
    DISPLAY_PRECISION = 3

    # Prespecified business assumptions and analysis settings
    CONTACT_COST = 0.10
    MARGIN = 0.40
    PRIMARY_BUDGET_FRACTION = 0.20
    BUDGET_FRACTIONS = (0., .05, .10, .20, .40, .60, 1.)
    CONTACT_COST_GRID = (0.01, 1.0, 35)
    SELECTION_STRENGTH = 1.0
    DR_N = 1500
    SENSITIVITY_N = 1200
    SENSITIVITY_SEED = 120
    FORMAL_N = 1500
    FORMAL_SEED = 81
    INTERFERENCE_SPILLOVERS = (0., 1.5, -1.)
    INTERFERENCE_INTERACTION = 0.0
    INTERFERENCE_CLUSTER_SIZE = 4
    INTERFERENCE_SATURATIONS = (0.15, 0.5, 0.85)

    # Demo and reference workloads: edit here before running comparisons.
    WORKLOADS = {
        'demo': {
            'accuracy_seeds': (11, 29), 'accuracy_sample_sizes': (256, 512),
            'accuracy_mechanisms': ('linear', 'nonlinear'), 'ihdp_reps': 3, 'accuracy_query_n': 512,
            'marketing_train_n': 512, 'marketing_test_n': 2500, 'marketing_val_n': 1500,
            'dr_repetitions': 30, 'sensitivity_repetitions': 12, 'interference_clusters': 500,
            'stress_train_n': 384, 'stress_query_n': 96, 'stress_repetitions': 2,
            'interval_query_n': 32, 'interval_samples': 500,
        },
        'reference': {
            'accuracy_seeds': tuple(range(11, 31)), 'accuracy_sample_sizes': (256, 512, 1024),
            'accuracy_mechanisms': ('linear', 'nonlinear', 'threshold'), 'ihdp_reps': 20, 'accuracy_query_n': 1024,
            'marketing_train_n': 1024, 'marketing_test_n': None, 'marketing_val_n': None,
            'dr_repetitions': 150, 'sensitivity_repetitions': 100, 'interference_clusters': 2500,
            'stress_train_n': 1024, 'stress_query_n': 256, 'stress_repetitions': 20,
            'interval_query_n': 64, 'interval_samples': 2000,
        },
    }

    # Download locations, feature definitions, model constants, and plot colors
    COLORS = {'CausalFM': '#b85a18', 'CausalPFN': '#6750a4',
              'Interaction OLS': '#607d8b', 'S-learner': '#548a9a',
              'T-learner': '#1e8573', 'X-learner': '#709339',
              'DR-learner': '#2b598b', 'Causal forest': '#a34666'}

    MODEL_ASSETS = {'CausalFM': {'path': 'vendor/CausalFM-toolkit/checkpoints/checkpoints_standard/best_model.pth',
                  'download_url': 'https://raw.githubusercontent.com/yccm/CausalFM-toolkit/bb74ef729c70d274fe2bb422c3b0edff4754fa37/checkpoints/checkpoints_standard/best_model.pth'},
     'CausalPFN': {'path': 'models/causalpfn_v0.pt',
                   'download_url': 'https://huggingface.co/vdblm/causalpfn/resolve/83aad07da1cb077cfda4236878a1b07dc9f72a54/causalpfn_v0.pt'}}

    DATA_ASSETS = [{'filename': 'ihdp_npci_1-100.train.npz',
      'download_url': 'https://www.fredjo.com/files/ihdp_npci_1-100.train.npz'},
     {'filename': 'ihdp_npci_1-100.test.npz',
      'download_url': 'https://www.fredjo.com/files/ihdp_npci_1-100.test.npz'},
     {'filename': 'hillstrom.csv.gz',
      'download_url': 'https://hillstorm1.s3.us-east-2.amazonaws.com/hillstorm_no_indices.csv.gz'}]

    HILLSTROM_NUMERIC = ["recency", "history", "mens", "womens", "newbie"]

    HILLSTROM_CATEGORICAL = ["history_segment", "zip_code", "channel"]

    HILLSTROM_FEATURES = HILLSTROM_NUMERIC + HILLSTROM_CATEGORICAL

    DEFAULT_EMSIZE = 128

    EXPOSURES = ((0, 0), (0, 1), (1, 0), (1, 1))

    CONTRASTS = {
        "Direct, no treated peer": {(1, 0): 1.0, (0, 0): -1.0},
        "Direct, any treated peer": {(1, 1): 1.0, (0, 1): -1.0},
        "Spillover on untreated": {(0, 1): 1.0, (0, 0): -1.0},
        "Spillover on treated": {(1, 1): 1.0, (1, 0): -1.0},
        "Total (1,1) versus (0,0)": {(1, 1): 1.0, (0, 0): -1.0},
    }


assert CFG.MODE in CFG.WORKLOADS
CFG.ROOT.mkdir(parents=True, exist_ok=True)
CFG.OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CFG.ROOT / '.cache/matplotlib'))
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ.setdefault(variable, str(CFG.THREADS))
torch.set_num_threads(CFG.THREADS)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
plt.style.use(CFG.PLOT_STYLE)
plt.rcParams.update({'figure.figsize': CFG.FIGSIZE, 'figure.dpi': CFG.FIGURE_DPI,
                     'savefig.bbox': 'tight'})
pd.set_option('display.max_columns', CFG.DISPLAY_MAX_COLUMNS)
pd.set_option('display.precision', CFG.DISPLAY_PRECISION)
pd.set_option('display.float_format', lambda value: f'{value:.{CFG.DISPLAY_PRECISION}f}')
warnings.filterwarnings('ignore', category=FutureWarning, module='econml')
print(f'Foundation models use {CFG.DEVICE}; classical estimators and data processing use CPU.')

`CFG` holds paths, seeds, device selection, workload sizes, business assumptions, and display settings.
Change these settings before running the notebook from the top.

As in the transformers notebook, `CFG.DEVICE` selects Apple Silicon **MPS** when available and CPU otherwise.
Both foundation models place their neural networks and inference tensors on that device. NumPy/pandas,
scikit-learn/EconML, SciPy optimization, and CausalPFN's scikit-learn retrieval model run on CPU.
GPU work is synchronized when timing model calls; cached results must match the configured device.

# Utils

## Device and array helpers

In [ ]:
def synchronize_device(device):
    """Wait for accelerator work before recording elapsed wall-clock time."""
    device = torch.device(device)
    if device.type == 'mps':
        torch.mps.synchronize()
    elif device.type == 'cuda':
        torch.cuda.synchronize(device)


@contextmanager
def isolate_torch_rng(seed: int, device: torch.device) -> Generator[None, None, None]:
    """Seed and restore the CPU and active accelerator RNGs for positional embeddings."""
    device = torch.device(device)
    cpu_state = torch.get_rng_state()
    accelerator_state = (torch.mps.get_rng_state() if device.type == 'mps' else
                         torch.cuda.get_rng_state(device) if device.type == 'cuda' else None)
    torch.random.default_generator.manual_seed(seed)
    if device.type == 'mps':
        torch.mps.manual_seed(seed)
    elif device.type == 'cuda':
        with torch.cuda.device(device):
            torch.cuda.manual_seed(seed)
    try:
        yield
    finally:
        torch.set_rng_state(cpu_state)
        if device.type == 'mps':
            torch.mps.set_rng_state(accelerator_state)
        elif device.type == 'cuda':
            torch.cuda.set_rng_state(accelerator_state, device)

This code provides utility functions for managing and synchronizing PyTorch devices and random number generators (RNGs).

The `synchronize_device` function ensures that all operations on a given PyTorch device (CPU, CUDA GPU, or MPS GPU) are completed before measuring elapsed time. This is crucial for accurate timing of computations, as asynchronous execution can lead to misleading results. It checks the device type and calls the appropriate synchronization function (`torch.mps.synchronize()` for MPS, `torch.cuda.synchronize()` for CUDA).

The `isolate_torch_rng` function is a context manager designed to isolate the random number generation state within a specific block of code. This ensures that any randomness generated during the execution of that block does not affect the global RNG state. It takes a seed value and a PyTorch device as input. 

Inside the context manager:
1.  It saves the current states of both the CPU and accelerator (MPS or CUDA) random number generators.
2.  It seeds the default CPU generator, and the accelerator’s generator with the provided `seed` value. It handles MPS and CUDA devices separately to use their respective seeding functions.
3.  The code within the `try...finally` block is executed.
4.  Upon exiting the context manager (in the `finally` block), it restores the CPU and accelerator RNG states to their original values, effectively isolating the randomness generated within the block.

This function is particularly useful when dealing with positional embeddings or any other part of a model that relies on random initialization, as it guarantees reproducibility by controlling the RNG state during those operations. The use of a context manager ensures that the RNG state is always restored, even if exceptions occur within the isolated code block.

In [ ]:
def _arrays(X, T, Y):
    X = np.asarray(X, dtype=np.float32)
    T = np.asarray(T, dtype=np.float32).reshape(-1)
    Y = np.asarray(Y, dtype=np.float32).reshape(-1)
    if X.ndim != 2 or not len(X) == len(T) == len(Y):
        raise ValueError("X, T and Y must have matching rows")
    if not np.all(np.isfinite(X)) or not np.all(np.isfinite(Y)):
        raise ValueError("Impute missing values inside the training pipeline first")
    if set(np.unique(T)) != {0, 1}:
        raise ValueError("Both binary treatment arms are required")
    return X, T, Y

This function, named `_arrays`, prepares input data for a causal inference model by converting it into NumPy arrays and performing several validation checks.

First, the function converts the inputs `X` (features), `T` (treatment assignment), and `Y` (outcome) into NumPy arrays with a data type of 32-bit floating point numbers (`np.float32`). The treatment variable `T` and outcome variable `Y` are reshaped into one-dimensional arrays using `.reshape(-1)`.

Next, it validates the input data:
1.  It checks if `X` is a two-dimensional array and if all three inputs (`X`, `T`, and `Y`) have the same number of rows (samples). If not, it raises a ValueError.
2.  It verifies that there are no missing or infinite values in `X` and `Y`. It raises a ValueError if any such values are found, indicating that missing value imputation should be performed before calling this function.
3.  It ensures that the treatment variable `T` contains only the values 0 and 1, representing binary treatment arms. If not, it raises a ValueError.

Finally, if all validation checks pass, the function returns the prepared NumPy arrays `X`, `T`, and `Y`. This function essentially acts as a data cleaning and preprocessing step to ensure that the input data is in the correct format and meets the requirements of the downstream causal inference model.

## Tables, plots, and exports

In [ ]:
# Mutable export bookkeeping, initialized separately from configuration.
_EXPORT_STATES = {}

def _write_manifest(state):
    path = state['directory'] / f'{CFG.EXPORT_PREFIX}manifest.json'
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps({
        'description': 'PNG exports in notebook display order; tables use three decimal places.',
        'entries': state['entries'],
    }, indent=2) + '\n')
    temporary.replace(path)
    links = []
    for entry in state['entries']:
        filename = html.escape(entry['filename'], quote=True)
        caption = html.escape(entry['display_title'])
        links.append(f'<article id="export-{entry["number"]:03d}"><h2>{caption}</h2>'
                     f'<a href="{filename}">Open full-size PNG</a>'
                     f'<a href="{filename}"><img loading="lazy" src="{filename}" alt="{caption}"></a></article>')
    (state['directory'] / f'{CFG.EXPORT_PREFIX}index.html').write_text(
        '<!doctype html><html lang="en"><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width, initial-scale=1">'
        '<title>Notebook PNG exports</title><style>'
        'body{font:16px system-ui,sans-serif;margin:32px auto;padding:0 24px;max-width:1500px;color:#17324d}'
        'article{margin:36px 0;padding:20px;border:1px solid #d8e1e8;border-radius:8px}'
        'h2{font-size:19px}img{display:block;max-width:100%;height:auto;margin-top:14px}'
        'a{color:#245b8d}</style><h1>Notebook PNG exports</h1>'
        '<p>Graphs and table pages in notebook order. Open a PNG to inspect it at full resolution.</p>'
        + ''.join(links) + '</html>'
    )


def begin_exports(directory):
    """Start one numbered sequence for every displayed chart and table page.

    Only prefixed PNG files recorded in this notebook's previous manifest are removed. Other files in
    the export folder are left alone, including manually added images.
    """
    path = Path(directory).resolve()
    path.mkdir(parents=True, exist_ok=True)
    manifest = path / f'{CFG.EXPORT_PREFIX}manifest.json'
    if manifest.exists():
        previous = json.loads(manifest.read_text())
        for entry in previous.get('entries', []):
            filename = entry.get('filename', '')
            if filename and Path(filename).name == filename and filename.startswith(CFG.EXPORT_PREFIX) and filename.endswith('.png'):
                (path / filename).unlink(missing_ok=True)
    state = {'directory': path, 'entries': []}
    _EXPORT_STATES[str(path)] = state
    _write_manifest(state)


def _export_state(directory):
    path = Path(directory).resolve()
    if str(path) not in _EXPORT_STATES:
        # Calling begin_exports explicitly is the reproducible notebook path.
        # Standalone use appends safely rather than silently replacing exports.
        path.mkdir(parents=True, exist_ok=True)
        manifest = path / f'{CFG.EXPORT_PREFIX}manifest.json'
        entries = json.loads(manifest.read_text()).get('entries', []) if manifest.exists() else []
        _EXPORT_STATES[str(path)] = {'directory': path, 'entries': entries}
    return _EXPORT_STATES[str(path)]


def _export_details(name, directory, kind, title):
    state = _export_state(directory)
    number = len(state['entries']) + 1
    slug = re.sub(r'[^a-z0-9]+', '_', str(name).lower()).strip('_') or kind
    filename = f'{CFG.EXPORT_PREFIX}{number:03d}_{kind}_{slug}.png'
    return state, state['directory'] / filename, {
        'number': number, 'type': kind, 'name': str(name), 'title': title,
        'display_title': f'{number:03d} — {title}', 'filename': filename,
    }

This code manages the export of figures and tables (primarily as PNG images) from a notebook environment. It provides functions to initialize an export directory, track exported files, generate manifest files, and create HTML index pages for browsing the exports.

The `_EXPORT_STATES` dictionary is a global variable used to store the state of each export session, keyed by the path to the export directory.

The `_write_manifest` function creates or updates a JSON manifest file within an export directory. This manifest contains a description and a list of exported entries (filename, display title, etc.). It first writes the data to a temporary file with a `.tmp` extension, then atomically replaces the original manifest file with the temporary one, ensuring data integrity.  It also generates HTML links for each entry that can be included in an index page.

The `begin_exports` function initializes a new export session within a specified directory. It creates the directory if it doesn't exist and removes any previously exported PNG files (identified by a prefix defined in `CFG.EXPORT_PREFIX`) from that directory to avoid conflicts with previous runs.  It then initializes a state dictionary containing the directory path and an empty list of entries, stores this state in `_EXPORT_STATES`, and writes an initial manifest file.

The `_export_state` function retrieves or creates the export state for a given directory. If a state already exists for that directory in `_EXPORT_STATES`, it returns it. Otherwise, it attempts to load the state from the existing manifest file (if present). If no manifest file is found, it creates a new state with an empty list of entries and stores it in `_EXPORT_STATES`. This function ensures that export information is consistently tracked across multiple calls.

The `_export_details` function prepares details for exporting a single figure or table. It retrieves the export state for the specified directory (creating one if necessary), generates a unique filename based on an incrementing number, the type of export (`kind`), and a sanitized version of the name provided.  It returns the export state, the full path to the intended output file, and a dictionary containing metadata about the exported entry (number, type, name, title, display title, filename). This function centralizes the logic for naming and tracking individual exports.



In essence, this code provides a robust system for managing and organizing exported content from a notebook, ensuring reproducibility and easy access to results. It handles file management, metadata tracking, and HTML index generation.

In [ ]:
def _record_export(state, entry):
    state['entries'].append(entry)
    _write_manifest(state)


def _format_value(value):
    """Format display strings without rounding or mutating the source data."""
    if isinstance(value, (bool, np.bool_)):
        return str(bool(value))
    if isinstance(value, Real):
        return f'{value:.3f}'
    if value is None or value is pd.NA or value is pd.NaT:
        return '—'
    return str(value)


def _column_label(value):
    if isinstance(value, tuple):
        return ' / '.join(str(part) for part in value if str(part))
    return str(value)


def _table_content(frame, include_index):
    if isinstance(frame, pd.Series):
        frame = frame.to_frame(name=frame.name if frame.name is not None else 'value')
    if not isinstance(frame, pd.DataFrame):
        frame = pd.DataFrame(frame)
    if include_index is None:
        index = frame.index
        include_index = not (
            isinstance(index, pd.RangeIndex) and index.start == 0 and index.step == 1
            and index.name is None
        )
    headers = [_column_label(column) for column in frame.columns]
    index_levels = frame.index.nlevels if include_index else 0
    if include_index:
        index_names = [name if name is not None else ('index' if index_levels == 1 else f'index_{i + 1}')
                       for i, name in enumerate(frame.index.names)]
        headers = [str(name) for name in index_names] + headers
    values = []
    alignments = []
    for index_value, row in zip(frame.index, frame.itertuples(index=False, name=None)):
        row = list(row)
        if include_index:
            index_values = list(index_value) if isinstance(frame.index, pd.MultiIndex) else [index_value]
            row = index_values + row
        values.append([_format_value(value) for value in row])
        alignments.append(['right' if isinstance(value, Real) and not isinstance(value, (bool, np.bool_))
                           else 'left' for value in row])
    return frame, headers, values, alignments, index_levels


def _wrap_text(text, font, available_width):
    """Wrap even long URLs/hash strings without truncating their contents."""
    lines = []
    for paragraph in str(text).split('\n'):
        if not paragraph:
            lines.append('')
            continue
        line = ''
        for word in paragraph.split(' '):
            candidate = f'{line} {word}' if line else word
            if font.getlength(candidate) <= available_width:
                line = candidate
                continue
            if line:
                lines.append(line)
                line = ''
            while font.getlength(word) > available_width:
                cut = 1
                while cut < len(word) and font.getlength(word[:cut + 1]) <= available_width:
                    cut += 1
                lines.append(word[:cut])
                word = word[cut:]
            line = word
        lines.append(line)
    return lines or ['']


def _render_table_png(headers, values, alignments, title, path):
    regular = ImageFont.truetype(font_manager.findfont('DejaVu Sans'), 16)
    bold = ImageFont.truetype(font_manager.findfont(font_manager.FontProperties(
        family='DejaVu Sans', weight='bold')), 16)
    padding, line_height, margin = 12, 23, 22
    widths = []
    for j, header in enumerate(headers):
        measured = [bold.getlength(line) for line in header.split('\n')]
        measured.extend(regular.getlength(line) for row in values for line in row[j].split('\n'))
        widths.append(math.ceil(min(280, max(92, max(measured, default=0) + 2 * padding))))
    if not headers:
        widths = [160]
        headers = ['Empty table']
        values = [['No columns']] if values else []
        alignments = [['left']] if values else []
    header_lines = [_wrap_text(header, bold, width - 2 * padding)
                    for header, width in zip(headers, widths)]
    body_lines = [[_wrap_text(value, regular, width - 2 * padding)
                   for value, width in zip(row, widths)] for row in values]
    header_height = max(map(len, header_lines), default=1) * line_height + 2 * padding
    row_heights = [max(map(len, row), default=1) * line_height + 2 * padding for row in body_lines]
    table_width = sum(widths)
    # PNGs contain only the table; captions remain in notebook HTML and the gallery.
    image_width = max(table_width + 2 * margin, 650)
    image = Image.new('RGB', (image_width, header_height + sum(row_heights) + 2 * margin), 'white')
    draw = ImageDraw.Draw(image)
    y = margin
    x = margin
    for width, lines in zip(widths, header_lines):
        draw.rectangle((x, y, x + width, y + header_height), fill='#17324d', outline='#d8e1e8')
        for line_number, line in enumerate(lines):
            draw.text((x + padding, y + padding + line_number * line_height), line, fill='white', font=bold)
        x += width
    y += header_height
    for row_number, (row, row_height) in enumerate(zip(body_lines, row_heights)):
        x = margin
        for column_number, (width, lines) in enumerate(zip(widths, row)):
            draw.rectangle((x, y, x + width, y + row_height),
                           fill='#f0f4f8' if row_number % 2 == 0 else 'white', outline='#d8e1e8')
            for line_number, line in enumerate(lines):
                left = x + padding
                if alignments[row_number][column_number] == 'right':
                    left = x + width - padding - regular.getlength(line)
                draw.text((left, y + padding + line_number * line_height), line, fill='#000000', font=regular)
            x += width
        y += row_height
    image.save(path, dpi=(150, 150))
    return image.size

This code provides functions for formatting data and rendering tables as PNG images. It's designed to create visually appealing and informative table exports from a notebook environment.

The `_record_export` function simply appends an entry (representing an exported item) to the state’s list of entries and then updates the manifest file with the new information.

The `_format_value` function formats values for display in tables, handling different data types gracefully. It converts booleans to strings ("True" or "False"), floats to strings with three decimal places, and replaces missing values (None, pd.NA, pd.NaT) with a dash ("—"). Other values are converted to their string representation.

The `_column_label` function formats column labels, handling tuples by joining the elements with " / ". It converts other values directly to strings.

The `_table_content` function prepares data for rendering as a table. It handles both Pandas Series and DataFrames, converting them to DataFrames if necessary. It determines whether to include the index based on the DataFrame’s index type. It then extracts headers, values, and alignments for each column, formatting the values using `_format_value`.

The `_wrap_text` function wraps long text strings (like URLs or hash strings) into multiple lines to fit within a specified width, preventing truncation. It splits the text into words and adds them to a line until the line exceeds the available width, then starts a new line.

The `_render_table_png` function is the core of the table rendering process. It takes headers, values, alignments, a title, and a path as input and generates a PNG image of the table. 

Here's a breakdown of its steps:
1.  It defines fonts (regular and bold) using `ImageFont`.
2.  It calculates column widths based on header and data content, ensuring minimum and maximum width constraints.
3.  It wraps text in headers and values to fit within the calculated column widths using `_wrap_text`.
4.  It determines row heights based on the number of lines in each row.
5.  It creates a new RGB image with appropriate dimensions.
6.  It draws rectangles for header and data cells, applying alternating background colors.
7.  It writes text into the cells, aligning values to the right for numeric columns and to the left for others.
8.  Finally, it saves the image as a PNG file at the specified path with a DPI of 150.

The function returns the size of the generated image. This code is designed to create high-quality table visualizations suitable for inclusion in reports or presentations.

In [ ]:
def _table_html(headers, values, alignments, title):
    head = ''.join(f'<th style="padding:8px 10px;text-align:left;background:#17324d;color:white;'
                   f'border:1px solid #d8e1e8;max-width:280px;overflow-wrap:anywhere">{html.escape(header)}</th>'
                   for header in headers)
    rows = []
    for i, (row, alignment) in enumerate(zip(values, alignments)):
        cells = ''.join(f'<td style="color:#000000;padding:7px 10px;text-align:{align};border:1px solid #d8e1e8;'
                        f'max-width:280px;min-width:70px;white-space:pre-wrap;overflow-wrap:anywhere">'
                        f'{html.escape(value)}</td>' for value, align in zip(row, alignment))
        rows.append(f'<tr style="background:{"#f0f4f8" if i % 2 == 0 else "white"}">{cells}</tr>')
    return (f'<div class="causal-export-table"><p style="font-weight:bold;color:#17324d">{html.escape(title)}</p>'
            f'<div style="overflow-x:auto"><table style="border-collapse:collapse;font-size:13px;'
            f'font-variant-numeric:tabular-nums"><thead><tr>{head}</tr></thead>'
            f'<tbody>{"".join(rows)}</tbody></table></div></div>')


def display_table(frame, name, directory, title=None, rows_per_page=25, include_index=None):
    """Display and export every table row/column with fixed three-decimal numbers.

    Tables paginate vertically; each visible page is its own sequentially named
    PNG. Meaningful index levels are included automatically. Underlying frames
    are never changed, and CSV/estimation precision is unaffected.
    """
    if rows_per_page < 1:
        raise ValueError('rows_per_page must be positive')
    frame, headers, values, alignments, index_levels = _table_content(frame, include_index)
    title = title or str(name).replace('_', ' ').capitalize()
    pages = max(1, math.ceil(len(values) / rows_per_page))
    for page in range(pages):
        start = page * rows_per_page
        stop = min(len(values), start + rows_per_page)
        page_title = title if pages == 1 else f'{title} (page {page + 1}/{pages}; rows {start + 1}–{stop})'
        page_name = name if pages == 1 else f'{name}_page_{page + 1:02d}'
        state, path, entry = _export_details(page_name, directory, 'table', page_title)
        dimensions = _render_table_png(headers, values[start:stop], alignments[start:stop], entry['display_title'], path)
        entry.update(rows=stop - start, columns=len(headers), data_columns=len(frame.columns),
                     index_levels=index_levels, page=page + 1, pages=pages,
                     row_start=start, row_stop=stop, total_rows=len(frame),
                     width=dimensions[0], height=dimensions[1])
        _record_export(state, entry)
        display(HTML(_table_html(headers, values[start:stop], alignments[start:stop], entry['display_title'])))




def finish_figure(name, directory):
    figure = plt.gcf()
    existing_title = figure._suptitle.get_text() if figure._suptitle is not None else ''
    if not existing_title and figure.axes:
        existing_title = figure.axes[0].get_title()
    title = existing_title or str(name).replace('_', ' ').capitalize()
    state, path, entry = _export_details(name, directory, 'figure', title)
    if figure._suptitle is not None:
        figure._suptitle.set_text(entry['display_title'])
    elif figure.axes:
        figure.axes[0].set_title(entry['display_title'])
    else:
        figure.suptitle(entry['display_title'])
    # Omit all axes titles and the figure title from the exported PNG only.
    # Keep labels, legends, annotations, filenames, and notebook captions intact.
    title_artists = [artist for axis in figure.axes
                     for artist in (axis.title, axis._left_title, axis._right_title)]
    if figure._suptitle is not None:
        title_artists.append(figure._suptitle)
    title_visibility = [artist.get_visible() for artist in title_artists]
    try:
        for artist in title_artists:
            artist.set_visible(False)
        plt.tight_layout()
        figure.savefig(path, dpi=150)
    finally:
        for artist, visible in zip(title_artists, title_visibility):
            artist.set_visible(visible)
        plt.tight_layout()
    with Image.open(path) as saved:
        entry.update(width=saved.width, height=saved.height)
    entry['graph_axes'] = sum(
        axis.get_label() != '<colorbar>' and not hasattr(axis, '_colorbar')
        for axis in figure.axes
    )
    _record_export(state, entry)
    plt.show()
    plt.close(figure)


def save_tables(bundle, prefix, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    for key, value in bundle.items():
        if isinstance(value, pd.DataFrame):
            value.to_csv(directory / f'{prefix}_{key}.csv', index=False)


def plot_intervals(frame, label, estimate, low, high, title, truth=None):
    frame = frame.reset_index(drop=True)
    y = np.arange(len(frame))
    plt.figure(figsize=(10, max(3.8, len(frame)*.38)))
    plt.errorbar(frame[estimate], y,
                 xerr=np.vstack([np.maximum(0,frame[estimate]-frame[low]),
                                 np.maximum(0,frame[high]-frame[estimate])]),
                 fmt='o', capsize=3, color='#2b598b')
    if truth is not None:
        if isinstance(truth, str):
            plt.scatter(frame[truth], y, marker='x', color='#b85a18', label='Simulator truth')
            plt.legend()
        else:
            plt.axvline(truth, ls='--', color='#b85a18')
    plt.yticks(y, frame[label])
    plt.title(title)
    plt.xlabel(estimate.replace('_', ' '))

This code provides functions for exporting tables and figures as PNG images and HTML representations from a notebook environment, along with utility functions for saving dataframes to CSV files and plotting interval estimates.

The `_table_html` function generates an HTML table string from headers, values, and alignments. It uses inline CSS styling for formatting and ensures proper escaping of HTML characters using `html.escape`.

The `display_table` function displays a Pandas DataFrame as both an interactive HTML table in the notebook and exports it as one or more PNG images, paginating the table if it has many rows. It calls `_table_content` to prepare the data, then iterates through pages of the table, exporting each page as a separate PNG image using `_render_table_png`.  It also generates HTML for displaying the current page in the notebook and records export metadata.

The `finish_figure` function exports a Matplotlib figure as a PNG image. It retrieves the figure title (or creates one if none exists), sets the display title on the figure, temporarily hides all titles from the exported image to avoid clutter, saves the figure as a PNG file using `plt.savefig`, restores the original visibility of the titles, and records export metadata.

The `save_tables` function saves a bundle of Pandas DataFrames to CSV files in a specified directory. It iterates through the bundle and writes each DataFrame to a separate CSV file with a prefix added to the filename.

Finally, the `plot_intervals` function creates a plot displaying interval estimates (e.g., confidence intervals) for a given variable. It uses error bars to represent the lower and upper bounds of the intervals and optionally includes a "truth" value for comparison. The function also labels the y-axis with values from a specified column in the input DataFrame and sets the title of the plot.



In summary, this code provides a comprehensive set of tools for exporting data visualizations and tables from a notebook environment, ensuring consistent formatting, pagination, and metadata tracking. It leverages both image rendering (PNG) and HTML generation to provide flexible output options.

In [ ]:
def json_default(value):
    if isinstance(value, (np.integer,np.floating)): return value.item()
    if isinstance(value,np.ndarray): return value.tolist()
    if isinstance(value,Path): return str(value)
    raise TypeError(type(value).__name__)


def save_json(value,path):
    Path(path).write_text(json.dumps(value,indent=2,default=json_default)+'\n')

This code provides functions for serializing Python objects to JSON format and saving them to a file.

The `json_default` function is a custom default handler used by the `json.dumps` function when it encounters an object that it doesn't know how to serialize natively. It handles three specific cases:

1.  **NumPy integers and floats:** Converts NumPy integer and floating-point numbers to their standard Python equivalents using `.item()`. This is necessary because the JSON format does not directly support NumPy numeric types.
2.  **NumPy arrays:** Converts NumPy arrays to lists using `.tolist()`. Again, this ensures compatibility with the JSON format.
3.  **Path objects:** Converts `pathlib.Path` objects to strings using `str(value)`. This allows file paths to be represented in the JSON output.

If the function encounters an object that it doesn't recognize (i.e., not a NumPy number, array, or Path), it raises a TypeError indicating the unsupported type.

The `save_json` function takes a Python value and a file path as input. It converts the value to a JSON string using `json.dumps`, specifying an indent of 2 for readability and passing the `json_default` function as the default handler. The resulting JSON string is then written to the specified file, followed by a newline character (`\n`).

In essence, this code provides a convenient way to serialize complex Python objects (including those from NumPy and pathlib) to JSON format while ensuring compatibility with standard JSON parsers.

## Asset downloads

In [ ]:
def ensure_asset(path, url):
    """Download a missing dataset or checkpoint; reuse existing files."""
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(dir=path.parent, suffix='.part', delete=False) as handle:
                temporary = Path(handle.name)
                print(f'Downloading {path.name} ...', flush=True)
                with urlopen(Request(url, headers={'User-Agent': 'causal-foundation-notebook'}), timeout=180) as response:
                    shutil.copyfileobj(response, handle)
            temporary.replace(path)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
    return path


def prepare_assets():
    for record in CFG.MODEL_ASSETS.values():
        ensure_asset(CFG.ROOT / record['path'], record['download_url'])
    for record in CFG.DATA_ASSETS:
        ensure_asset(CFG.ROOT / 'data/raw' / record['filename'], record['download_url'])

This code provides functions for downloading and ensuring the availability of necessary assets, such as datasets or pre-trained models, required by the project.

The `ensure_asset` function downloads a file from a given URL if it doesn’t already exist at the specified path. It first checks if the file exists; if not, it creates the parent directory (if it doesn't exist).  It then uses a temporary file to download the content in chunks, preventing incomplete files in case of interruptions.

Here's a breakdown:
1.  It constructs a `Path` object from the provided path string.
2.  If the file does not exist, it creates the parent directory using `path.parent.mkdir(parents=True, exist_ok=True)`.
3.  It creates a temporary file in the same directory as the target file with a `.part` suffix. This is done to ensure that an incomplete download doesn't leave a partially written file at the final destination.
4.  It opens a connection to the URL using `urllib.request.urlopen`, setting a User-Agent header for identification and a timeout of 180 seconds.
5.  It copies the content from the URL response to the temporary file using `shutil.copyfileobj`.
6.  Once the download is complete, it atomically replaces the original destination path with the temporary file using `temporary.replace(path)`. This ensures that the file is fully written before being made available.
7.  A `finally` block ensures that the temporary file is deleted even if an error occurs during the download or replacement process.

The `prepare_assets` function iterates through lists of model assets and data assets defined in a configuration object (`CFG`). For each asset, it calls `ensure_asset` to download the file from its URL to the appropriate location within the project’s directory structure. This ensures that all required files are present before proceeding with the analysis.



In essence, this code provides a robust mechanism for managing and downloading dependencies, ensuring that the necessary data and models are available for the project to run correctly. It handles potential download errors gracefully and uses temporary files to prevent incomplete downloads from corrupting the final assets.

## Data preparation helpers

In [ ]:
"""
Potential-outcome truth is returned separately from X for simulation evaluation;
it must never be passed to a learner or used for real-data model selection.
"""


def simulate_individual(n, seed, scenario="nonlinear", overlap=1.0, hidden=False):
    """Generate known CATE truth; larger overlap parameter means worse overlap.

    Hidden=True introduces an omitted common cause U. Returned e is then the
    oracle propensity conditional on U, not an identifiable propensity from X.
    """
    if n < 2 or overlap < 0 or scenario not in {"linear", "nonlinear", "threshold"}:
        raise ValueError("Require n>=2, overlap>=0, and linear/nonlinear/threshold scenario")
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(int(n), 8))
    X[:, 6:] = rng.binomial(1, 0.5, size=(n, 2))
    latent = rng.normal(size=n)
    logit = overlap * (0.7 * X[:, 0] - 0.6 * X[:, 1] + 0.4 * X[:, 2]
                       + (1.2 * latent if hidden else 0.0))
    e = expit(logit)
    mu0 = 1.5 + X[:, 0] + 0.5 * X[:, 1] - 0.3 * X[:, 2]
    tau = 0.7 + 0.6 * X[:, 0] - 0.5 * X[:, 6]
    if scenario == "nonlinear":
        mu0 += 0.7 * np.sin(X[:, 2]) + 0.3 * X[:, 3] ** 2
        tau += 0.8 * np.tanh(X[:, 1] * X[:, 2]) + 0.4 * X[:, 7]
    elif scenario == "threshold":
        mu0 += 1.2 * (X[:, 2] > 0) + 0.5 * X[:, 3] * X[:, 4]
        tau = -0.4 + 1.8 * ((X[:, 0] > 0) & (X[:, 1] < 0.5)) - 0.5 * X[:, 6]
    if hidden:
        mu0 += 1.2 * latent
    mu1 = mu0 + tau
    T = rng.binomial(1, e).astype(int)
    Y = mu0 + T * tau + rng.normal(scale=1.0, size=n)
    return {"X": X, "T": T, "Y": Y, "tau": tau, "mu0": mu0,
            "mu1": mu1, "e": e, "id": np.arange(n),
            "meta": {"synthetic": True, "scenario": scenario,
                     "hidden_confounding": bool(hidden),
                     "e_conditions_on_hidden_U": bool(hidden)}}


def load_ihdp(replication=0):
    """Load one author-distributed IHDP-100 realization and original split."""
    result = {}
    n_replications = None
    for name, offset in [("train", 0), ("test", 10000)]:
        path = CFG.RAW / f"ihdp_npci_1-100.{name}.npz"
        if not path.exists():
            raise FileNotFoundError(f"Missing {path}; run the asset preparation cell")
        with np.load(path, allow_pickle=False) as archive:
            n_replications = archive["x"].shape[-1]
            if not 0 <= replication < n_replications:
                raise ValueError(f"replication must be in [0,{n_replications})")
            X = np.asarray(archive["x"][:, :, replication], dtype=float)
            T = np.asarray(archive["t"][:, replication], dtype=int).reshape(-1)
            Y = np.asarray(archive["yf"][:, replication], dtype=float).reshape(-1)
            tau = np.asarray(archive["mu1"][:, replication] - archive["mu0"][:, replication],
                             dtype=float).reshape(-1)
        result[name] = {"X": X, "T": T, "Y": Y, "tau": tau,
                        "id": np.arange(len(T)) + offset + replication * 100000}
    result["feature_names"] = [f"x{i}" for i in range(result["train"]["X"].shape[1])]
    result["meta"] = {"dataset": "IHDP-100", "replication": replication,
                      "n_replications": n_replications, "semi_synthetic": True,
                      "truth": "mu1-mu0", "original_split": True,
                      "ids": "Local split-row identifiers, not original participant identifiers",
                      "source": "https://www.fredjo.com/"}
    return result


def load_hillstrom():
    """Read and validate all three randomized arms without modifying raw data."""
    candidates = [CFG.RAW / "hillstrom.csv.gz", CFG.RAW / "hillstrom.csv"]
    path = next((path for path in candidates if path.exists()), None)
    if path is None:
        raise FileNotFoundError("Missing Hillstrom data; run the asset preparation cell")
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.lower()
    expected = set(CFG.HILLSTROM_FEATURES + ["segment", "visit", "conversion", "spend"])
    if set(frame.columns) != expected or len(frame) != 64000:
        raise ValueError(f"Unexpected Hillstrom schema/rows: {frame.shape}, {frame.columns.tolist()}")
    if frame.isna().any().any():
        raise ValueError("Unexpected missing values in source Hillstrom data")
    if set(frame["segment"]) != {"Mens E-Mail", "Womens E-Mail", "No E-Mail"}:
        raise ValueError("Unexpected randomized arm labels")
    for column in ["visit", "conversion"]:
        if not set(frame[column]).issubset({0, 1}):
            raise ValueError(f"{column} is not binary")
    if (frame["spend"] < 0).any() or (frame["conversion"] > frame["visit"]).any():
        raise ValueError("Inconsistent Hillstrom outcomes")
    return frame

This code provides functions for simulating causal data and loading real-world datasets used in causal inference.

The `simulate_individual` function generates synthetic data with known causal effects, allowing for evaluation of causal estimation methods. It takes parameters to control the sample size (`n`), random seed (`seed`), scenario (linear, nonlinear, or threshold), overlap between treatment and control groups (`overlap`), and whether to include a hidden confounder (`hidden`). The function creates features `X`, generates a propensity score `e` based on these features (and potentially a latent variable if `hidden` is True), defines potential outcomes `mu0` and `mu1`, assigns treatments `T` based on the propensity score, and calculates observed outcomes `Y`. It returns a dictionary containing the generated data, true causal effects (`tau`), and metadata about the simulation.

The `load_ihdp` function loads one realization of the IHDP (Individualized Heterogeneous Treatment Effect Prediction) dataset, a benchmark dataset for causal inference. It takes a replication index as input to specify which realization to load. The function checks if the necessary files exist, loads the data using NumPy, and returns a dictionary containing the features `X`, treatment assignments `T`, outcomes `Y`, true individual treatment effects (`tau`), and metadata about the dataset.

The `load_hillstrom` function loads the Hillstrom customer marketing dataset, which is commonly used for causal inference in marketing research. It checks if the data file exists, reads it into a Pandas DataFrame, validates the schema (column names and number of rows), ensures there are no missing values, verifies that the randomized arms have expected labels, and confirms consistency of outcomes. The function returns the validated DataFrame.

In summary, this code provides tools for generating synthetic causal data with known ground truth and loading real-world datasets commonly used in causal inference research. These functions are essential for evaluating and comparing different causal estimation methods.

In [ ]:
def prepare_hillstrom(seed=CFG.SEED):
    """Men's email vs no email; 50/20/30 split and train-only preprocessing.

    The effect is assignment-to-email (ITT); Y is two-week spend. No outcome,
    treatment, ID, or potential-outcome column enters X. No costs are observed.
    """
    frame = load_hillstrom()
    frame = frame.loc[frame["segment"].isin(["Mens E-Mail", "No E-Mail"])].copy()
    frame["T"] = (frame["segment"] == "Mens E-Mail").astype(int)
    train, rest = train_test_split(frame, train_size=0.5, random_state=seed, stratify=frame["T"])
    valid, test = train_test_split(rest, train_size=0.4, random_state=seed + 1,
                                  stratify=rest["T"])
    encoder = ColumnTransformer([
        ("numeric", StandardScaler(), CFG.HILLSTROM_NUMERIC),
        ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         CFG.HILLSTROM_CATEGORICAL),
    ], remainder="drop", sparse_threshold=0)
    encoder.fit(train[CFG.HILLSTROM_FEATURES])
    result = {}
    for name, rows in [("train", train), ("valid", valid), ("test", test)]:
        result[name] = {"X": np.asarray(encoder.transform(rows[CFG.HILLSTROM_FEATURES]), dtype=float),
                        "T": rows["T"].to_numpy(dtype=int),
                        "Y": rows["spend"].to_numpy(dtype=float),
                        "visit": rows["visit"].to_numpy(dtype=int),
                        "conversion": rows["conversion"].to_numpy(dtype=int),
                        "id": rows.index.to_numpy(dtype=int),
                        "e": np.full(len(rows), 0.5)}
    result["feature_names"] = encoder.get_feature_names_out().tolist()
    result["meta"] = {"dataset": "Hillstrom", "treatment": "Mens E-Mail",
                      "control": "No E-Mail", "outcome": "two-week spend",
                      "randomization_probability": 0.5, "split_seed": seed,
                      "split_fractions": {"train": 0.5, "valid": 0.2, "test": 0.3},
                      "preprocessing_fitted_on": "train only",
                      "raw_features": CFG.HILLSTROM_FEATURES,
                      "numeric_training_mean": encoder.named_transformers_["numeric"].mean_.tolist(),
                      "costs_observed": False, "individual_effect_truth_observed": False,
                      "source": "https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html"}
    return result


def select_observational(dataset, strength=1.0, seed=CFG.SEED, p=0.5):
    """Select factual RCT records to induce q(X) while preserving the X law.

    s1=k*q/p and s0=k*(1-q)/(1-p), with k=min(p,1-p), give
    P(S=1|X)=k and P(T=1|X,S=1)=q. The source must have constant randomized
    propensity p. Selection uses X and T, never Y; no treatment is relabeled.
    Returned e is q, known for this constructed observational experiment.
    """
    if not 0 < p < 1 or strength < 0:
        raise ValueError("Require 0<p<1 and nonnegative strength")
    X, T = np.asarray(dataset["X"]), np.asarray(dataset["T"])
    if len(X) != len(T) or X.ndim != 2 or X.shape[1] < 2:
        raise ValueError("Require matching X/T rows and at least two covariates")
    if "e" in dataset and not np.allclose(dataset["e"], p):
        raise ValueError("Source must be an RCT with constant propensity p")
    standardized = (X[:, :4] - X[:, :4].mean(axis=0)) / np.maximum(X[:, :4].std(axis=0), 1e-8)
    coeff = np.array([0.8, -0.6, 0.4, -0.3])[:standardized.shape[1]]
    q = np.clip(expit(strength * (standardized @ coeff)), 0.025, 0.975)
    k = min(p, 1 - p)
    selection_probability = np.where(T == 1, k * q / p, k * (1 - q) / (1 - p))
    keep = np.random.default_rng(seed).uniform(size=len(T)) < selection_probability
    result = {key: (value[keep].copy() if isinstance(value, np.ndarray)
                    and value.ndim > 0 and len(value) == len(T) else value)
              for key, value in dataset.items()}
    result["e"] = q[keep]
    result["selection_probability"] = selection_probability[keep]
    result["selection_meta"] = {"strength": float(strength), "seed": seed,
                                "source_n": len(T), "selected_n": int(keep.sum()),
                                "source_p": p, "expected_acceptance": k,
                                "uses_outcome": False, "target_X_law_preserved": True,
                                "propensity_is_known_by_construction": True}
    return result

This code provides functions for preparing and manipulating datasets for causal inference experiments.

The `prepare_hillstrom` function preprocesses the Hillstrom dataset for a specific causal inference task: estimating the effect of sending an email on customer spending. It filters the data to include only customers in the "Mens E-Mail" and "No E-Mail" segments, creates a binary treatment variable `T` indicating whether a customer received the email, splits the data into training, validation, and test sets using stratified sampling based on the treatment assignment, and applies feature scaling (StandardScaler for numeric features) and one-hot encoding for categorical features. The function returns a dictionary containing the preprocessed data for each split, along with metadata about the dataset and preprocessing steps.

The `select_observational` function simulates an observational study from a randomized controlled trial (RCT) by selectively including records based on a propensity score. This allows researchers to investigate the performance of causal inference methods in settings where treatment assignment is not fully random. The function takes a dataset assumed to be from an RCT with constant propensity `p`, a strength parameter controlling the magnitude of the induced confounding, and a seed for reproducibility. It calculates a propensity score `q` based on covariates using a logistic function, then selects records based on a probability that depends on both the true treatment assignment `T` and the calculated propensity score `q`. The function returns a new dataset containing only the selected records, along with the calculated propensity scores `e`, selection probabilities, and metadata about the selection process.

In summary, this code provides tools for preparing real-world datasets (Hillstrom) and simulating observational data from RCTs, enabling researchers to evaluate causal inference methods in various settings. The functions handle data preprocessing, splitting, feature engineering, and the creation of realistic confounding scenarios.

## Effect scores and policy evaluation

In [ ]:
"""Common orthogonal evaluation, diagnostics, and constrained treatment policies.

All policies must be fixed independently of the outcomes passed to these helpers.
Confidence intervals describe evaluation-sample uncertainty for a frozen policy;
they do not include refitting, policy selection, multiple comparisons, or spillovers.
"""




def _vector(value, n, name):
    result = np.broadcast_to(np.asarray(value, dtype=float), (n,)).copy()
    if not np.isfinite(result).all():
        raise ValueError(f"{name} must contain only finite values")
    return result


def _inputs(X, T, Y):
    X, T, Y = np.asarray(X, dtype=float), np.asarray(T), np.asarray(Y, dtype=float)
    if X.ndim != 2 or T.shape != (len(X),) or Y.shape != (len(X),):
        raise ValueError("Expected X (n,p), T (n,), Y (n,)")
    if not np.isfinite(X).all() or not np.isfinite(Y).all():
        raise ValueError("Impute missing values using training data before fitting")
    if not np.isin(T, [0, 1]).all() or len(np.unique(T)) != 2:
        raise ValueError("Both binary treatment arms 0 and 1 are required")
    return X, T.astype(int), Y


def outcome_model(seed=CFG.SEED):
    """Small, deliberately fixed nuisance specification for a reproducible demo."""
    return GradientBoostingRegressor(
        n_estimators=70, max_depth=2, min_samples_leaf=12,
        learning_rate=0.06, loss="squared_error", random_state=seed,
    )


def propensity_model(seed=CFG.SEED):
    # Quadratic features permit more than linear confounding without extensive tuning.
    return make_pipeline(
        StandardScaler(), PolynomialFeatures(degree=2, include_bias=False),
        LogisticRegression(C=0.2, max_iter=2000, random_state=seed),
    )


def aipw_score(T, Y, mu0, mu1, e):
    """Per-person AIPW treatment-effect score (no implicit propensity clipping)."""
    T, Y = np.asarray(T), np.asarray(Y, dtype=float)
    mu0, mu1, e = (_vector(v, len(Y), name) for v, name in
                   [(mu0, "mu0"), (mu1, "mu1"), (e, "propensity")])
    if T.shape != Y.shape or not np.isin(T, [0, 1]).all():
        raise ValueError("T must be a binary vector matching Y")
    if np.any((e <= 0) | (e >= 1)):
        raise ValueError("Propensity must be strictly between zero and one")
    return mu1 - mu0 + T * (Y - mu1) / e - (1 - T) * (Y - mu0) / (1 - e)


def crossfit_scores(X, T, Y, propensity=None, seed=CFG.SEED, n_splits=3,
                    outcome_estimator=None, propensity_estimator=None,
                    estimated_propensity_clip=0.01):
    """Fit a common evaluator entirely out of fold and return its causal scores.

    A known scalar/vector propensity is used exactly, never clipped. Estimated
    propensities are clipped to [clip, 1-clip], and their raw values are returned
    to make this explicit variance/bias tradeoff inspectable. Every outcome fit
    uses only the training-fold observations in the corresponding treatment arm.
    """
    X, T, Y = _inputs(X, T, Y)
    if n_splits < 2 or np.bincount(T).min() < n_splits:
        raise ValueError("Each treatment arm needs at least n_splits observations")
    if not 0 < estimated_propensity_clip < 0.5:
        raise ValueError("estimated_propensity_clip must lie in (0, 0.5)")
    n = len(Y)
    mu0, mu1, e_raw = np.empty(n), np.empty(n), np.empty(n)
    fold_id = np.empty(n, dtype=int)
    if propensity is not None:
        e_raw = _vector(propensity, n, "propensity")
        if np.any((e_raw <= 0) | (e_raw >= 1)):
            raise ValueError("Known propensities must lie strictly in (0,1)")
    model_y = outcome_estimator if outcome_estimator is not None else outcome_model(seed)
    model_t = propensity_estimator if propensity_estimator is not None else propensity_model(seed)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (train, test) in enumerate(cv.split(X, T)):
        fold_id[test] = fold
        for arm, prediction in [(0, mu0), (1, mu1)]:
            arm_train = train[T[train] == arm]
            prediction[test] = clone(model_y).fit(X[arm_train], Y[arm_train]).predict(X[test])
        if propensity is None:
            fitted = clone(model_t).fit(X[train], T[train])
            e_raw[test] = fitted.predict_proba(X[test])[:, list(fitted.classes_).index(1)]
    e = e_raw.copy() if propensity is not None else np.clip(
        e_raw, estimated_propensity_clip, 1 - estimated_propensity_clip)
    return {
        "gamma": aipw_score(T, Y, mu0, mu1, e),
        "ipw": T * Y / e - (1 - T) * Y / (1 - e),
        "mu0": mu0, "mu1": mu1, "e": e, "e_raw": e_raw,
        "fold_id": fold_id, "propensity_known": np.full(n, propensity is not None, dtype=bool),
    }

This code provides a collection of functions for evaluating causal effects and implementing various estimation techniques, primarily focusing on Average Treatment Effect (ATE) estimation. It emphasizes reproducibility and careful handling of potential biases.

The `_vector` function ensures that input values are converted to NumPy arrays with the correct data type (float) and shape (n), raising an error if any value is not finite. This helps prevent unexpected behavior due to invalid numerical inputs.

The `_inputs` function validates the input data for causal estimation, ensuring that X, T, and Y have compatible shapes and data types, and that T contains only binary values (0 or 1). It also checks for missing values in X and Y.

The `outcome_model` and `propensity_model` functions define default machine learning models used for estimating the potential outcomes (`mu0`, `mu1`) and the propensity score (the probability of receiving treatment), respectively. These are deliberately kept simple for reproducibility.

The `aipw_score` function calculates the Augmented Inverse Probability Weighting (AIPW) estimator, a doubly robust estimator that combines inverse probability weighting with regression adjustment. It takes the treatment assignment (`T`), outcome (`Y`), potential outcomes (`mu0`, `mu1`), and propensity score (`e`) as input.

The `crossfit_scores` function implements cross-fitting, a technique for reducing bias in causal effect estimation by fitting models on different folds of the data. It splits the data into multiple folds using stratified k-fold cross-validation, fits outcome and propensity models separately on each fold, and then uses these models to estimate treatment effects on the held-out fold. The function returns various scores, including AIPW estimates, inverse probability weighting (IPW) estimates, potential outcomes, estimated propensity scores, and information about the folds used for estimation. It also allows for clipping of estimated propensities to avoid extreme weights.

In summary, this code provides a comprehensive set of tools for estimating causal effects using various techniques, with a focus on robustness, reproducibility, and careful handling of potential biases. The functions are designed to be modular and flexible, allowing researchers to easily experiment with different estimation methods and model specifications.

In [ ]:
def mean_interval(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    estimate = float(np.mean(values))
    se = float(np.std(values, ddof=1) / np.sqrt(len(values))) if len(values) > 1 else np.nan
    radius = norm.ppf(1 - alpha / 2) * se
    return {"estimate": estimate, "se": se, "ci_low": estimate - radius,
            "ci_high": estimate + radius}


def ate_estimates(T, Y, scores, propensity_known=None):
    """ATE estimators; g-computation uncertainty requires refitting and is omitted.

    IPW/Hajek intervals are shown only for known design propensities. Estimated
    propensity adds uncertainty not captured by their simple score variances.
    AIPW uses cross-fitted influence scores under identification, sufficient
    overlap and nuisance convergence conditions, not a finite-sample guarantee.
    """
    T, Y = np.asarray(T), np.asarray(Y, dtype=float)
    treated, control = Y[T == 1], Y[T == 0]
    diff = float(treated.mean() - control.mean())
    se = np.sqrt(treated.var(ddof=1) / len(treated) + control.var(ddof=1) / len(control))
    if propensity_known is None:
        propensity_known = bool(np.all(scores.get("propensity_known", False)))
    rows = {"Difference in means": {"estimate": diff, "se": se,
                                   "ci_low": diff - 1.96 * se, "ci_high": diff + 1.96 * se,
                                   "interval_method": "Unadjusted two-arm normal approximation"}}
    rows["G-computation"] = {"estimate": float(np.mean(scores["mu1"] - scores["mu0"])),
                              "se": np.nan, "ci_low": np.nan, "ci_high": np.nan,
                              "interval_method": "Omitted: outcome refitting uncertainty required"}
    rows["IPW"] = mean_interval(scores["ipw"])
    e = np.asarray(scores["e"])
    w1, w0 = T / e, (1 - T) / (1 - e)
    m1, m0 = np.sum(w1 * Y) / w1.sum(), np.sum(w0 * Y) / w0.sum()
    influence = w1 * (Y - m1) / w1.mean() - w0 * (Y - m0) / w0.mean()
    hse = np.std(influence, ddof=1) / np.sqrt(len(Y))
    rows["Hajek IPW"] = {"estimate": m1 - m0, "se": hse,
                         "ci_low": m1 - m0 - 1.96 * hse, "ci_high": m1 - m0 + 1.96 * hse}
    rows["AIPW"] = mean_interval(scores["gamma"])
    rows["AIPW"]["interval_method"] = "Cross-fitted influence score; requires nuisance-rate conditions"
    for method in ("IPW", "Hajek IPW"):
        if propensity_known:
            rows[method]["interval_method"] = "Known-propensity influence-score normal approximation"
        else:
            rows[method].update(se=np.nan, ci_low=np.nan, ci_high=np.nan,
                                interval_method="Omitted: propensity refitting uncertainty required")
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("method")


def balance_diagnostics(X, T, e, feature_names=None):
    """SMDs use the same unweighted pooled within-arm SD before and after IPW."""
    X, T = np.asarray(X, dtype=float), np.asarray(T)
    e = _vector(e, len(T), "propensity")
    if np.any((e <= 0) | (e >= 1)):
        raise ValueError("Propensities must lie strictly in (0,1)")
    w1, w0 = 1 / e[T == 1], 1 / (1 - e[T == 0])
    x1, x0 = X[T == 1], X[T == 0]
    scale = np.sqrt((x1.var(axis=0, ddof=1) + x0.var(axis=0, ddof=1)) / 2)
    scale = np.where(scale > 1e-12, scale, 1.0)
    frame = pd.DataFrame({
        "feature": feature_names if feature_names is not None else [f"x{j}" for j in range(X.shape[1])],
        "smd_unweighted": (x1.mean(axis=0) - x0.mean(axis=0)) / scale,
        "smd_ipw": (np.average(x1, axis=0, weights=w1) - np.average(x0, axis=0, weights=w0)) / scale,
    })
    weights = np.concatenate([w1, w0])
    diagnostics = {
        "ess_treated": float(w1.sum() ** 2 / np.square(w1).sum()),
        "ess_control": float(w0.sum() ** 2 / np.square(w0).sum()),
        "n_treated": len(w1), "n_control": len(w0),
        "weight_max": float(weights.max()), "weight_p99": float(np.quantile(weights, .99)),
        "e_min": float(e.min()), "e_max": float(e.max()),
    }
    return frame, diagnostics


def budget_policy(tau, cost, budget, margin=1.0):
    """Maximize sum(pi * (margin*tau-cost)) with binary decisions and budget.

    Equal costs use exact top-positive-profit selection; variable costs use the
    exact mixed-integer knapsack solver. Zero-cost, positive-profit units are
    always selected. There is no requirement to spend the whole budget.
    """
    tau = np.asarray(tau, dtype=float)
    if tau.ndim != 1 or not np.isfinite(tau).all():
        raise ValueError("tau must be a finite vector")
    cost, margin = _vector(cost, len(tau), "cost"), _vector(margin, len(tau), "margin")
    if not np.isfinite(budget) or budget < 0 or np.any(cost < 0) or np.any(margin < 0):
        raise ValueError("Budget, costs, and margins must be finite and nonnegative")
    profit = margin * tau - cost
    policy = (cost == 0) & (profit > 0)
    eligible = np.flatnonzero((profit > 0) & (cost > 0) & (cost <= budget))
    if not len(eligible) or budget == 0:
        return policy
    selected_cost, selected_profit = cost[eligible], profit[eligible]
    if np.all(selected_cost == selected_cost[0]):
        k = min(len(eligible), int(np.floor((budget + 1e-10) / selected_cost[0])))
        order = np.argsort(-selected_profit, kind="stable")
        policy[eligible[order[:k]]] = True
    else:
        result = milp(c=-selected_profit, integrality=np.ones(len(eligible)),
                      bounds=Bounds(np.zeros(len(eligible)), np.ones(len(eligible))),
                      constraints=LinearConstraint(selected_cost[None, :], -np.inf, budget),
                      options={"mip_rel_gap": 0.0})
        if not result.success:
            raise RuntimeError(f"Budget optimization failed: {result.message}")
        policy[eligible] = result.x > 0.5
    if np.dot(policy, cost) > budget + 1e-7 * max(1., budget):
        raise RuntimeError("Optimizer returned an infeasible policy")
    return policy

This code provides functions for evaluating causal effect estimates, performing balance diagnostics, and implementing constrained treatment policies. It focuses on providing robust and informative results with clear documentation of assumptions and limitations.

The `mean_interval` function calculates the mean and confidence interval for a given set of values using a normal approximation.

The `ate_estimates` function computes various estimators of the Average Treatment Effect (ATE), including the difference in means, G-computation, Inverse Probability Weighting (IPW), Hajek IPW, and Augmented Inverse Probability Weighting (AIPW). It also provides confidence intervals for some of these estimators. The function highlights the assumptions underlying each estimator and notes when uncertainty due to model refitting is not accounted for.

The `balance_diagnostics` function assesses covariate balance between treatment groups using standardized mean differences (SMDs) before and after inverse probability weighting. It calculates SMDs for each feature, as well as effective sample size (ESS), maximum weights, and other diagnostic statistics.

The `budget_policy` function implements a constrained treatment policy that maximizes the expected profit given a budget constraint. It selects individuals to treat based on their estimated individual treatment effect (`tau`) and associated costs, subject to a limited budget. The function uses mixed-integer programming (MILP) to find the optimal policy when costs vary across individuals.

In summary, this code provides a comprehensive set of tools for analyzing causal inference results, assessing covariate balance, and implementing constrained treatment policies. It emphasizes robustness, transparency, and careful consideration of assumptions and limitations.

In [ ]:
def _policy_scores(policy, scores, cost, margin):
    gamma = np.asarray(scores["gamma"] if isinstance(scores, dict) else scores, dtype=float)
    policy = _vector(policy, len(gamma), "policy")
    if not np.isin(policy, [0, 1]).all():
        raise ValueError("Policy decisions must be binary")
    cost, margin = _vector(cost, len(gamma), "cost"), _vector(margin, len(gamma), "margin")
    return policy, policy * gamma, policy * (margin * gamma - cost), policy * cost


def evaluate_policy(policy, scores, cost=0.0, margin=1.0):
    """Incremental value relative to treating nobody, per evaluation individual."""
    policy, revenue, profit, spend = _policy_scores(policy, scores, cost, margin)
    r, p = mean_interval(revenue), mean_interval(profit)
    n = len(policy)
    return {
        "n": n, "treated": int(policy.sum()), "treatment_rate": float(policy.mean()),
        "revenue_per_person": r["estimate"], "revenue_total": float(revenue.sum()),
        "profit_per_person": p["estimate"], "profit_total": float(profit.sum()),
        "cost_total": float(spend.sum()),
        "roi": float(profit.sum() / spend.sum()) if spend.sum() else np.nan,
        "profit_se": p["se"], "profit_ci_low": p["ci_low"], "profit_ci_high": p["ci_high"],
        "profit_total_ci_low": n * p["ci_low"], "profit_total_ci_high": n * p["ci_high"],
        "revenue_se": r["se"], "revenue_ci_low": r["ci_low"], "revenue_ci_high": r["ci_high"],
    }


def paired_policy_difference(policy_a, policy_b, scores, cost=0.0, margin=1.0):
    """Paired per-person profit difference A minus B on the same evaluation rows."""
    _, _, profit_a, _ = _policy_scores(policy_a, scores, cost, margin)
    _, _, profit_b, _ = _policy_scores(policy_b, scores, cost, margin)
    return mean_interval(profit_a - profit_b)


def gain_curve(tau, gamma, n_points=21):
    """G(q)=sum(top-q gamma)/n; centered gain G(q)-q*G(1)."""
    tau, gamma = np.asarray(tau), np.asarray(gamma)
    order = np.argsort(-tau, kind="stable")
    n = len(tau)
    k = np.unique(np.rint(np.linspace(0, n, n_points)).astype(int))
    cumsum = np.r_[0., np.cumsum(gamma[order])]
    gain, q = cumsum[k] / n, k / n
    return pd.DataFrame({"q": q, "gain": gain, "centered_gain": gain - q * gamma.mean()})


def calibration_table(tau, gamma, n_bins=5):
    """Equal-count groups based only on predicted CATE; highest scores first."""
    tau, gamma = np.asarray(tau), np.asarray(gamma)
    groups = np.array_split(np.argsort(-tau, kind="stable"), min(n_bins, len(tau)))
    return pd.DataFrame([
        {"bin": i + 1, "n": len(group), "predicted_cate": float(tau[group].mean()),
         **mean_interval(gamma[group])}
        for i, group in enumerate(groups)
    ])


def cate_metrics(predicted, truth):
    predicted, truth = np.asarray(predicted), np.asarray(truth)
    if predicted.shape != truth.shape:
        raise ValueError("Predictions and known CATE truth must have matching shapes")
    return {"cate_rmse": float(np.sqrt(np.mean((predicted - truth) ** 2))),
            "cate_mae": float(np.mean(np.abs(predicted - truth))),
            "ate_pred": float(predicted.mean()), "ate_true": float(truth.mean()),
            "ate_bias": float(np.mean(predicted - truth)),
            "ate_abs_error": float(abs(np.mean(predicted - truth)))}

This code provides functions for evaluating the performance of treatment policies and assessing the accuracy of causal effect estimates.

The `_policy_scores` function calculates key metrics associated with a given treatment policy, including the number of individuals treated, total revenue generated, total profit earned, and total cost incurred. It takes the policy (a binary array indicating which individuals receive treatment), estimated individual treatment effects (`gamma`), costs, and margins as input.

The `evaluate_policy` function evaluates the overall performance of a treatment policy by calculating various metrics such as the number of treated individuals, treatment rate, revenue per person, total revenue, profit per person, total profit, return on investment (ROI), and confidence intervals for profit and revenue. It uses the `_policy_scores` function to obtain the necessary data.

The `paired_policy_difference` function calculates the difference in average profit between two treatment policies applied to the same set of individuals. This allows for a direct comparison of the effectiveness of different policies.

The `gain_curve` function generates a gain curve, which plots the cumulative lift (or gain) achieved by targeting individuals with higher predicted causal effects. It sorts individuals based on their estimated individual treatment effect and calculates the cumulative average treatment effect for increasing proportions of the population.

The `calibration_table` function creates a calibration table that assesses the accuracy of predicted individual treatment effects. It divides individuals into bins based on their predicted CATE scores and calculates summary statistics (mean, standard error, confidence intervals) for the actual causal effects within each bin.

The `cate_metrics` function computes various metrics to evaluate the performance of a model predicting individual causal treatment effects (CATE). These include root mean squared error (RMSE), mean absolute error (MAE), predicted average treatment effect (ATE), true ATE, and bias in the ATE prediction.

In summary, this code provides a comprehensive set of tools for evaluating treatment policies, assessing the accuracy of CATE predictions, and understanding the trade-offs between different policy options. It emphasizes clear metrics, confidence intervals, and visualizations to facilitate informed decision-making.

## Benchmark orchestration

In [ ]:
"""Reproducible matched-data benchmark orchestration with real model execution.

The default runs require both released foundation models and all six classical
methods. Failures are recorded and raised, never substituted by another learner.
Validation chooses a policy model before the randomized test set is evaluated.
"""


def _models(seed, fast=True, model_names=None):
    models = {**make_classical_models(seed=seed, fast=fast),
              **make_foundation_models(seed=seed, device=CFG.DEVICE)}
    if model_names is None:
        return models
    missing = set(model_names) - set(models)
    if missing:
        raise ValueError(f"Unknown model names: {sorted(missing)}")
    return {name: models[name] for name in model_names}


def _manifest(kind, config):
    config = {**config, "foundation_device": CFG.DEVICE}
    payload = json.dumps(config, sort_keys=True)
    return {"experiment": kind, "status": "running", "config": config,
            "config_sha256": hashlib.sha256(payload.encode()).hexdigest(),
            "started_utc": datetime.now(timezone.utc).isoformat(),
            "python": sys.version, "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "sklearn": sklearn.__version__, "scipy": scipy.__version__,
            "matched_data": True, "test_based_tuning": False,
            "fit_timing_includes_checkpoint_loading": True,
            "pretraining_cost_included": False,
            "notes": [], "failures": []}


def _save(output_dir, kind, tables, predictions, manifest):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(output_dir / f"{kind}_{name}.csv", index=False)
    if predictions:
        np.savez_compressed(output_dir / f"{kind}_predictions.npz", **predictions)
    (output_dir / f"{kind}_manifest.json").write_text(
        json.dumps(manifest, indent=2, default=str) + "\n")


def _fit_predict(model, X, T, Y, query):
    synchronize_device(getattr(model, "device", "cpu"))
    started = time.perf_counter()
    model.fit(X, T, Y)
    synchronize_device(getattr(model, "device", "cpu"))
    fit_seconds = time.perf_counter() - started
    synchronize_device(getattr(model, "device", "cpu"))
    started = time.perf_counter()
    prediction = np.asarray(model.predict_cate(query), dtype=float).reshape(-1)
    synchronize_device(getattr(model, "device", "cpu"))
    prediction_seconds = time.perf_counter() - started
    if prediction.shape != (len(query),) or not np.isfinite(prediction).all():
        raise ValueError("Model returned invalid CATE predictions")
    timing = {"device": getattr(model, "device", "cpu"), "fit_seconds": fit_seconds, "predict_seconds": prediction_seconds,
              "total_seconds": fit_seconds + prediction_seconds,
              "checkpoint_load_seconds": getattr(model, "load_seconds", np.nan),
              "training_rows_accessed": getattr(model, "unique_training_rows", len(X)),
              "context_rows": getattr(model, "context_size", len(X)),
              "query_rows": len(query)}
    return prediction, timing


def _subset(part, n, seed):
    """Uniform, outcome-blind subset; no stratification on realized outcomes."""
    if n is None or n >= len(part["T"]):
        return {key: value.copy() if hasattr(value, "copy") else value for key, value in part.items()}
    rows = np.sort(np.random.default_rng(seed).choice(len(part["T"]), n, replace=False))
    return {key: (value[rows] if isinstance(value, np.ndarray) and value.ndim > 0 and len(value) == len(part["T"]) else value)
            for key, value in part.items()}


def _top_k(score, n_select):
    policy = np.zeros(len(score), dtype=bool)
    policy[np.argsort(-np.asarray(score), kind="stable")[:n_select]] = True
    return policy

This code provides the core orchestration logic for running a benchmark of causal inference models on matched data. It focuses on reproducibility, detailed logging, and robust error handling.

The `_models` function creates a dictionary of causal inference models, including both classical machine learning methods and foundation models. It allows specifying a subset of models to run based on their names.

The `_manifest` function generates a manifest file containing metadata about the experiment, including configuration details, timestamps, software versions, and other relevant information. This ensures reproducibility and traceability.

The `_save` function saves the results of an experiment to disk, including tables of performance metrics, predicted causal effects, and the experiment manifest.

The `_fit_predict` function fits a given model to training data and then uses it to predict individual treatment effects on a query set. It measures the time taken for both fitting and prediction and includes device information in timing logs.  It also performs validation checks on the predictions. The `synchronize_device` calls ensure accurate timing measurements by waiting for asynchronous operations to complete.

The `_subset` function creates a random subset of data, ensuring that the selection is outcome-blind (i.e., not influenced by the observed outcomes). This is important for evaluating model performance on unseen data.

The `_top_k` function selects the top k individuals based on their predicted causal effects, creating a binary policy indicating which individuals should receive treatment.

In summary, this code provides the infrastructure for running and logging experiments with various causal inference models. It emphasizes reproducibility, detailed timing information, and robust error handling to ensure reliable results. The functions are designed to be modular and flexible, allowing researchers to easily customize the benchmark setup and evaluate different model configurations.

In [ ]:
def _fail(manifest, output_dir, kind, tables, predictions, context, error):
    manifest["status"] = "failed"
    manifest["failures"].append({**context, "exception": repr(error),
                                 "traceback": traceback.format_exc()})
    _save(output_dir, kind, tables, predictions, manifest)


def load_benchmark_results(kind="accuracy", output_dir=CFG.OUT):
    """Read generated artifacts without rerunning inference; require completion."""
    output_dir = Path(output_dir)
    manifest = json.loads((output_dir / f"{kind}_manifest.json").read_text())
    if manifest["status"] != "complete":
        raise RuntimeError(f"Cached {kind} run is {manifest['status']}; inspect its manifest")
    if manifest['config'].get('foundation_device') != CFG.DEVICE or manifest['config']['mode'] != CFG.MODE:
        raise ValueError('Cached results use a different mode/device. Set CFG.REUSE_RESULTS = False and rerun.')
    result = {file.stem.removeprefix(f"{kind}_"): pd.read_csv(file)
              for file in output_dir.glob(f"{kind}_*.csv")}
    path = output_dir / f"{kind}_predictions.npz"
    with np.load(path, allow_pickle=False) as arrays:
        result["predictions"] = {key: arrays[key] for key in arrays.files}
    result["manifest"] = manifest
    return result


def run_accuracy_benchmark(mode=CFG.MODE, output_dir=CFG.OUT, *, seeds=None,
                           sample_sizes=None, mechanisms=None, ihdp_reps=None,
                           query_n=None, model_names=None, progress=True):
    """Known-CATE comparison using independent queries and original IHDP splits.

    Demo: two seeds, n=256/512, linear/nonlinear mechanisms, three IHDP
    replications. Reference expands mechanisms, sample sizes, and repetitions.
    This is deliberately a frozen-default comparison with no test-based tuning.
    """

    if mode not in ("demo", "reference"):
        raise ValueError("mode must be 'demo' or 'reference'")
    workload = CFG.WORKLOADS[mode]
    seeds = list(seeds if seeds is not None else workload["accuracy_seeds"])
    sample_sizes = list(sample_sizes if sample_sizes is not None else workload["accuracy_sample_sizes"])
    mechanisms = list(mechanisms if mechanisms is not None else workload["accuracy_mechanisms"])
    ihdp_reps = workload["ihdp_reps"] if ihdp_reps is None else ihdp_reps
    query_n = workload["accuracy_query_n"] if query_n is None else query_n
    names = list(_models(seeds[0], model_names=model_names))
    config = {"mode": mode, "seeds": seeds, "sample_sizes": sample_sizes,
              "mechanisms": mechanisms, "ihdp_reps": ihdp_reps, "query_n": query_n,
              "models": names, "policy_contact_cost": CFG.CONTACT_COST, "policy_budget_fraction": CFG.PRIMARY_BUDGET_FRACTION,
              "defaults_frozen": True}
    manifest = _manifest("accuracy", config)
    manifest["notes"].extend([
        "Each simulation seed has a separate query draw, shared across models and sample sizes.",
        "IHDP uses author test partitions; no true outcomes enter model fitting or preprocessing.",
        "Two demo seeds and three IHDP replications are illustrative, not publication-strength inference.",
        "Policy regret is known conditional-mean profit regret on the fixed query population.",
    ])
    tasks = []
    for mechanism in mechanisms:
        for seed in seeds:
            query = simulate_individual(query_n, seed=seed + 100_000, scenario=mechanism)
            for n in sample_sizes:
                train = simulate_individual(n, seed=seed, scenario=mechanism)
                tasks.append(({"dataset": "Simulation", "scenario": mechanism,
                               "seed": seed, "replication": -1, "n_train": n}, train, query))
    for replication in range(ihdp_reps):
        data = load_ihdp(replication=replication)
        tasks.append(({"dataset": "IHDP", "scenario": "semi-synthetic",
                       "seed": 100 + replication, "replication": replication,
                       "n_train": len(data["train"]["T"])}, data["train"], data["test"]))
    rows, predictions = [], {}
    for task_id, (setting, train, query) in enumerate(tasks):
        key = f"task{task_id:03d}"
        predictions[f"{key}__truth"] = query["tau"]
        predictions[f"{key}__query_id"] = query["id"]
        oracle_policy = budget_policy(query["tau"], CFG.CONTACT_COST, len(query["T"]) * CFG.PRIMARY_BUDGET_FRACTION * CFG.CONTACT_COST)
        oracle_value = np.mean(oracle_policy * (query["tau"] - CFG.CONTACT_COST))
        for name, model in _models(setting["seed"], fast=mode == "demo", model_names=names).items():
            if progress:
                print(f"Accuracy {task_id + 1}/{len(tasks)} {setting['dataset']} "
                      f"{setting['scenario']} n={setting['n_train']} {name}", flush=True)
            try:
                prediction, timing = _fit_predict(model, train["X"], train["T"], train["Y"], query["X"])
                policy = budget_policy(prediction, CFG.CONTACT_COST, len(query["T"]) * CFG.PRIMARY_BUDGET_FRACTION * CFG.CONTACT_COST)
                value = np.mean(policy * (query["tau"] - CFG.CONTACT_COST))
                prediction_key = f"{key}__{name.replace(' ', '_')}"
                predictions[prediction_key] = prediction
                rows.append({**setting, "method": name, "track": "Matched data",
                             "prediction_key": prediction_key, "truth_key": f"{key}__truth",
                             **cate_metrics(prediction, query["tau"]), **timing,
                             "policy_value_true": value, "policy_regret": oracle_value - value,
                             "oracle_policy_value": oracle_value, "status": "success"})
            except Exception as error:
                _fail(manifest, output_dir, "accuracy", {"metrics": pd.DataFrame(rows)}, predictions,
                      {**setting, "method": name}, error)
                raise RuntimeError(f"Required accuracy model failed: {name}, {setting}") from error
            finally:
                del model
                gc.collect()
        _save(output_dir, "accuracy", {"metrics": pd.DataFrame(rows)}, predictions, manifest)
    manifest.update(status="complete", completed_utc=datetime.now(timezone.utc).isoformat(),
                    successful_model_runs=len(rows))
    result = {"metrics": pd.DataFrame(rows), "predictions": predictions, "manifest": manifest}
    _save(output_dir, "accuracy", {"metrics": result["metrics"]}, predictions, manifest)
    return result

This code orchestrates a comprehensive benchmark of causal inference models on both simulated and real-world datasets (IHDP). It focuses on evaluating the accuracy of CATE (Conditional Average Treatment Effect) predictions.

The `_fail` function handles errors during model fitting or prediction, logging the error details in the manifest file and saving the current state of the experiment.

The `load_benchmark_results` function loads previously generated benchmark results from disk, ensuring that the cached run is complete and compatible with the current configuration (mode and device). It raises an error if the cached results are incomplete or incompatible.

The `run_accuracy_benchmark` function performs the main benchmark execution. It iterates through different simulation mechanisms, random seeds, sample sizes, and IHDP replications. For each combination, it trains a set of causal inference models on training data and evaluates their performance on a separate query dataset. The evaluation metrics include CATE RMSE, MAE, ATE bias, policy value (profit), and regret compared to an oracle policy.

Here's a breakdown of the key steps:

1.  **Configuration:** Loads configuration parameters from `CFG`, including seeds, sample sizes, mechanisms, and model names.
2.  **Manifest Creation:** Creates a manifest file containing metadata about the experiment.
3.  **Task Generation:** Generates a list of tasks, each representing a specific combination of dataset, scenario, seed, and sample size.
4.  **Model Iteration:** Iterates through the specified causal inference models.
5.  **Fitting and Prediction:** Fits each model to the training data and predicts CATEs on the query data using `_fit_predict`.
6.  **Policy Evaluation:** Implements a budget-constrained policy based on the predicted CATEs and evaluates its performance in terms of profit and regret.
7.  **Result Saving:** Saves the evaluation metrics, predictions, and manifest file to disk.
8.  **Error Handling:** Uses `_fail` to handle any errors that occur during model fitting or prediction.

The function also includes checks for reproducibility (seed management) and resource management (garbage collection). It provides progress updates during execution and raises an error if a required model fails. The code is designed to be modular and flexible, allowing researchers to easily customize the benchmark setup and evaluate different model configurations.

In [ ]:
def run_hillstrom_benchmark(mode=CFG.MODE, output_dir=CFG.OUT, *, train_n=None,
                            test_n=None, val_n=None, seed=CFG.SEED, model_names=None,
                            include_practical=True, progress=True, contact_cost=CFG.CONTACT_COST,
                            margin=CFG.MARGIN, budget_fractions=CFG.BUDGET_FRACTIONS):
    """Randomized marketing evaluation after randomized/observational training.

    Holdout sizes are bounded for local inference. Their row identities are
    fixed without outcome-dependent sampling. Evaluation scores are common across
    all policies, with known randomized propensity 0.5. Actual spend remains in
    its original units. Costs and margins are explicit scenario assumptions.
    """

    if mode not in ("demo", "reference"):
        raise ValueError("mode must be 'demo' or 'reference'")
    workload = CFG.WORKLOADS[mode]
    train_n = workload["marketing_train_n"] if train_n is None else train_n
    test_n = workload["marketing_test_n"] if test_n is None else test_n
    val_n = workload["marketing_val_n"] if val_n is None else val_n
    primary_fraction = CFG.PRIMARY_BUDGET_FRACTION
    if contact_cost <= 0 or margin <= 0:
        raise ValueError("The marketing scenario requires positive cost and margin")
    if primary_fraction not in budget_fractions:
        raise ValueError("Include CFG.PRIMARY_BUDGET_FRACTION in budget_fractions")
    data = prepare_hillstrom(seed=seed)
    test, valid = _subset(data["test"], test_n, seed + 1), _subset(data["valid"], val_n, seed + 2)
    randomized_pool = data["train"]
    observational_pool = select_observational(randomized_pool, strength=CFG.SELECTION_STRENGTH, seed=seed + 3)
    pools = {"Randomized": randomized_pool, "Observed confounding": observational_pool}
    if any(len(part["T"]) < train_n for part in pools.values()):
        raise ValueError("Insufficient rows for the requested matched context size")
    names = list(_models(seed, model_names=model_names))
    config = {"mode": mode, "seed": seed, "train_n": train_n, "test_n": len(test["T"]),
              "val_n": len(valid["T"]), "models": names, "include_practical": include_practical,
              "contact_cost": contact_cost, "margin": margin,
              "budget_fractions": list(budget_fractions), "primary_budget_fraction": primary_fraction,
              "regimes": list(pools), "selection_strength": CFG.SELECTION_STRENGTH, "known_test_propensity": .5}
    manifest = _manifest("hillstrom", config)
    manifest["dataset_metadata"] = data["meta"]
    manifest["selection_metadata"] = observational_pool.get("selection_meta", {})
    manifest["notes"].extend([
        "Dollar spend is the observed outcome; contact costs and margin are assumed scenarios.",
        "Common categorical encoding/scaling is fitted on the full training pool; matched model fitting accesses the same limited labeled context rows. No holdout rows fit preprocessing.",
        "Validation chooses a matched-data model separately per training regime at the fixed 20% budget.",
        "Abstention is allowed: a best-candidate validation profit at or below zero locks the treat-nobody policy.",
        "The test cohort is never used to select the reported primary policy or its budget.",
        "Common evaluator outcome regressions are cross-fit within each independent holdout.",
        "Intervals condition on frozen policies; training and model-selection uncertainty are excluded.",
        "Practical track fits Interaction OLS and T-learner on full available training pools; it is separate from matched-data selection.",
    ])
    predictions = {"test_id": test["id"], "valid_id": valid["id"],
                   "test_T": test["T"], "test_Y": test["Y"],
                   "valid_T": valid["T"], "valid_Y": valid["Y"]}
    test_scores = crossfit_scores(test["X"], test["T"], test["Y"], propensity=.5, seed=seed)
    valid_scores = crossfit_scores(valid["X"], valid["T"], valid["Y"], propensity=.5, seed=seed + 1)
    for split, scores in [("test", test_scores), ("valid", valid_scores)]:
        predictions.update({f"{split}_score_{key}": value for key, value in scores.items()})
    tables = {"metrics": [], "policies": [], "paired": [], "gains": [],
              "calibration": [], "ate": [], "balance": [], "diagnostics": [], "selection": []}
    all_queries = np.concatenate([valid["X"], test["X"]])
    n_valid = len(valid["T"])
    for regime_id, (regime, pool) in enumerate(pools.items()):
        train = _subset(pool, train_n, seed + 5 + regime_id)
        prefix = f"regime{regime_id}"
        predictions[f"{prefix}_train_id"] = train["id"]
        # These observed-data ATE panels concern the selected training cohort.
        # The Monte Carlo/sensitivity experiments separately establish truth.
        estimated = crossfit_scores(train["X"], train["T"], train["Y"], seed=seed)
        for propensity_kind, scores in [("Estimated propensity", estimated),
                                        ("Known design propensity", crossfit_scores(
                                            train["X"], train["T"], train["Y"],
                                            propensity=train["e"], seed=seed))]:
            frame = ate_estimates(train["T"], train["Y"], scores).reset_index()
            frame["regime"], frame["propensity_kind"] = regime, propensity_kind
            tables["ate"].append(frame)
            balance, diag = balance_diagnostics(train["X"], train["T"], scores["e"], data["feature_names"])
            balance["regime"], balance["propensity_kind"] = regime, propensity_kind
            tables["balance"].append(balance)
            tables["diagnostics"].append({"regime": regime, "propensity_kind": propensity_kind, **diag})
        candidates = {name: ("Matched data", model, train)
                      for name, model in _models(seed, fast=mode == "demo", model_names=names).items()}
        if include_practical:
            for name in ("Interaction OLS", "T-learner"):
                candidates[f"{name} [full data]"] = ("Practical full data",
                    make_classical_models(seed=seed, fast=mode == "demo")[name], pool)
        candidate_predictions, validation_values = {}, {}
        for name, (track, model, fit_data) in candidates.items():
            if progress:
                print(f"Hillstrom {regime} {track} n={len(fit_data['T'])} {name}", flush=True)
            try:
                prediction, timing = _fit_predict(model, fit_data["X"], fit_data["T"], fit_data["Y"], all_queries)
                pred_valid, pred_test = prediction[:n_valid], prediction[n_valid:]
                key = f"{prefix}__{name.replace(' ', '_')}"
                predictions[f"{key}__valid"], predictions[f"{key}__test"] = pred_valid, pred_test
                candidate_predictions[name] = (track, pred_valid, pred_test)
                validation_policy = budget_policy(pred_valid, contact_cost,
                    len(pred_valid) * primary_fraction * contact_cost, margin)
                validation_values[name] = evaluate_policy(validation_policy, valid_scores, contact_cost, margin)["profit_per_person"]
                tables["metrics"].append({"regime": regime, "method": name, "track": track,
                                          "n_train": len(fit_data["T"]), **timing,
                                          "validation_profit": validation_values[name],
                                          "test_prediction_key": f"{key}__test", "status": "success"})
            except Exception as error:
                _fail(manifest, output_dir, "hillstrom", _tables_to_frames(tables), predictions,
                      {"regime": regime, "method": name, "track": track}, error)
                raise RuntimeError(f"Required Hillstrom model failed: {name}, {regime}") from error
        # Lock the primary winner using validation outcomes before scoring test policies.
        best_candidate = max(names, key=lambda name: validation_values[name])
        winner = best_candidate if validation_values[best_candidate] > 0 else "Treat nobody"
        validation_values["Treat nobody"] = 0.
        tables["selection"].append({"regime": regime, "selected_method": winner,
                                    "best_candidate_model": best_candidate,
                                    "best_candidate_validation_profit": validation_values[best_candidate],
                                    "no_deploy": winner == "Treat nobody",
                                    "selection_split": "validation", "budget_fraction": primary_fraction,
                                    "validation_profit_per_person": validation_values[winner]})
        response_model = outcome_model(seed).fit(train["X"], train["Y"])
        response_score = response_model.predict(test["X"])
        history_columns = [i for i, label in enumerate(data["feature_names"]) if label == "history" or label.endswith("__history")]
        # Only expose the business rule if an identified pretreatment history feature exists.
        history_score = test["X"][:, history_columns[0]] if history_columns else None
        random_score = np.random.default_rng(seed + 20).uniform(size=len(test["T"]))
        policies_by_budget = {}
        for fraction in budget_fractions:
            budget = len(test["T"]) * fraction * contact_cost
            n_select = min(len(test["T"]), int(np.floor(budget / contact_cost + 1e-8)))
            policies = {name: budget_policy(pred_test, contact_cost, budget, margin)
                        for name, (_, _, pred_test) in candidate_predictions.items()}
            policies.update({"Random targeting": _top_k(random_score, n_select),
                             "Response targeting": _top_k(response_score, n_select),
                             "Treat nobody": np.zeros(len(test["T"]), dtype=bool)})
            if history_score is not None:
                policies["History rule"] = _top_k(history_score, n_select)
            policies_by_budget[fraction] = policies
            for name, policy in policies.items():
                track = candidate_predictions[name][0] if name in candidate_predictions else "Business baseline"
                value = evaluate_policy(policy, test_scores, contact_cost, margin)
                tables["policies"].append({"regime": regime, "method": name, "track": track,
                                           "budget_fraction": fraction, "budget": budget,
                                           "selected_on_validation": name == winner, **value})
                # Retain randomization-only IPW evaluation as an independent reference.
                ipw_value = evaluate_policy(policy, test_scores["ipw"], contact_cost, margin)
                tables["policies"][-1]["ipw_profit_per_person"] = ipw_value["profit_per_person"]
                difference = paired_policy_difference(policy, policies["Random targeting"], test_scores, contact_cost, margin)
                tables["paired"].append({"regime": regime, "method": name,
                                         "reference": "Random targeting", "budget_fraction": fraction,
                                         "selected_on_validation": name == winner, **difference})
        predictions[f"{prefix}__selected_policy"] = policies_by_budget[primary_fraction][winner]
        for name, (track, _, pred_test) in candidate_predictions.items():
            curve = gain_curve(pred_test, test_scores["gamma"])
            metric_row = next(row for row in tables["metrics"] if row["regime"] == regime and row["method"] == name)
            metric_row["auuc"] = float(np.trapz(curve["gain"], curve["q"]))
            metric_row["centered_auuc"] = float(np.trapz(curve["centered_gain"], curve["q"]))
            curve["regime"], curve["method"], curve["track"] = regime, name, track
            tables["gains"].append(curve)
            calibration = calibration_table(pred_test, test_scores["gamma"])
            calibration["regime"], calibration["method"], calibration["track"] = regime, name, track
            tables["calibration"].append(calibration)
        _save(output_dir, "hillstrom", _tables_to_frames(tables), predictions, manifest)
        del candidates
        gc.collect()
    manifest.update(status="complete", completed_utc=datetime.now(timezone.utc).isoformat(),
                    successful_model_runs=len(tables["metrics"]))
    frames = _tables_to_frames(tables)
    _save(output_dir, "hillstrom", frames, predictions, manifest)
    return {**frames, "predictions": predictions, "scores": test_scores, "manifest": manifest}

This code orchestrates a comprehensive benchmark of causal inference models applied to the Hillstrom customer marketing dataset. It simulates a real-world scenario where treatment (sending an email) is randomized, and then evaluates different policies for targeting customers based on predicted individual treatment effects.

The `run_hillstrom_benchmark` function performs the following key steps:

1.  **Data Preparation:** Loads and preprocesses the Hillstrom dataset using `prepare_hillstrom`, creating training, validation, and test sets. It also creates an observational pool by applying a selection mechanism to simulate confounding.
2.  **Model Setup:** Creates a dictionary of causal inference models using `_models`.
3.  **Manifest Creation:** Generates a manifest file containing metadata about the experiment.
4.  **Training and Prediction:** Trains each model on the training data and predicts individual treatment effects on the validation and test sets using `_fit_predict` and `crossfit_scores`.
5.  **Policy Evaluation:** Evaluates different policies for targeting customers based on their predicted treatment effects, considering a budget constraint and contact cost. It selects the best policy based on its performance on the validation set.
6.  **Performance Metrics:** Calculates various performance metrics, including ATE estimates, balance diagnostics, gain curves, calibration tables, and profit/regret measures.
7.  **Result Saving:** Saves the evaluation results, predictions, and manifest file to disk using `_save`.

Here's a breakdown of some key features:

*   **Randomized vs. Observational Data:** The benchmark evaluates models trained on both randomized (control group) and observational data (confounded data), allowing for an assessment of their robustness to confounding bias.
*   **Cross-Fitting:** Uses cross-fitting to reduce bias in the estimation of treatment effects.
*   **Budget Constraint:** Implements a budget constraint when evaluating policies, ensuring that the total cost of treatment does not exceed a specified limit.
*   **Policy Selection:** Selects the best policy based on its performance on a validation set.
*   **Comprehensive Metrics:** Calculates a wide range of metrics to assess model performance and policy effectiveness.
*   **Detailed Logging:** Logs all relevant information about the experiment, including configuration parameters, timing information, and error messages.

The function also includes checks for reproducibility (seed management) and resource management (garbage collection). It provides progress updates during execution and raises an error if a required model fails. The code is designed to be modular and flexible, allowing researchers to easily customize the benchmark setup and evaluate different model configurations.

In [ ]:
def _tables_to_frames(tables):
    return {name: (pd.concat(parts, ignore_index=True) if parts and isinstance(parts[0], pd.DataFrame)
                   else pd.DataFrame(parts)) for name, parts in tables.items()}

This function, `_tables_to_frames`, consolidates a dictionary of lists of Pandas DataFrames or other data structures into a single dictionary where each key maps to a single Pandas DataFrame.

It iterates through the input `tables` dictionary. For each key-value pair:

1.  It checks if the value (`parts`) is a non-empty list and if the first element in the list is a Pandas DataFrame.
2.  If both conditions are true, it concatenates all DataFrames in the list vertically using `pd.concat(..., ignore_index=True)`, creating a single DataFrame with a continuous index.
3.  Otherwise (if the value is not a list of DataFrames or if the list is empty), it creates a Pandas DataFrame directly from the value using `pd.DataFrame(parts)`.

The function then returns a new dictionary where each key maps to a single Pandas DataFrame, effectively flattening the original structure. This is useful for preparing data for saving to files or further analysis.

## Robustness and sensitivity helpers

Repeated cohorts, nuisance specification checks, and formal additive-OLS sensitivity.

In [ ]:
"""Identification stress tests and an explicitly OLS-only sensitivity analysis.

The four-cell experiment varies *functional specification*, not access to causal
truth: correct models are fitted on an expanded basis containing the DGP, while
incorrect models see only raw linear covariates. All predictions are cross-fit.

OLS sensitivity follows Cinelli & Hazlett's omitted-variable bias identities:
https://carloscinelli.com/sensemakr/reference/sensemakr.html
It is not a sensitivity bound for arbitrary CATE models or policy profit.
"""




def nonlinear_basis(x: np.ndarray) -> np.ndarray:
    """Declared basis; its use constitutes a correctly specified parametric model."""
    x = np.asarray(x, dtype=float)
    return np.column_stack([x, x[:, 0] ** 2 - 0.75,
                            np.sin(2 * x[:, 1]), x[:, 0] * x[:, 1]])


def simulate_robustness_data(
    n: int = 1200, seed: int=CFG.SEED, overlap_strength: float = 1.0,
    hidden_strength: float = 0.0,
) -> dict:
    """Generate a population with ATE=1 and an optional hidden confounder U.

    X is the learner-visible matrix. U, e, tau, and mu0 are simulator truth,
    not additional predictors. basis_full is used only by the labeled oracle.
    Increasing overlap_strength multiplies the treatment log-odds; increasing
    hidden_strength makes U predict both treatment and untreated outcome.
    """
    if n < 30 or overlap_strength < 0 or hidden_strength < 0:
        raise ValueError("Use n>=30 and nonnegative stress strengths.")
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1.5, 1.5, size=(n, 3))
    u = rng.uniform(-1.5, 1.5, n)
    basis = nonlinear_basis(x)
    mu0 = (1 + 0.8*x[:, 0] - 0.5*x[:, 1] + 1.2*basis[:, 3]
           + 0.9*basis[:, 4] + 0.7*basis[:, 5] + hidden_strength*u)
    tau = 1 + 0.3*x[:, 0] - 0.2*x[:, 1]
    logit = 0.15 + overlap_strength*(0.4*x[:, 0] + 0.9*basis[:, 3]
                                   + 0.8*basis[:, 4] + 0.5*basis[:, 5])
    propensity = expit(logit + hidden_strength*u)
    treatment = rng.binomial(1, propensity)
    outcome = mu0 + treatment*tau + rng.normal(0, 1, n)
    return {"X": x, "T": treatment, "Y": outcome, "e": propensity,
            "tau": tau, "mu0": mu0, "U": u, "basis": basis,
            "basis_full": np.column_stack([basis, u]), "population_ate": 1.0}


def _crossfit_nuisances(
    outcome_features: np.ndarray, propensity_features: np.ndarray,
    treatment: np.ndarray, outcome: np.ndarray, folds: int, seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = len(treatment)
    mu0, mu1, propensity = (np.empty(n) for _ in range(3))
    splitter = StratifiedKFold(folds, shuffle=True, random_state=seed)
    for train, test in splitter.split(outcome_features, treatment):
        for arm, output in ((0, mu0), (1, mu1)):
            rows = train[treatment[train] == arm]
            model = LinearRegression().fit(outcome_features[rows], outcome[rows])
            output[test] = model.predict(outcome_features[test])
        # Large C approximates unpenalized maximum likelihood, compatible with
        # both older and newer scikit-learn releases.
        propensity_model = LogisticRegression(C=1e6, max_iter=1000)
        propensity_model.fit(propensity_features[train], treatment[train])
        propensity[test] = propensity_model.predict_proba(propensity_features[test])[:, 1]
    return mu0, mu1, propensity


def _scores(t, y, mu0, mu1, e, clip=1e-6) -> dict:
    e = np.clip(e, clip, 1-clip)
    return {"Outcome regression": mu1-mu0,
            "IPW": t*y/e - (1-t)*y/(1-e),
            "AIPW": mu1-mu0 + t*(y-mu1)/e - (1-t)*(y-mu0)/(1-e)}


def _summary(draws: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    rows = []
    for key, frame in draws.groupby(group_columns, sort=False):
        key = key if isinstance(key, tuple) else (key,)
        error = frame["estimate"] - frame["truth"]
        row = dict(zip(group_columns, key))
        row.update(mean_estimate=frame["estimate"].mean(), bias=error.mean(),
                   rmse=np.sqrt(np.mean(error**2)), repetitions=len(frame),
                   bias_mcse=error.std(ddof=1)/np.sqrt(len(frame)),
                   mean_effective_sample_size=frame["effective_sample_size"].mean()
                   if "effective_sample_size" in frame else np.nan)
        if "covered" in frame:
            valid = frame["covered"].dropna()
            row["coverage"] = valid.mean() if len(valid) else np.nan
            row["coverage_mcse"] = (np.sqrt(row["coverage"]*(1-row["coverage"])/len(valid))
                                    if len(valid) else np.nan)
        rows.append(row)
    return pd.DataFrame(rows)

This code implements robustness checks and sensitivity analyses for causal inference models, focusing on identifying potential biases and limitations. It simulates data with varying degrees of confounding and model misspecification to assess the performance of different estimation techniques.

The `nonlinear_basis` function defines a nonlinear basis expansion of input features, which can be used to improve the flexibility of parametric models.

The `simulate_robustness_data` function generates synthetic data with a known causal effect (ATE=1) and optional hidden confounding. It allows controlling the strength of overlap between treatment groups and the influence of a hidden confounder on both treatment assignment and outcome. The function returns a dictionary containing the generated data, true causal effects, and other relevant information.

The `_crossfit_nuisances` function estimates potential outcomes (mu0, mu1) and propensity scores using cross-fitting, a technique for reducing bias in causal effect estimation. It splits the data into folds using stratified k-fold cross-validation and fits linear regression models to estimate the potential outcomes and logistic regression to estimate the propensity score.

The `_scores` function calculates various estimators of the Average Treatment Effect (ATE), including outcome regression, Inverse Probability Weighting (IPW), and Augmented Inverse Probability Weighting (AIPW). It also clips the estimated propensity scores to avoid extreme weights.

The `_summary` function summarizes the results of a simulation study by calculating mean estimates, biases, root mean squared errors (RMSEs), coverage probabilities, and other relevant metrics. It groups the results by different experimental conditions and provides standard errors for the estimated metrics.

In summary, this code provides tools for evaluating the robustness of causal inference models to model misspecification and confounding bias. It simulates data with varying degrees of complexity and uses cross-fitting to reduce bias in estimation. The code also includes functions for calculating various performance metrics and summarizing the results of simulation studies.

In [ ]:
def run_double_robustness_experiment(
    n: int = 1200, repetitions: int = 30, seed: int=CFG.SEED, folds: int = 3,
) -> dict[str, pd.DataFrame]:
    """Cross-fitted four-cell specification experiment, population ATE target.

    Coverage is reported only for AIPW's influence-score interval. With an
    incorrect nuisance, consistency can persist but the simple variance formula
    need not include all first-order nuisance estimation effects; the empirical
    coverage panel exposes this rather than claiming a universal guarantee.
    """
    if repetitions < 2:
        raise ValueError("At least two repetitions are needed for Monte Carlo summaries.")
    rows = []
    for rep in range(repetitions):
        data = simulate_robustness_data(n, seed+rep)
        t, y = data["T"], data["Y"]
        for outcome_correct in (True, False):
            for propensity_correct in (True, False):
                mu0, mu1, propensity = _crossfit_nuisances(
                    data["basis"] if outcome_correct else data["X"],
                    data["basis"] if propensity_correct else data["X"],
                    t, y, folds, seed+rep,
                )
                for method, score in _scores(t, y, mu0, mu1, propensity).items():
                    estimate = float(score.mean())
                    standard_error = float(score.std(ddof=1)/np.sqrt(n)) if method == "AIPW" else np.nan
                    rows.append({"rep": rep, "outcome_spec": "Correct" if outcome_correct else "Wrong",
                                 "propensity_spec": "Correct" if propensity_correct else "Wrong",
                                 "method": method, "estimate": estimate, "truth": data["population_ate"],
                                 "se": standard_error,
                                 "covered": float(abs(estimate-1) <= 1.96*standard_error)
                                 if method == "AIPW" else np.nan})
    draws = pd.DataFrame(rows)
    return {"draws": draws, "summary": _summary(draws, ["outcome_spec", "propensity_spec", "method"])}


def run_sensitivity_sweeps(
    n: int = 1200, repetitions: int = 20, seed: int = 120,
    overlap_strengths=(0.0, 1.0, 2.0, 3.0), hidden_strengths=(0.0, 0.5, 1.0, 1.5),
    folds: int = 3, clips=(0.001, 0.02, 0.1),
) -> dict[str, pd.DataFrame]:
    """Separate overlap, hidden-confounding, and clipping stress experiments.

    These are controlled empirical stress tests, not formal sensitivity bounds.
    Oracle AIPW is a diagnostic using U in both fitted nuisance models. All
    non-oracle methods are denied U. Clipping keeps the population target and
    can bias its estimator; no trimming or target-population change is hidden.
    """
    if repetitions < 2:
        raise ValueError("At least two repetitions are needed.")
    rows, diagnostics = [], []
    settings = [("Overlap", s, s, 0.0) for s in overlap_strengths]
    settings += [("Hidden confounding", s, 1.0, s) for s in hidden_strengths]
    for kind, strength, overlap, hidden in settings:
        for rep in range(repetitions):
            data = simulate_robustness_data(n, seed+rep, overlap, hidden)
            t, y = data["T"], data["Y"]
            mu0, mu1, e = _crossfit_nuisances(data["basis"], data["basis"], t, y, folds, seed+rep)
            weights = t/np.clip(e, 1e-6, 1) + (1-t)/np.clip(1-e, 1e-6, 1)
            ess_by_arm = [weights[t == arm].sum()**2 / np.sum(weights[t == arm]**2) for arm in (0, 1)]
            diagnostics.append({"experiment": kind, "strength": strength, "rep": rep,
                                "propensity_min": e.min(), "propensity_max": e.max(),
                                "outside_01_09": np.mean((e < 0.1) | (e > 0.9)),
                                "max_weight": weights.max(), "ess_control": ess_by_arm[0],
                                "ess_treated": ess_by_arm[1]})
            estimates = {name: float(score.mean()) for name, score in _scores(t, y, mu0, mu1, e).items()}
            estimates["Unadjusted"] = float(y[t == 1].mean()-y[t == 0].mean())
            if kind == "Hidden confounding":
                f0, f1, fe = _crossfit_nuisances(data["basis_full"], data["basis_full"], t, y, folds, seed+rep)
                estimates["Oracle AIPW (observes U)"] = float(_scores(t, y, f0, f1, fe)["AIPW"].mean())
            for name, estimate in estimates.items():
                rows.append({"experiment": kind, "strength": strength, "rep": rep, "method": name,
                             "estimate": estimate, "truth": 1.0, "effective_sample_size": min(ess_by_arm)})
            if kind == "Overlap":
                for clip in clips:
                    for name in ("IPW", "AIPW"):
                        rows.append({"experiment": "Clipping", "strength": strength, "rep": rep,
                                     "method": f"{name}; clip={clip:g}", "estimate": float(_scores(t, y, mu0, mu1, e, clip)[name].mean()),
                                     "truth": 1.0, "effective_sample_size": min(ess_by_arm)})
    draws = pd.DataFrame(rows)
    return {"draws": draws, "summary": _summary(draws, ["experiment", "strength", "method"]),
            "diagnostics": pd.DataFrame(diagnostics)}


def omitted_variable_bias(se: float, dof: int, r2_treatment, r2_outcome):
    """Maximum absolute OLS coefficient bias at specified partial-R² strengths."""
    rt, ry = np.asarray(r2_treatment), np.asarray(r2_outcome)
    if se <= 0 or dof <= 1 or np.any((rt < 0) | (rt >= 1)) or np.any((ry < 0) | (ry > 1)):
        raise ValueError("Require se>0, dof>1, 0<=treatment R²<1, 0<=outcome R²<=1.")
    return se*np.sqrt(dof*rt*ry/(1-rt))

This code implements a series of experiments to assess the robustness and sensitivity of causal inference methods to various factors, including model misspecification, confounding bias, and data quality issues.

The `run_double_robustness_experiment` function conducts a four-cell experiment where it systematically varies whether the outcome and propensity models are correctly specified (using a nonlinear basis expansion) or incorrectly specified (using only linear covariates). It then evaluates the performance of different estimators (outcome regression, IPW, AIPW) under these different conditions. The function calculates coverage probabilities for AIPW's influence-score interval to assess its ability to provide valid confidence intervals.

The `run_sensitivity_sweeps` function performs a series of sensitivity analyses by varying the strength of overlap between treatment groups, the degree of hidden confounding, and the level of propensity score clipping. It evaluates the performance of different estimators under these different conditions and calculates diagnostic statistics such as effective sample size (ESS) and propensity score ranges. The function also includes an oracle AIPW estimator that uses a hidden confounder in both nuisance models to assess the potential bias introduced by unobserved confounding.

The `omitted_variable_bias` function calculates the maximum absolute OLS coefficient bias due to omitted variable bias, based on Cinelli & Hazlett's identities. This provides a theoretical benchmark for assessing the sensitivity of OLS estimates to unobserved confounders.

In summary, this code provides a comprehensive set of tools for evaluating the robustness and sensitivity of causal inference methods. It simulates data with varying degrees of complexity and uses cross-fitting to reduce bias in estimation. The code also includes functions for calculating various performance metrics, diagnostic statistics, and theoretical benchmarks.

In [ ]:
def robustness_value(estimate: float, se: float, dof: int) -> float:
    """Equal-strength partial-R² required to reduce the OLS point estimate to zero."""
    if se <= 0 or dof <= 1:
        raise ValueError("Require positive standard error and dof>1.")
    f2 = (estimate/se)**2/dof
    return float((np.sqrt(f2**2+4*f2)-f2)/2)


def linear_sensitivity_analysis(
    X, treatment, outcome, r2_grid=None, feature_names=None,
) -> dict:
    """Formal sensitivity of the coefficient in Y~1+T+X (homoskedastic OLS).

    A causal ATE interpretation additionally requires the constant/additive
    treatment-effect model and causal identification assumptions. For nonlinear
    models this is a linear-projection coefficient, not general model CATE.
    Benchmarks show observed covariate partial R²s when omitted one at a time;
    they are reference scales, not automatic bounds on a missing confounder.
    """
    x = np.asarray(X, dtype=float)
    if x.ndim == 1:
        x = x[:, None]
    t, y = np.asarray(treatment, dtype=float), np.asarray(outcome, dtype=float)
    names = list(feature_names) if feature_names is not None else [f"X{j}" for j in range(x.shape[1])]
    if len(names) != x.shape[1]:
        raise ValueError("feature_names must match X columns.")
    design = np.column_stack([np.ones(len(y)), t, x])
    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError("Sensitivity requires a full-rank OLS design.")
    fitted = sm.OLS(y, design).fit()
    estimate, se, dof = float(fitted.params[1]), float(fitted.bse[1]), int(fitted.df_resid)
    grid = np.linspace(0, 0.5, 41) if r2_grid is None else np.asarray(r2_grid)
    rt, ry = np.meshgrid(grid, grid)
    rt, ry = rt.ravel(), ry.ravel()
    bias = omitted_variable_bias(se, dof, rt, ry)
    adjusted = estimate - (1 if estimate >= 0 else -1)*bias
    adjusted_se = se*np.sqrt((1-ry)/(1-rt)*dof/(dof-1))
    critical = student_t.ppf(0.975, dof-1)
    contours = pd.DataFrame({"r2_treatment": rt, "r2_outcome": ry, "max_absolute_bias": bias,
                             "adjusted_estimate": adjusted, "adjusted_se": adjusted_se,
                             "ci_low": adjusted-critical*adjusted_se,
                             "ci_high": adjusted+critical*adjusted_se,
                             "point_sign_can_reverse": bias >= abs(estimate)})
    benchmarks = []
    treatment_full = sm.OLS(t, sm.add_constant(x)).fit()
    for j, name in enumerate(names):
        reduced_x = np.delete(x, j, axis=1)
        treatment_reduced = sm.OLS(t, np.column_stack([np.ones(len(t)), reduced_x])).fit()
        outcome_reduced = sm.OLS(y, np.column_stack([np.ones(len(t)), t, reduced_x])).fit()
        rtd = max(0.0, 1-treatment_full.ssr/treatment_reduced.ssr)
        ryd = max(0.0, 1-fitted.ssr/outcome_reduced.ssr)
        benchmarks.append({"feature": name, "r2_treatment": rtd, "r2_outcome": ryd})
    return {"estimate": estimate, "se": se, "df": dof,
            "robustness_value": robustness_value(estimate, se, dof),
            "treatment_outcome_partial_r2": (estimate/se)**2/((estimate/se)**2+dof),
            "contours": contours, "benchmarks": pd.DataFrame(benchmarks),
            "estimand": "Treatment coefficient in additive OLS; ATE only under the stated causal constant-effect model.",
            "source": "https://carloscinelli.com/sensemakr/reference/sensemakr.html"}

This code provides tools for conducting sensitivity analyses of linear regression models, specifically focusing on assessing the potential impact of omitted variable bias.

The `robustness_value` function calculates the amount of shared variance (partial R²) between an omitted confounder and both the treatment and outcome variables that would be required to reduce the observed coefficient in a linear regression model to zero. This provides a measure of how sensitive the estimated effect is to unobserved confounding.

The `linear_sensitivity_analysis` function performs a more comprehensive sensitivity analysis by systematically varying the partial R² between omitted confounders and the treatment and outcome variables. It calculates the resulting bias in the estimated coefficient, adjusted estimates, confidence intervals, and assesses whether the sign of the point estimate can be reversed due to confounding. The function also provides benchmark values based on observed covariate partial R²s when each covariate is omitted one at a time from the model.

Here's a breakdown of key features:

*   **Omitted Variable Bias Calculation:** Uses Cinelli & Hazlett’s identities to calculate the maximum absolute bias due to omitted variable confounding.
*   **Sensitivity Surface:** Generates a sensitivity surface showing how the estimated coefficient changes as a function of the partial R² between omitted confounders and the treatment and outcome variables.
*   **Confidence Intervals:** Calculates adjusted confidence intervals that account for potential confounding bias.
*   **Benchmark Values:** Provides benchmark values based on observed covariate partial R²s to provide context for interpreting the sensitivity analysis results.
*   **Clear Documentation:** Includes detailed documentation explaining the assumptions underlying the analysis and the interpretation of the results.

In summary, this code provides a powerful set of tools for assessing the robustness of linear regression models to omitted variable bias. It allows researchers to systematically explore the potential impact of unobserved confounding and identify conditions under which the estimated effect may be sensitive to bias.

## Interference helpers

A known randomized saturation design gives the exposure probabilities used below.

In [ ]:
"""Partial-interference experiment with an explicit, supported exposure mapping.

Clusters are independent; all units in a cluster are peers. A cluster is assigned
a Bernoulli saturation q, then its units receive treatment independently with
probability q. Exposure G is 1 if any *other* member is treated. Known joint
P(T=t,G=g) identifies four mean potential outcomes. Cluster score uncertainty
respects shared assignments, outcomes, and exposure. Merely changing standard
errors on an ordinary outcome model does not repair its estimand.

References: Hudgens & Halloran (2008), https://pmc.ncbi.nlm.nih.gov/articles/PMC2600548/
Aronow & Samii, https://arxiv.org/abs/1305.6156
"""






def exposure_probabilities(cluster_size: int, saturations=CFG.INTERFERENCE_SATURATIONS, weights=None) -> dict:
    """Analytic joint exposure probabilities, averaging over saturation assignment.

    Endpoints 0 and 1 are allowed, but may leave exposures unsupported. Estimators
    explicitly reject a requested contrast with zero assignment probability.
    """
    q = np.asarray(saturations, dtype=float)
    w = np.full(len(q), 1/len(q)) if weights is None else np.asarray(weights, dtype=float)
    if (cluster_size < 2 or q.ndim != 1 or not len(q) or np.any((q < 0) | (q > 1))
            or w.shape != q.shape or np.any(w < 0) or not np.isclose(w.sum(), 1)):
        raise ValueError("Require cluster_size>=2, saturations in [0,1], and nonnegative weights summing to one.")
    p_none = (1-q)**(cluster_size-1)
    return {(t, g): float(np.sum(w*(q if t else 1-q)*(1-p_none if g else p_none)))
            for t, g in CFG.EXPOSURES}


def simulate_interference(
    n_clusters: int = 400, cluster_size: int = 4, spillover: float = 1.5,
    interaction: float = 0.3, seed: int=CFG.SEED,
    saturations=CFG.INTERFERENCE_SATURATIONS, saturation_weights=None,
    noise_sd: float = 0.6,
) -> dict:
    """Generate Y(t,g)=mu(X)+tau(X)t+gamma(X)g+interaction*t*g+epsilon.

    mu includes a shared cluster shock. No cross-cluster spillovers occur. Truth
    columns remain evaluation-only. To feed ordinary CATE methods use x plus
    saturation (or its indicators), never realized G as a baseline covariate.
    The latter is a post-assignment exposure, with a separate causal role.
    """
    if n_clusters < 3 or noise_sd < 0:
        raise ValueError("Require at least three independent clusters and nonnegative noise.")
    probabilities = exposure_probabilities(cluster_size, saturations, saturation_weights)
    q = np.asarray(saturations, dtype=float)
    weights = np.full(len(q), 1/len(q)) if saturation_weights is None else np.asarray(saturation_weights)
    rng = np.random.default_rng(seed)
    n = n_clusters*cluster_size
    cluster = np.repeat(np.arange(n_clusters), cluster_size)
    x = rng.normal(size=n)
    cluster_shock = np.repeat(rng.normal(0, 0.35, n_clusters), cluster_size)
    saturation = np.repeat(rng.choice(q, n_clusters, p=weights), cluster_size)
    treatment = rng.binomial(1, saturation)
    cluster_treated = treatment.reshape(n_clusters, cluster_size).sum(axis=1)
    exposure = (np.repeat(cluster_treated, cluster_size)-treatment > 0).astype(int)
    mu = 1+0.4*x+cluster_shock
    tau = 0.8+0.2*x
    gamma = spillover*(1+0.15*x)
    epsilon = rng.normal(0, noise_sd, n)
    outcome = mu+tau*treatment+gamma*exposure+interaction*treatment*exposure+epsilon
    frame = pd.DataFrame({"cluster": cluster, "x": x, "saturation": saturation,
                          "treatment": treatment, "exposure": exposure, "outcome": outcome,
                          "mu_truth": mu, "tau_truth": tau, "gamma_truth": gamma,
                          "interaction_truth": interaction})
    truth = {"Direct, no treated peer": float(tau.mean()),
             "Direct, any treated peer": float(tau.mean()+interaction),
             "Spillover on untreated": float(gamma.mean()),
             "Spillover on treated": float(gamma.mean()+interaction),
             "Total (1,1) versus (0,0)": float(tau.mean()+gamma.mean()+interaction)}
    return {"data": frame, "frame": frame, "truth": truth,
            "probabilities": probabilities, "cluster_size": cluster_size,
            "saturations": q, "saturation_weights": weights,
            "assumption": "Independent clusters; any-treated-peer exposure mapping is sufficient."}

This code simulates data for a partial interference experiment, where the treatment effect on one individual can be influenced by the treatment status of their peers within a cluster. It focuses on modeling direct and spillover effects in a network setting.

The `exposure_probabilities` function calculates the joint probabilities of different exposure (G) and treatment (T) combinations for each unit within a cluster, given a saturation level (q). The saturation level represents the probability that any individual within the cluster receives treatment. This function is crucial for understanding the causal structure of the experiment.

The `simulate_interference` function generates synthetic data with partial interference. It simulates clusters of individuals where treatment assignment depends on a saturation level and exposure is defined as whether any other member of the cluster received treatment. The outcome is modeled as a function of individual characteristics (x), treatment, exposure, and an interaction term between treatment and exposure.

Here's a breakdown of key features:

*   **Partial Interference:** Models a scenario where the effect of treatment on one individual depends on the treatment status of their peers within a cluster.
*   **Explicit Exposure Mapping:** Defines exposure as whether any other member of the cluster received treatment, providing a clear causal structure.
*   **Cluster Structure:** Simulates data with independent clusters, ensuring that there is no cross-cluster interference.
*   **Known Causal Effects:** Generates data with known direct and spillover effects, allowing for evaluation of causal inference methods.
*   **Detailed Documentation:** Includes detailed documentation explaining the assumptions underlying the simulation and the interpretation of the results.

In summary, this code provides a powerful tool for simulating data in settings with partial interference. It allows researchers to explore the challenges of estimating causal effects in network settings and evaluate the performance of different causal inference methods under realistic conditions.


In [ ]:
def _cluster_se(influence: np.ndarray, clusters: np.ndarray) -> float:
    """SE of the unit mean using independent equal-sized cluster score means."""
    frame = pd.DataFrame({"score": influence, "cluster": clusters})
    sizes = frame.groupby("cluster", sort=False).size()
    if sizes.nunique() != 1:
        raise ValueError("This tutorial variance formula requires equal-size clusters.")
    means = frame.groupby("cluster", sort=False)["score"].mean().to_numpy()
    return float(means.std(ddof=1)/np.sqrt(len(means)))


def _exposure_scores(frame: pd.DataFrame, probabilities: dict) -> dict:
    y = frame["outcome"].to_numpy()
    t, g = frame["treatment"].to_numpy(), frame["exposure"].to_numpy()
    scores = {}
    for exposure in CFG.EXPOSURES:
        p = probabilities[exposure]
        if p <= 0:
            scores[exposure] = None
            continue
        weight = ((t == exposure[0]) & (g == exposure[1])).astype(float)/p
        mean_weight = weight.mean()
        ht = weight*y
        # A supported exposure can still be absent by chance in a small sample.
        # HT is defined; Hájek is undefined rather than silently divided by zero.
        hajek = ht.mean()/mean_weight if mean_weight > 0 else np.nan
        hajek_if = weight*(y-hajek)/mean_weight if mean_weight > 0 else np.full(len(y), np.nan)
        scores[exposure] = {"ht": ht, "hajek": hajek, "hajek_if": hajek_if,
                            "n": int(np.sum(weight > 0)), "probability": p}
    return scores


def _estimate_contrast(frame: pd.DataFrame, scores: dict, coefficients: dict) -> dict:
    unsupported = [exposure for exposure, value in coefficients.items()
                   if value != 0 and scores[exposure] is None]
    if unsupported:
        raise ValueError(f"Requested contrast has unsupported exposures: {unsupported}")
    n = len(frame)
    ht_score, hajek_if = np.zeros(n), np.zeros(n)
    hajek = 0.0
    for exposure, value in coefficients.items():
        if value == 0:
            continue
        score = scores[exposure]
        ht_score += value*score["ht"]
        hajek += value*score["hajek"]
        hajek_if += value*score["hajek_if"]
    ht = float(ht_score.mean())
    cluster = frame["cluster"].to_numpy()
    se = _cluster_se(ht_score, cluster)
    hse = _cluster_se(hajek_if, cluster) if np.isfinite(hajek) else np.nan
    critical = student_t.ppf(0.975, frame["cluster"].nunique()-1)
    return {"ht_estimate": ht, "ht_se": se, "ht_ci_low": ht-critical*se,
            "ht_ci_high": ht+critical*se, "hajek_estimate": float(hajek),
            "hajek_se": hse, "hajek_ci_low": hajek-critical*hse,
            "hajek_ci_high": hajek+critical*hse}


def estimate_exposure_contrasts(simulation: dict) -> pd.DataFrame:
    """HT and Hájek direct, spillover, and total effects with cluster uncertainty."""
    frame = simulation["data"]
    scores = _exposure_scores(frame, simulation["probabilities"])
    return pd.DataFrame([dict(contrast=name, truth=simulation["truth"][name],
                              **_estimate_contrast(frame, scores, coefficients))
                         for name, coefficients in CFG.CONTRASTS.items()])


def policy_exposure_probabilities(saturation: float, cluster_size: int) -> dict:
    """Exposure probabilities under a new independent Bernoulli policy."""
    return exposure_probabilities(cluster_size, [saturation], [1.0])


def evaluate_saturation_policies(
    simulation: dict, policy_saturations=(0.0, 0.1, 0.25, 0.5, 0.75, 1.0),
    treatment_cost: float = 0.1,
) -> pd.DataFrame:
    """Incremental per-person welfare versus treating nobody.

    Estimation transports supported exposure means, not a regression trained at
    one saturation. A new q is identifiable here because outcomes depend only
    on (T,G,X) and all relevant exposures have support. Without that mapping,
    transporting to untested saturations need not be justified. The expected
    budget is q*N*cost; Bernoulli assignment does not enforce an exact headcount.
    """
    frame = simulation["data"]
    scores = _exposure_scores(frame, simulation["probabilities"])
    # Condition on randomized saturation to remove own/peer assignment association.
    # This leaves a trial-design-averaged DIRECT contrast, not total policy welfare.
    saturation_dummies = pd.get_dummies(frame["saturation"], drop_first=True, dtype=float)
    design = np.column_stack([np.ones(len(frame)), frame["treatment"], frame["x"], saturation_dummies])
    ordinary = sm.OLS(frame["outcome"].to_numpy(), design).fit(
        cov_type="cluster", cov_kwds={"groups": frame["cluster"].to_numpy()})
    ordinary_effect = float(ordinary.params[1])
    rows = []
    for q in policy_saturations:
        target = policy_exposure_probabilities(float(q), simulation["cluster_size"])
        coefficients = target.copy()
        coefficients[(0, 0)] -= 1.0
        estimates = _estimate_contrast(frame, scores, coefficients)
        r = 1-(1-q)**(simulation["cluster_size"]-1)
        truth = float(q*frame["tau_truth"].mean()+r*frame["gamma_truth"].mean()
                      +q*r*frame["interaction_truth"].mean())
        rows.append({"saturation": q, "truth_increment": truth,
                     "naive_additive_increment": q*ordinary_effect,
                     "oracle_direct_only_increment": q*frame["tau_truth"].mean(),
                     "ordinary_direct_estimate": ordinary_effect,
                     "ordinary_direct_cluster_se": float(ordinary.bse[1]),
                     "expected_cost_per_person": q*treatment_cost,
                     "truth_net_increment": truth-q*treatment_cost,
                     "ht_net_increment": estimates["ht_estimate"]-q*treatment_cost,
                     **estimates})
    return pd.DataFrame(rows)

This code implements a detailed analysis of partial interference in a network setting, focusing on estimating direct and spillover effects of treatment. It builds upon the data simulation framework to evaluate different causal inference methods and policy interventions.

The `_cluster_se` function calculates the standard error of the mean for cluster-level scores, assuming equal-sized clusters. This is used to account for the correlation within clusters when estimating treatment effects.

The `_exposure_scores` function calculates Hájek's estimator and Horvitz-Thompson (HT) estimators for different exposure combinations. These estimators are used to estimate the average treatment effect under interference, accounting for the fact that individuals’ outcomes may be affected by their peers’ treatment status.

The `_estimate_contrast` function estimates the contrast between different exposure groups using the calculated scores and specified coefficients. It calculates point estimates and confidence intervals for the estimated effects.

The `estimate_exposure_contrasts` function estimates direct, spillover, and total effects of treatment by applying the above functions to the simulated data. It returns a DataFrame containing the estimated effects and their associated standard errors and confidence intervals.

The `policy_exposure_probabilities` function calculates exposure probabilities under a new independent Bernoulli policy with a specified saturation level.

The `evaluate_saturation_policies` function evaluates different policies for setting the saturation level (the probability of treatment within each cluster). It estimates the incremental per-person welfare associated with each policy, taking into account both the benefits of treatment and the costs of intervention. The function also compares the estimated effects to a naive additive model and an oracle estimate based on known causal parameters.

In summary, this code provides a comprehensive framework for analyzing partial interference in network settings. It implements various estimation techniques, evaluates different policies, and provides detailed diagnostics for assessing the robustness of the results.

In [ ]:
def evaluate_locked_individual_policy(
    simulation: dict, decisions, predicted_cate=None, treatment_cost: float = 0.1,
) -> dict:
    """Evaluate any frozen exact-budget targeting policy on independent clusters.

    Fit the policy (including budget/threshold selection) outside these clusters.
    Decisions may use their baseline covariates but NOT their assignment/outcome.
    This helper accepts CausalFM/CausalPFN or classical rankings on equal terms.
    Known exposure probabilities supply the same HT evaluator for every policy.
    The simulator's tau/gamma are accessed only to report evaluation truth.
    """
    frame = simulation["data"]
    decisions = np.asarray(decisions)
    if decisions.shape != (len(frame),) or not np.isin(decisions, [0, 1]).all():
        raise ValueError("decisions must be one binary decision per evaluation unit.")
    if treatment_cost < 0:
        raise ValueError("Treatment cost must be nonnegative.")
    cluster_counts = pd.Series(decisions, index=frame.index).groupby(frame["cluster"]).transform("sum")
    target_g = (cluster_counts.to_numpy()-decisions > 0).astype(int)
    actual_t, actual_g = frame["treatment"].to_numpy(), frame["exposure"].to_numpy()
    y = frame["outcome"].to_numpy()
    probabilities = simulation["probabilities"]
    coefficients = {e: ((decisions == e[0]) & (target_g == e[1])).astype(float)
                    - (1.0 if e == (0, 0) else 0.0) for e in CFG.EXPOSURES}
    score = np.zeros(len(frame))
    for exposure, coefficient in coefficients.items():
        p = probabilities[exposure]
        if np.any(coefficient != 0) and p <= 0:
            raise ValueError(f"Policy requires unsupported exposure {exposure}.")
        if p > 0:
            observed = (actual_t == exposure[0]) & (actual_g == exposure[1])
            score += coefficient*observed*y/p
    direct = float(np.mean(decisions*frame["tau_truth"].to_numpy()))
    spillover = float(np.mean(target_g*frame["gamma_truth"].to_numpy()))
    interaction = float(np.mean(decisions*target_g*frame["interaction_truth"].to_numpy()))
    cost = float(decisions.mean()*treatment_cost)
    se = _cluster_se(score, frame["cluster"].to_numpy())
    critical = student_t.ppf(0.975, frame["cluster"].nunique()-1)
    predicted = np.asarray(predicted_cate) if predicted_cate is not None else None
    if predicted is not None and (predicted.shape != decisions.shape or not np.isfinite(predicted).all()):
        raise ValueError("predicted_cate must be one finite value per evaluation unit.")
    estimate = float(score.mean())
    return {"treated_count": int(decisions.sum()), "treated_fraction": float(decisions.mean()),
            "direct_truth": direct, "spillover_truth": spillover, "interaction_truth": interaction,
            "truth_increment": direct+spillover+interaction,
            "truth_net_increment": direct+spillover+interaction-cost,
            "naive_predicted_increment": float(np.mean(decisions*predicted)) if predicted is not None else np.nan,
            "ht_estimate": estimate, "ht_se": se,
            "ht_ci_low": estimate-critical*se, "ht_ci_high": estimate+critical*se,
            "ht_net_increment": estimate-cost, "cost_per_person": cost}


def run_interference_experiment(
    n_clusters: int = 400, cluster_size: int = 4, seed: int=CFG.SEED,
    spillovers=(0.0, 1.5, -1.0), interaction: float = 0.0,
    saturations=CFG.INTERFERENCE_SATURATIONS,
    policy_saturations=(0.0, 0.1, 0.25, 0.5, 0.75, 1.0),
) -> dict[str, pd.DataFrame]:
    """Paired assignment draws with zero, positive, and negative spillovers.

    interaction defaults to zero so the zero-spillover panel actually satisfies
    no-interference. Set interaction nonzero for an additional SUTVA violation.
    """
    contrasts, policies, diagnostics, data = [], [], [], []
    for spillover in spillovers:
        simulation = simulate_interference(n_clusters, cluster_size, spillover,
                                            interaction, seed, saturations)
        contrasts.append(estimate_exposure_contrasts(simulation).assign(spillover=spillover))
        policies.append(evaluate_saturation_policies(simulation, policy_saturations).assign(spillover=spillover))
        frame = simulation["data"]
        data.append(frame.assign(spillover=spillover))
        for exposure, probability in simulation["probabilities"].items():
            mask = (frame["treatment"] == exposure[0]) & (frame["exposure"] == exposure[1])
            diagnostics.append({"spillover": spillover, "treatment": exposure[0], "exposure": exposure[1],
                                "known_probability": probability, "observed_count": int(mask.sum()),
                                "expected_count": len(frame)*probability,
                                "clusters_with_exposure": frame.loc[mask, "cluster"].nunique()})
    return {"contrasts": pd.concat(contrasts, ignore_index=True),
            "policy_values": pd.concat(policies, ignore_index=True),
            "exposure_diagnostics": pd.DataFrame(diagnostics),
            "data": pd.concat(data, ignore_index=True)}

This code implements a comprehensive experiment to evaluate the effects of partial interference in a network setting. It simulates data with varying degrees of spillover and interaction effects, estimates causal contrasts using different methods, and evaluates the performance of various policies for targeting treatment.

The `evaluate_locked_individual_policy` function evaluates a pre-defined policy (a set of binary decisions indicating which individuals receive treatment) on independent clusters. It calculates the estimated effect of the policy using Horvitz-Thompson (HT) estimators, accounting for the exposure probabilities and cluster structure. The function also provides estimates of direct, spillover, and interaction effects based on known causal parameters from the simulation.

The `run_interference_experiment` function orchestrates the entire experiment by simulating data with different levels of spillover and interaction, estimating causal contrasts using `estimate_exposure_contrasts`, evaluating saturation policies using `evaluate_saturation_policies`, and collecting diagnostic statistics about exposure probabilities. It returns a dictionary containing DataFrames summarizing the results of the experiment.

Here's a breakdown of key features:

*   **Partial Interference Simulation:** Simulates data with varying degrees of spillover effects, where the treatment effect on one individual can be influenced by the treatment status of their peers within a cluster.
*   **Exposure Mapping:** Defines exposure as whether any other member of the cluster received treatment, providing a clear causal structure.
*   **Policy Evaluation:** Evaluates different policies for targeting treatment based on saturation levels (the probability of treatment within each cluster).
*   **Causal Contrast Estimation:** Estimates direct, spillover, and total effects of treatment using HT estimators.
*   **Diagnostic Statistics:** Collects diagnostic statistics about exposure probabilities to assess the validity of the simulation and estimation methods.

In summary, this code provides a powerful framework for studying partial interference in network settings. It allows researchers to explore the challenges of estimating causal effects in complex social systems and evaluate the performance of different policies under realistic conditions.

## Foundation-model stress tests

In [ ]:
"""Run both real foundation models through matched identification stress tests.

Small demo counts are descriptive demonstrations, not a reliable model ranking
or interval-coverage study. The interference panel deliberately asks ordinary
individual CATE models the wrong policy question and measures the resulting gap.
"""






def _model_factories(seed):
    return {
        "Interaction OLS": lambda: InteractionOLS(),
        "DR-learner": lambda: DRLearner(seed=seed),
        "CausalFM": lambda: CausalFMAdapter(seed=seed, device=CFG.DEVICE),
        "CausalPFN": lambda: CausalPFNAdapter(seed=seed, device=CFG.DEVICE),
    }


def _stress_fit_predict(model, X, T, Y, query):
    synchronize_device(getattr(model, "device", "cpu"))
    start = time.perf_counter()
    model.fit(X, T, Y)
    synchronize_device(getattr(model, "device", "cpu"))
    fit_seconds = time.perf_counter()-start
    synchronize_device(getattr(model, "device", "cpu"))
    start = time.perf_counter()
    prediction = np.asarray(model.predict_cate(query), dtype=float).reshape(-1)
    synchronize_device(getattr(model, "device", "cpu"))
    predict_seconds = time.perf_counter()-start
    if prediction.shape != (len(query),) or not np.isfinite(prediction).all():
        raise ValueError("Invalid model CATE output in stress experiment.")
    return prediction, {"device": getattr(model, "device", "cpu"), "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
                        "total_seconds": fit_seconds+predict_seconds,
                        "n_train": len(X), "n_query": len(query)}


def run_model_stress(mode=CFG.MODE, seed=CFG.SEED, output_dir=None) -> dict:
    """Compare four estimators under observed/hidden bias and peer interference.

    Demo: two paired data repetitions; context=384, query=96; CausalPFN native
    interval sample=500 and query=32. Full: 20 repetitions, context=1024,
    query=256, interval query=64 and samples=2000. Query outcomes/truth never
    enter model fit, normalization, hyperparameters, or context selection.

    Ordinary interference models see ONLY baseline x. They do not receive
    randomized saturation or peer exposure, so their fitted contrast has no
    asserted direct-effect interpretation. The prediction's mean is deliberately
    used as a naive full-rollout forecast and compared with E[Y(1,1)-Y(0,0)].
    Models are neither modified nor represented as interference-aware.
    """
    if mode not in ("demo", "reference"):
        raise ValueError("mode must be 'demo' or 'reference'.")
    workload = CFG.WORKLOADS[mode]
    n, n_query = workload['stress_train_n'], workload['stress_query_n']
    repetitions = workload['stress_repetitions']
    interval_query, samples = workload['interval_query_n'], workload['interval_samples']
    config = {"mode": mode, "seed": seed, "n_train": n, "n_query": n_query,
              "repetitions": repetitions, "interval_query": interval_query,
              "interval_samples": samples, "scenario_order": ["baseline", "weak_overlap", "hidden_confounding"],
              "models": list(_model_factories(seed)), "foundation_device": CFG.DEVICE}
    manifest = {"status": "running", "experiment": "model_stress", "config": config,
                "config_sha256": hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest(),
                "started_utc": datetime.now(timezone.utc).isoformat(),
                "matched_training_rows": True, "query_outcomes_used_for_training": False,
                "interval_semantics": "Native CausalPFN intervals checked against mean CATE; empirical coverage is descriptive, not a guarantee.",
                "interval_replication_warning": "Query coverages within a fitted model are dependent; two fits do not establish population coverage.",
                "interference_features": ["baseline x"],
                "interference_warning": "Deliberate wrong-estimand full-rollout forecast; none of these four models is interference-aware.",
                "pretraining_runtime_included": False, "checkpoint_load_in_fit_time": True,
                "failures": []}
    metrics, intervals, interference_models = [], [], []
    started = time.perf_counter()

    def save():
        if output_dir is None:
            return
        directory = Path(output_dir)
        directory.mkdir(parents=True, exist_ok=True)
        for name, rows in (("metrics", metrics), ("intervals", intervals),
                           ("interference_models", interference_models)):
            pd.DataFrame(rows).to_csv(directory / f"stress_{name}.csv", index=False)
        (directory / "stress_manifest.json").write_text(json.dumps(manifest, indent=2)+"\n")

    context = {}
    try:
        for rep in range(repetitions):
            draw_seed = seed+1000*rep
            for scenario, overlap, hidden in (("baseline", 1.0, 0.0),
                                              ("weak_overlap", 3.0, 0.0),
                                              ("hidden_confounding", 1.0, 1.5)):
                train = simulate_robustness_data(n, draw_seed, overlap, hidden)
                query = simulate_robustness_data(n_query, draw_seed+100, overlap, hidden)
                # tau=1+.3*X0-.2*X1 does not depend on hidden U: tau(X) is
                # precisely the learner-visible conditional mean effect target.
                truth = query["tau"]
                for name, factory in _model_factories(draw_seed).items():
                    context = {"scenario": scenario, "rep": rep, "model": name}
                    model = factory()
                    prediction, timing = _stress_fit_predict(model, train["X"], train["T"], train["Y"], query["X"])
                    metrics.append({**context, "cate_rmse": float(np.sqrt(np.mean((prediction-truth)**2))),
                                    "ate_bias": float(prediction.mean()-truth.mean()),
                                    "truth_ate": float(truth.mean()), "predicted_ate": float(prediction.mean()),
                                    "train_propensity_min": float(train["e"].min()),
                                    "train_propensity_max": float(train["e"].max()),
                                    **timing})
                    if name == "CausalPFN":
                        interval_start = time.perf_counter()
                        native = model.predict_interval(query["X"][:interval_query], alpha=0.05, n_samples=samples)
                        lower, upper = native["lower_bound"], native["upper_bound"]
                        if (lower.shape != (interval_query,) or upper.shape != (interval_query,)
                                or not np.isfinite(lower).all() or not np.isfinite(upper).all()
                                or np.any(lower > upper)):
                            raise ValueError("CausalPFN returned invalid native intervals.")
                        target = truth[:interval_query]
                        intervals.append({**context, "coverage": float(np.mean((target >= lower) & (target <= upper))),
                                          "mean_width": float(np.mean(upper-lower)), "nominal_level": 0.95,
                                          "n_query": interval_query, "n_samples": samples,
                                          "interval_seconds": time.perf_counter()-interval_start,
                                          "interval_target_check": "Known mean CATE tau(X)",
                                          "coverage_claim": "Descriptive fraction; correlated queries and very few fits"})
                    del model
                    gc.collect()
            # Split complete clusters. All models see the same rows and one
            # baseline covariate. Spillover scenarios reuse assignment draws.
            cluster_size = CFG.INTERFERENCE_CLUSTER_SIZE
            for spillover in CFG.INTERFERENCE_SPILLOVERS:
                simulated = simulate_interference(n_clusters=(n+n_query)//cluster_size,
                                                  cluster_size=cluster_size, spillover=spillover,
                                                  interaction=CFG.INTERFERENCE_INTERACTION, seed=draw_seed+200)
                frame = simulated["data"]
                train, query = frame.iloc[:n], frame.iloc[n:]
                assert not set(train["cluster"]) & set(query["cluster"])
                truth = float((query["tau_truth"]+query["gamma_truth"]+query["interaction_truth"]).mean())
                for name, factory in _model_factories(draw_seed).items():
                    context = {"scenario": "interference", "spillover": spillover, "rep": rep, "model": name}
                    model = factory()
                    prediction, timing = _stress_fit_predict(model, train[["x"]].to_numpy(),
                                                     train["treatment"].to_numpy(), train["outcome"].to_numpy(),
                                                     query[["x"]].to_numpy())
                    interference_models.append({**context,
                                                "predicted_full_rollout_increment": float(prediction.mean()),
                                                "truth_full_rollout_increment": truth,
                                                "error": float(prediction.mean()-truth),
                                                "truth_direct_only": float(query["tau_truth"].mean()),
                                                "pretreatment_features": "x only",
                                                "estimand_warning": "Naive additive forecast; not an identified total policy effect",
                                                **timing})
                    del model
                    gc.collect()
            save()
    except Exception as error:
        manifest["status"] = "failed"
        manifest["failures"].append({**context, "exception": repr(error)})
        manifest["elapsed_seconds"] = time.perf_counter()-started
        save()
        raise
    manifest["status"] = "complete"
    manifest["elapsed_seconds"] = time.perf_counter()-started
    manifest["completed_utc"] = datetime.now(timezone.utc).isoformat()
    save()
    return {"metrics": pd.DataFrame(metrics), "intervals": pd.DataFrame(intervals),
            "interference_models": pd.DataFrame(interference_models), "manifest": manifest}

This code implements a comprehensive stress test for various causal inference models, evaluating their performance under different conditions of confounding, model misspecification, and partial interference. It focuses on comparing the performance of both traditional machine learning methods (Interaction OLS, DR-learner) and foundation models (CausalFM, CausalPFN).

The `_model_factories` function creates a dictionary mapping model names to factory functions that can be used to instantiate the corresponding models.

The `_stress_fit_predict` function fits a given model to training data and predicts causal effects on a query set, measuring the time taken for both fitting and prediction. It also performs validation checks on the predictions.

The `run_model_stress` function orchestrates the entire stress test experiment. It simulates data with varying degrees of confounding and interference, trains each model on the simulated data, and evaluates its performance using various metrics, including RMSE, ATE bias, and coverage probability for confidence intervals (for CausalPFN).

Here's a breakdown of key features:

*   **Robustness Checks:** Evaluates models under different scenarios of confounding (baseline, weak overlap, hidden confounding) to assess their robustness to model misspecification.
*   **Partial Interference Simulation:** Simulates data with partial interference, where the treatment effect on one individual can be influenced by the treatment status of their peers within a cluster.
*   **Foundation Model Evaluation:** Includes evaluation of foundation models (CausalFM and CausalPFN) alongside traditional machine learning methods.
*   **Confidence Interval Coverage:** Estimates coverage probabilities for confidence intervals generated by CausalPFN to assess the reliability of its uncertainty estimates.
*   **Detailed Logging:** Logs all relevant information about the experiment, including configuration parameters, timing information, and error messages.

The code also includes detailed documentation explaining the assumptions underlying the simulation and the interpretation of the results. It emphasizes that the demo counts are descriptive demonstrations and not a reliable model ranking or interval-coverage study. The interference panel deliberately asks ordinary individual CATE models the wrong policy question to measure the resulting gap.

# Groundwork


## Runtime and assets

In [ ]:
begin_exports(CFG.GRAPH_DIR)
run_info = {'mode': CFG.MODE, 'reuse_completed_model_results': CFG.REUSE_RESULTS,
            'plot_style': CFG.PLOT_STYLE, 'seed': CFG.SEED, 'python': platform.python_version(),
            'platform': platform.platform(), 'torch': torch.__version__,
            'foundation_device': CFG.DEVICE, 'mps_available': torch.backends.mps.is_available(), 'torch_threads': torch.get_num_threads()}
display_table(pd.Series(run_info, name='Local execution'), 'local_execution', CFG.GRAPH_DIR)
save_json(run_info, CFG.OUT / 'runtime.json')
save_json({name: version(name) for name in [
    'numpy', 'scipy', 'pandas', 'scikit-learn', 'econml', 'statsmodels', 'matplotlib',
    'seaborn', 'torch', 'tabpfn', 'causalpfn', 'huggingface-hub', 'einops', 'networkx', 'pillow'
]}, CFG.OUT / 'runtime_packages.json')

This code snippet prepares for and initiates the export of figures and tables generated during a causal inference analysis, while also logging crucial runtime information.

First, it calls `begin_exports(CFG.GRAPH_DIR)` to initialize an export directory (defined by `CFG.GRAPH_DIR`) and set up the necessary infrastructure for saving visualizations and data summaries. This function ensures that any previously existing PNG files from prior runs are removed, preventing clutter.

Next, it creates a dictionary called `run_info` containing metadata about the current execution environment. This includes information such as the mode (demo or reference), whether completed model results are being reused, the plot style, random seed, Python version, platform details, PyTorch version, device used for computations, MPS availability, and number of Torch threads.

The `display_table` function is then called to display this `run_info` dictionary as a formatted table in the notebook environment using Pandas. The table is saved as a PNG image in the export directory (`CFG.GRAPH_DIR`) with the filename "local_execution".

Finally, two JSON files are created and saved:
1.  `runtime.json`: Contains the `run_info` dictionary, providing a comprehensive record of the execution environment.
2.  `runtime_packages.json`: Contains a dictionary mapping package names (e.g., 'numpy', 'scipy') to their corresponding versions, ensuring reproducibility by documenting the software dependencies used in the analysis.

In essence, this code snippet sets up the logging and export infrastructure for the causal inference workflow, capturing essential information about the execution environment and saving it for future reference and reproducibility.

### Download missing assets

Download missing datasets and checkpoint weights; reuse files already on disk.
CausalFM's original checkpoint path is reused if present.

Optional integrity checks and provenance metadata are in `verify_assets.py`.
Run `python verify_assets.py --root /path/to/asset/root` separately after downloading
(or `python verify_assets.py` for the default location). Verification is not required to run this notebook.

In [ ]:
prepare_assets()

This single line of code calls the `prepare_assets` function, which is responsible for downloading any necessary datasets or pre-trained models that are required for the subsequent analysis.

The `prepare_assets` function iterates through lists of model assets and data assets defined in a configuration object (`CFG`). For each asset, it checks if the file already exists locally. If not, it downloads the file from a specified URL using the `ensure_asset` function. The `ensure_asset` function handles creating necessary directories, downloading the file with a temporary name to prevent incomplete files, and atomically replacing the temporary file with the final downloaded version.

In short, this line ensures that all required data and models are available before proceeding with the rest of the analysis, automatically downloading them if they are not already present on the system.

## What effect are we estimating?

In [ ]:
display_table(pd.DataFrame([
    ['Mean CATE', 'RMSE against known conditional means, only in simulations/IHDP'],
    ['Population ATE', 'Bias/coverage in repeated simulations; experiment-based estimates on real RCT data'],
    ['Targeting rule', 'Independent randomized holdout value and paired policy differences'],
    ['Individual-effect distribution', 'Its own distributional target; not automatically a CATE confidence interval']
], columns=['Output', 'How we validate it']), 'effect_targets', CFG.GRAPH_DIR)

This code snippet displays a table summarizing the different types of causal effect estimates being computed and how their validity is assessed within the analysis.

It creates a Pandas DataFrame with two columns: "Output" and "How we validate it". The rows of the DataFrame describe four key outputs: Mean CATE, Population ATE, Targeting rule, and Individual-effect distribution. For each output, the second column explains the methods used to validate its accuracy or reliability.

Specifically:
*   **Mean CATE:** Validated by calculating RMSE (Root Mean Squared Error) against known conditional means, but only in simulated data or the IHDP dataset where true values are available.
*   **Population ATE:** Assessed through bias and coverage analysis in repeated simulations, as well as experiment-based estimates on real RCT data.
*   **Targeting rule:** Evaluated using independent randomized holdout value and paired policy differences to compare different targeting strategies.
*   **Individual-effect distribution:** Validated against its own distributional target, but it's noted that this is not automatically equivalent to a CATE confidence interval.

The `display_table` function then renders this DataFrame as an interactive table in the notebook environment and saves it as a PNG image named "effect_targets" in the specified export directory (`CFG.GRAPH_DIR`). This provides a clear overview of the validation strategies used for each type of causal effect estimate, enhancing the interpretability of the results.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.1))
ax.axis('off')
positions = {'X: customer history': (.2, .7), 'T: email assignment': (.2, .15), 'Y: two-week spend': (.78, .15)}
for label, (x,y) in positions.items():
    ax.text(x, y, label, ha='center', va='center', fontsize=13,
            bbox=dict(boxstyle='round,pad=.55', fc='#edf2f5', ec='#8b9daa'))
for start,end in [((.2,.59),(.2,.27)), ((.34,.65),(.7,.26)), ((.35,.15),(.64,.15))]:
    ax.annotate('', xy=end, xytext=start, arrowprops=dict(arrowstyle='->', lw=2,color='#476276'))
ax.text(.58,.77,'Observational training: X can affect assignment.\nRandomized evaluation: assignment probability is known.',ha='center')
ax.set_xlim(0,1); ax.set_ylim(0,1)
finish_figure('identification', CFG.GRAPH_DIR)

This code generates a diagram illustrating the causal relationships and identification strategy used in the Hillstrom dataset analysis.

It creates a Matplotlib figure and axes, then turns off the axis display to create a clean visual representation. It defines positions for three key variables: customer history (X), email assignment (T), and two-week spend (Y). These labels are placed on the plot using `ax.text` with rounded boxes for emphasis.

Arrows are added using `ax.annotate` to visually represent the causal relationships between these variables. Specifically, arrows point from X to T (indicating that customer history can influence email assignment), from T to Y (indicating that email assignment affects spend), and from X directly to Y (representing a potential confounding effect).

A text box is added to explain the identification strategy: observational training allows for X to affect assignment, while randomized evaluation ensures that the assignment probability is known.

Finally, `finish_figure` is called to save the plot as a PNG image named "identification" in the specified export directory (`CFG.GRAPH_DIR`). This function also handles any necessary cleanup and ensures that the figure title is properly set.

In essence, this code creates a clear visual representation of the causal model underlying the Hillstrom analysis, highlighting the potential for confounding and the identification strategy used to address it.

## Data preparation and exploratory analysis

In [ ]:
ihdp = load_ihdp(replication=0)
hillstrom = prepare_hillstrom(seed=CFG.SEED)
data_summary = []
for dataset_name, dataset in [('IHDP', ihdp), ('Hillstrom', hillstrom)]:
    for split in ['train', 'valid', 'test']:
        if split not in dataset: continue
        d = dataset[split]
        data_summary.append({'dataset':dataset_name, 'split':split, 'rows':len(d['T']),
                             'features':d['X'].shape[1], 'treated_fraction':np.mean(d['T']),
                             'mean_outcome':np.mean(d['Y'])})
display_table(pd.DataFrame(data_summary), 'dataset_summary', CFG.GRAPH_DIR)
display(Markdown('**Hillstrom predictors:** ' + ', '.join(hillstrom['feature_names'])))
assert not any(name in ['visit', 'conversion', 'spend'] for name in hillstrom['feature_names'])
pd.DataFrame(data_summary).to_csv(CFG.OUT/'dataset_summary.csv',index=False)

This code snippet provides a summary of the datasets used in the analysis – IHDP and Hillstrom – including their sizes, feature counts, treatment prevalence, and mean outcomes for each split (train, valid, test).

First, it loads the IHDP dataset with replication 0 using `load_ihdp` and prepares the Hillstrom dataset using `prepare_hillstrom`.

Then, it iterates through both datasets and their respective splits. For each split that exists in a given dataset, it extracts information such as the number of rows (samples), the number of features, the fraction of treated units, and the mean outcome. This information is stored in a list of dictionaries called `data_summary`.

The `display_table` function is used to display this summary data as an interactive table in the notebook environment and save it as a PNG image named "dataset_summary" in the export directory (`CFG.GRAPH_DIR`).

Next, it displays a Markdown string listing the feature names from the Hillstrom dataset using `display(Markdown(...))`. An assertion is included to verify that the columns 'visit', 'conversion', and 'spend' are not present as features, ensuring data integrity.

Finally, the `data_summary` DataFrame is saved to a CSV file named "dataset_summary.csv" in the output directory (`CFG.OUT`).

In essence, this code provides a concise overview of the characteristics of the datasets used in the analysis, facilitating understanding and reproducibility.

In [ ]:
raw_hillstrom = load_hillstrom()
display_table(raw_hillstrom.groupby('segment').agg(customers=('segment','size'),
    mean_pretreatment_history=('history','mean')), 'baseline_customer_history', CFG.GRAPH_DIR)
display(Markdown('Only counts and pretreatment history are inspected here. '
                 'Model and policy selection below use only the designated validation partition.'))

This code snippet analyzes and displays baseline characteristics of customers in the Hillstrom dataset, specifically focusing on customer segment and pretreatment history.

First, it loads the raw Hillstrom dataset using `load_hillstrom`.

Then, it groups the data by 'segment' (Mens E-Mail, Womens E-Mail, No E-Mail) and calculates two aggregate statistics:
*   `customers`: The number of customers in each segment (using `.size`).
*   `mean_pretreatment_history`: The mean value of the 'history' feature for each segment.

The resulting aggregated DataFrame is then displayed as an interactive table using `display_table`, and saved as a PNG image named "baseline_customer_history" in the export directory (`CFG.GRAPH_DIR`). This provides insights into the characteristics of customers in different segments before any treatment (email assignment) occurs.

Finally, a Markdown string is displayed to clarify that this analysis only involves inspecting baseline data and does not influence model or policy selection, which are based on a separate validation partition.

In essence, this code snippet provides a descriptive overview of the customer base in the Hillstrom dataset, highlighting differences in pretreatment history across different segments. It emphasizes that this information is used for exploratory purposes only and does not affect the subsequent modeling and policy evaluation steps.

## Construct an observational training cohort from Hillstrom

In [ ]:
selected_pool = select_observational(hillstrom['train'], strength=CFG.SELECTION_STRENGTH, seed=CFG.SEED+17)
balance, balance_stats = balance_diagnostics(selected_pool['X'], selected_pool['T'], selected_pool['e'],
                                            feature_names=hillstrom['feature_names'])
display_table(pd.Series(balance_stats, name='Known selection propensity diagnostics'), 'propensity_diagnostics', CFG.GRAPH_DIR)
display_table(balance.head(12), 'covariate_balance', CFG.GRAPH_DIR)
plt.figure(figsize=(9,4))
for arm in [0,1]:
    plt.hist(selected_pool['e'][selected_pool['T']==arm], bins=25, alpha=.5, density=True,label=f'T={arm}')
plt.xlabel('Known propensity after selection'); plt.ylabel('Density'); plt.legend()
plt.title('Observed confounding creates different assignment distributions')
finish_figure('propensity_overlap', CFG.GRAPH_DIR)

This code snippet analyzes the impact of observational selection on covariate balance and propensity score distribution in the Hillstrom dataset.

First, it applies the `select_observational` function to the training data (`hillstrom['train']`) to create a selected pool with induced confounding. The strength of the confounding is controlled by `CFG.SELECTION_STRENGTH`, and the seed ensures reproducibility.

Next, it calls `balance_diagnostics` on the selected pool to assess covariate balance between treatment groups. This function calculates standardized mean differences (SMDs) for each feature and provides diagnostic statistics such as effective sample size, maximum weight, and propensity score ranges. The diagnostic statistics are displayed as a Pandas Series using `display_table`, saved as "propensity_diagnostics" in the export directory.

The first 12 rows of the covariate balance DataFrame (showing SMDs) are then displayed using `display_table` and saved as "covariate_balance". This provides a quick overview of how well balanced the covariates are after observational selection.

Finally, it generates a histogram visualizing the distribution of propensity scores for treated (T=1) and control (T=0) groups in the selected pool. The plot illustrates how observational selection creates different assignment distributions for each group, highlighting the induced confounding. This figure is saved as "propensity_overlap" using `finish_figure`.

In essence, this code snippet demonstrates the effect of observational selection on covariate balance and propensity score distribution, providing visual and quantitative evidence of the induced confounding. It highlights the importance of addressing confounding when estimating causal effects in observational studies.

# The models

## Vintage models: regression and classical CATE learners

In [ ]:
"""Transparent vintage regression and strong classical CATE estimators.

All learners accept numeric pretreatment features, binary treatment, and a
continuous outcome on its original scale. Fixed configurations keep the demo
reproducible; these are baselines, not claims of optimal hyperparameter tuning.
"""




class InteractionOLS:
    """Linear outcome regression with treatment-covariate interactions."""

    @staticmethod
    def _design(X, T):
        return np.column_stack([X, T, X * T[:, None]])

    def fit(self, X, T, Y):
        X, T, Y = _inputs(X, T, Y)
        self.model_ = LinearRegression().fit(self._design(X, T), Y)
        return self

    def predict_outcomes(self, X):
        X = np.asarray(X)
        return tuple(self.model_.predict(self._design(X, np.full(len(X), arm))) for arm in (0, 1))

    def predict_cate(self, X):
        mu0, mu1 = self.predict_outcomes(X)
        return mu1 - mu0


class SLearner:
    def __init__(self, seed=CFG.SEED):
        self.seed = seed

    def fit(self, X, T, Y):
        X, T, Y = _inputs(X, T, Y)
        self.model_ = outcome_model(self.seed).fit(np.column_stack([X, T]), Y)
        return self

    def predict_outcomes(self, X):
        X = np.asarray(X)
        return tuple(self.model_.predict(np.column_stack([X, np.full(len(X), arm)])) for arm in (0, 1))

    def predict_cate(self, X):
        mu0, mu1 = self.predict_outcomes(X)
        return mu1 - mu0


class TLearner:
    def __init__(self, seed=CFG.SEED):
        self.seed = seed

    def fit(self, X, T, Y):
        X, T, Y = _inputs(X, T, Y)
        self.models_ = [outcome_model(self.seed + arm).fit(X[T == arm], Y[T == arm]) for arm in (0, 1)]
        return self

    def predict_outcomes(self, X):
        return tuple(model.predict(X) for model in self.models_)

    def predict_cate(self, X):
        mu0, mu1 = self.predict_outcomes(X)
        return mu1 - mu0


class XLearner:
    """T-learner imputation followed by separate treatment-effect regressions."""

    def __init__(self, seed=CFG.SEED):
        self.seed = seed

    def fit(self, X, T, Y):
        X, T, Y = _inputs(X, T, Y)
        self.first_ = TLearner(self.seed).fit(X, T, Y)
        mu0, mu1 = self.first_.predict_outcomes(X)
        d0, d1 = mu1[T == 0] - Y[T == 0], Y[T == 1] - mu0[T == 1]
        self.effect0_ = outcome_model(self.seed + 2).fit(X[T == 0], d0)
        self.effect1_ = outcome_model(self.seed + 3).fit(X[T == 1], d1)
        self.propensity_ = propensity_model(self.seed).fit(X, T)
        return self

    def predict_cate(self, X):
        e = self.propensity_.predict_proba(X)[:, 1]
        return e * self.effect0_.predict(X) + (1 - e) * self.effect1_.predict(X)

This code defines several classical causal inference estimators: Interaction OLS, S-Learner, T-Learner, and X-Learner. These models are designed to estimate the Conditional Average Treatment Effect (CATE) from observational data.

**InteractionOLS:** This model estimates the treatment effect by including interaction terms between the treatment variable and covariates in a linear regression model. It assumes that the treatment effect varies linearly with the covariates.

**SLearner:** This model trains a single outcome regression model on the combined dataset of treated and control units, including both the covariates and the treatment indicator as predictors. The CATE is then estimated by taking the difference in predicted outcomes for treated and control groups.

**TLearner:** This model trains separate outcome regression models for the treated and control groups. The CATE is then estimated by taking the difference in predicted outcomes from the two models.

**XLearner:** This model combines elements of both T-Learner and S-Learner. It first fits separate outcome models for treated and control units (like T-Learner) to estimate potential outcomes. Then, it estimates treatment effects within each group using additional regression models trained on the residuals from the initial outcome models. Finally, it uses a propensity score model to combine these individual effect estimates into an overall CATE prediction.

Each class implements `fit` and `predict_cate` methods:
*   **`fit(X, T, Y)`:** Trains the model using the provided training data (features X, treatment assignments T, and outcomes Y).
*   **`predict_cate(X)`:** Predicts the CATE for new instances represented by features X.

The code also includes a `_inputs` function (defined elsewhere) that validates the input data before fitting the models. The use of fixed configurations and seeds ensures reproducibility of the results, making these estimators suitable as baselines for comparison with more complex methods.

In [ ]:
class DRLearner:
    """Regress out-of-fold AIPW pseudo-outcomes on X to estimate CATE."""

    def __init__(self, seed=CFG.SEED, n_splits=3):
        self.seed, self.n_splits = seed, n_splits

    def fit(self, X, T, Y):
        X, T, Y = _inputs(X, T, Y)
        self.scores_ = crossfit_scores(X, T, Y, seed=self.seed, n_splits=self.n_splits)
        self.effect_ = outcome_model(self.seed).fit(X, self.scores_["gamma"])
        return self

    def predict_cate(self, X):
        return self.effect_.predict(X)


class CausalForest:
    """EconML's orthogonalized causal forest with cross-fitted nuisances."""

    def __init__(self, seed=CFG.SEED, fast=True):
        self.seed, self.fast = seed, fast

    def fit(self, X, T, Y):

        X, T, Y = _inputs(X, T, Y)
        self.model_ = CausalForestDML(
            model_y=outcome_model(self.seed), model_t=propensity_model(self.seed),
            discrete_treatment=True, n_estimators=100 if self.fast else 400,
            max_depth=6 if self.fast else None, min_samples_leaf=8,
            cv=3, random_state=self.seed, n_jobs=1, inference=False,
        )
        self.model_.fit(Y, T, X=X)
        return self

    def predict_cate(self, X):
        return np.asarray(self.model_.effect(np.asarray(X))).reshape(-1)


def make_classical_models(seed=CFG.SEED, fast=True):
    return {
        "Interaction OLS": InteractionOLS(),
        "S-learner": SLearner(seed),
        "T-learner": TLearner(seed),
        "X-learner": XLearner(seed),
        "DR-learner": DRLearner(seed),
        "Causal forest": CausalForest(seed, fast),
    }

This code defines two more causal inference estimators – the DR-Learner and a Causal Forest implementation – and provides a function to create a dictionary of all classical models.

**DRLearner:** This model uses Double/Debiased Machine Learning (DML) principles. It leverages cross-fitted nuisance functions (estimated propensity scores and potential outcomes) from the `crossfit_scores` function, then regresses these pseudo-outcomes on the features X to estimate the CATE.

**CausalForest:** This model utilizes EconML's `CausalForestDML` implementation, a causal forest algorithm that estimates heterogeneous treatment effects while controlling for confounding bias. It uses cross-fitted nuisance models for estimating propensity scores and potential outcomes. The number of estimators and maximum depth are adjusted based on the `fast` flag to control computational cost.

**make_classical_models:** This function creates a dictionary containing instances of all the classical causal inference models defined in this and previous code blocks (Interaction OLS, S-Learner, T-Learner, X-Learner, DR-Learner, and Causal Forest). The `seed` parameter is used to initialize random number generators for reproducibility. The `fast` flag controls whether a faster but potentially less accurate configuration is used for the Causal Forest model.

These models provide a diverse set of approaches for estimating causal effects from observational data, ranging from simple linear regression-based methods (Interaction OLS) to more sophisticated machine learning techniques (Causal Forest). They serve as baselines for comparison with foundation models and other advanced estimation methods.

## Foundation models

In [ ]:
class LayerStack(nn.Module):
    """Similar to nn.Sequential, but with support for passing keyword arguments
    to layers and stacks the same layer multiple times.
    """

    def __init__(
        self,
        *,
        layer_creator: Callable[[], nn.Module],
        num_layers: int,
        recompute_each_layer: bool = False,
        min_num_layers_layer_dropout: int | None = None,
    ):
        super().__init__()
        self.layers = nn.ModuleList([layer_creator() for _ in range(num_layers)])
        self.num_layers = num_layers
        self.min_num_layers_layer_dropout = (
            min_num_layers_layer_dropout
            if min_num_layers_layer_dropout is not None
            else num_layers
        )
        self.recompute_each_layer = recompute_each_layer

    def forward(
        self,
        x: torch.Tensor,
        *,
        half_layers: bool = False,
        **kwargs: Any,
    ) -> torch.Tensor:
        if half_layers:
            assert (
                self.min_num_layers_layer_dropout == self.num_layers
            ), "half_layers only works without layer dropout"
            n_layers = self.num_layers // 2
        else:
            n_layers = torch.randint(
                low=self.min_num_layers_layer_dropout,
                high=self.num_layers + 1,
                size=(1,),
            ).item()

        for layer in self.layers[:n_layers]:
            if self.recompute_each_layer and x.requires_grad:
                x = checkpoint(partial(layer, **kwargs), x, use_reentrant=False)
            else:
                x = layer(x, **kwargs)

        return x

This code defines a custom PyTorch module called `LayerStack` that provides more flexibility than the standard `nn.Sequential` container for building neural networks.

The `LayerStack` allows you to stack the same type of layer multiple times and pass keyword arguments to each layer during the forward pass. It also includes features for controlling recomputation and dropout within the layers.

Here's a breakdown of the key components:

*   **`__init__(...)`:** The constructor initializes the `LayerStack`.
    *   `layer_creator`: A callable (e.g., a function or lambda expression) that creates an instance of the layer to be stacked.
    *   `num_layers`: The number of layers to stack.
    *   `recompute_each_layer`: A boolean flag indicating whether to recompute each layer during the forward pass using `torch.utils.checkpoint.checkpoint`. This can help reduce memory usage, especially for deep networks.
    *   `min_num_layers_layer_dropout`:  Specifies the minimum number of layers that will always be used, even when dropout is applied (see `forward` method).

*   **`forward(..., half_layers=False, **kwargs)`:** The forward pass method performs the computation.
    *   `half_layers`: If set to True, only the first half of the layers are used. This requires that `min_num_layers_layer_dropout` is equal to `num_layers`.
    *   `**kwargs`: Allows passing keyword arguments to each layer during the forward pass.
    *   The number of layers to use (`n_layers`) is determined based on the `half_layers` flag and a random integer between `min_num_layers_layer_dropout` and `num_layers`. This introduces a form of stochastic depth or dropout at the layer level.
    *   For each layer up to `n_layers`, it checks if recomputation is enabled (`recompute_each_layer`) and if gradients are required for the input tensor (`x.requires_grad`). If both conditions are true, it uses `torch.utils.checkpoint.checkpoint` to recompute the layer's forward pass during the backward pass, reducing memory usage at the cost of some computation time. Otherwise, it simply calls the layer's forward method with the input and any provided keyword arguments.

In essence, this `LayerStack` provides a flexible and efficient way to build neural networks with repeated layers, customizable recomputation behavior, and support for passing arbitrary keyword arguments to each layer. It is particularly useful for building deep networks where memory usage can be a concern.

In [ ]:
class GMMHead(nn.Module):

    def __init__(self, z_dim: int, n_components: int, min_sigma: float = 1e-3, pi_temp: float = 1.0):
        super().__init__()
        self.K = n_components
        self.min_sigma = min_sigma
        self.pi_temp = pi_temp
        self.fc_pi = nn.Linear(z_dim, n_components) 
        self.fc_mu = nn.Linear(z_dim, n_components) 
        self.fc_sigma = nn.Linear(z_dim, n_components) 

    def forward(self, z: torch.Tensor):

        logits = self.fc_pi(z) / self.pi_temp # temperature scaling
        pi = F.softmax(logits, dim=-1) # mixture weight 
        mu = self.fc_mu(z)
        sigma = F.softplus(self.fc_sigma(z)) + self.min_sigma
        return pi, mu, sigma


class PerFeatureTransformerCATE(nn.Module):

    def __init__(
        self,
        *,
        use_gmm_head: bool = True,
        gmm_n_components: int =5,
        gmm_min_sigma: float = 1e-3,
        gmm_pi_temp: float = 1.0,
        x_encoder: nn.Module | None = None,  # For covariates X
        a_encoder: nn.Module | None = None,  # For treatment A  
        y_encoder: nn.Module | None = None,  # For factual outcome Y
        ninp: int = CFG.DEFAULT_EMSIZE,
        nhead: int = 4,
        nhid: int = CFG.DEFAULT_EMSIZE * 4,
        nlayers: int = 10,
        decoder_dict: dict[str, tuple[type[nn.Module] | None, int]] | None = None,
        init_method: str | None = None,
        activation: Literal["gelu", "relu"] = "gelu",
        recompute_layer: bool = False,
        min_num_layers_layer_dropout: int | None = None,
        repeat_same_layer: bool = False,
        dag_pos_enc_dim: int = 0,
        features_per_group: int = 1,
        feature_positional_embedding: (
            Literal[
                "normal_rand_vec",
                "uni_rand_vec", 
                "learned",
                "subspace",
            ]
            | None
        ) = None,
        zero_init: bool = True,
        use_separate_decoder: bool = False,
        nlayers_decoder: int | None = None,
        use_encoder_compression_layer: bool = False,
        precomputed_kv: (
            list[torch.Tensor | tuple[torch.Tensor, torch.Tensor]] | None
        ) = None,
        cache_trainset_representation: bool = False,
        seed: int | None = None,
        **layer_kwargs: Any,
    ):


        decoder_dict = {"cate": (None, 1)}   


        super().__init__()

        # Set up encoders for X, A, Y
        if x_encoder is None:
            x_encoder = SequentialEncoder(
                LinearInputEncoderStep(
                    num_features=1,
                    emsize=CFG.DEFAULT_EMSIZE,
                    replace_nan_by_zero=False,
                    bias=True,
                    in_keys=("main",),
                    out_keys=("output",),
                ),
            )
        if a_encoder is None:
            a_encoder = SequentialEncoder(
                NanHandlingEncoderStep(),
                LinearInputEncoderStep(
                    num_features=2,  # A + nan indicators
                    emsize=CFG.DEFAULT_EMSIZE,
                    replace_nan_by_zero=False,
                    bias=True,
                    out_keys=("output",),
                    in_keys=("main", "nan_indicators"),
                ),
            )

        if y_encoder is None:
            y_encoder = SequentialEncoder(
                NanHandlingEncoderStep(),
                LinearInputEncoderStep(
                    num_features=2,  # Y + nan indicators
                    emsize=CFG.DEFAULT_EMSIZE,
                    replace_nan_by_zero=False,
                    bias=True,
                    out_keys=("output",),
                    in_keys=("main", "nan_indicators"),
                ),
            )


        self.x_encoder = x_encoder  # Covariate encoder
        self.a_encoder = a_encoder  # Treatment encoder  
        self.y_encoder = y_encoder  # Outcome encoder

        self.ninp = ninp
        self.nhead = nhead
        self.nhid = nhid
        self.init_method = init_method
        self.features_per_group = features_per_group
        self.cache_trainset_representation = cache_trainset_representation
        self.cached_embeddings: torch.Tensor | None = None

        layer_creator = lambda: PerFeatureEncoderLayer(
            d_model=ninp,
            nhead=nhead,
            dim_feedforward=nhid,
            activation=activation,
            zero_init=zero_init,
            precomputed_kv=(
                precomputed_kv.pop(0) if precomputed_kv is not None else None
            ),
            **layer_kwargs,
        )
        if repeat_same_layer:
            layer = layer_creator()
            layer_creator = lambda: layer

        nlayers_encoder = nlayers
        if use_separate_decoder and nlayers_decoder is None:
            nlayers_decoder = max((nlayers // 3) * 1, 1)
            nlayers_encoder = max((nlayers // 3) * 2, 1)

        self.transformer_encoder = LayerStack(
            layer_creator=layer_creator,
            num_layers=nlayers_encoder,
            recompute_each_layer=recompute_layer,
            min_num_layers_layer_dropout=min_num_layers_layer_dropout,
        )

        self.transformer_decoder = None
        if use_separate_decoder:
            assert nlayers_decoder is not None
            self.transformer_decoder = LayerStack(
                layer_creator=layer_creator,
                num_layers=nlayers_decoder,
            )

        self.global_att_embeddings_for_compression = None
        if use_encoder_compression_layer:
            assert use_separate_decoder
            num_global_att_tokens_for_compression = 512

            self.global_att_embeddings_for_compression = nn.Embedding(
                num_global_att_tokens_for_compression,
                ninp,
            )

            self.encoder_compression_layer = LayerStack(
                layer_creator=layer_creator,
                num_layers=2,
            )

        initialized_decoder_dict = {}
        for decoder_key in decoder_dict:
            decoder_model, decoder_n_out = decoder_dict[decoder_key]
            if decoder_model is None:
                initialized_decoder_dict[decoder_key] = nn.Sequential(
                    nn.Linear(ninp, nhid),
                    nn.GELU(),
                    nn.Linear(nhid, decoder_n_out),
                )
            else:
                initialized_decoder_dict[decoder_key] = decoder_model(
                    ninp,
                    nhid,
                    decoder_n_out,
                )
        self.decoder_dict = nn.ModuleDict(initialized_decoder_dict)

        self.feature_positional_embedding = feature_positional_embedding
        if feature_positional_embedding == "learned":
            self.feature_positional_embedding_embeddings = nn.Embedding(1_000, ninp)
        elif feature_positional_embedding == "subspace":
            self.feature_positional_embedding_embeddings = nn.Linear(ninp // 4, ninp)

        self.dag_pos_enc_dim = dag_pos_enc_dim
        self.cached_feature_positional_embeddings: torch.Tensor | None = None
        self.seed = seed if seed is not None else random.randint(0, 1_000_000)


        self.use_gmm_head = use_gmm_head
        if self.use_gmm_head:
            self.gmm_head = GMMHead(
                z_dim = self.ninp,
                n_components = gmm_n_components,
                min_sigma = gmm_min_sigma,
                pi_temp = gmm_pi_temp,
            )

    def _pack_eval_io(
        self,
        X_train: torch.Tensor,
        A_train: torch.Tensor,
        Y_train: torch.Tensor,
        X_test: torch.Tensor,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ):
        """
        Expected input shapes: 
            X_train: [N_tr, Dx],  A_train: [N_tr, 1],  Y_train: [N_tr, 1],  X_test: [N_te, Dx]
        """
        if device is None:
            device = next(self.parameters()).device
        if dtype is None:
            dtype = torch.float32

        X_train = X_train.to(device=device, dtype=dtype)
        A_train = A_train.to(device=device, dtype=dtype)
        Y_train = Y_train.to(device=device, dtype=dtype)
        X_test  = X_test.to(device=device,  dtype=dtype)

        n_tr = X_train.shape[0]
        n_te = X_test.shape[0]
        single_eval_pos = int(n_tr)

        # concat along seq (support + query)
        X_all = torch.cat([X_train, X_test], dim=0)            
        A_all = torch.cat([A_train, torch.full((n_te, A_train.shape[1]), torch.nan, device=device, dtype=dtype)], dim=0)
        Y_all = torch.cat([Y_train, torch.full((n_te, Y_train.shape[1]), torch.nan, device=device, dtype=dtype)], dim=0)

        # add batch dim = 1 -> [S, 1, D]
        X_all = X_all.unsqueeze(1)
        A_all = A_all.unsqueeze(1)
        Y_all = Y_all.unsqueeze(1)

        x = {"main": X_all}   
        a = {"main": A_all}   
        y = {"main": Y_all}   
        return x, a, y, single_eval_pos

    @torch.no_grad()
    def estimate_cate(
        self,
        X_train: torch.Tensor,
        A_train: torch.Tensor,
        Y_train: torch.Tensor,
        X_test: torch.Tensor,
        *,
        return_gmm: bool = False,
        **forward_kwargs,
    ):
        """
            inputs: X_train, A_train, Y_train, X_test 
            returns: dict with predicted CATE on X_test (and optional GMM params)
        """
        x, a, y, single_eval_pos = self._pack_eval_io(X_train, A_train, Y_train, X_test)
        out = self.forward(x, a, y, single_eval_pos=single_eval_pos, **forward_kwargs)

        cate = out["cate_mean"][:, single_eval_pos:, :]
        cate = cate.squeeze()
        result = {"cate": cate}

        if return_gmm and ("gmm_pi" in out):
            result.update({
                "gmm_pi":    out["gmm_pi"][0, single_eval_pos:, :],   
                "gmm_mu":    out["gmm_mu"][0, single_eval_pos:, :],   
                "gmm_sigma": out["gmm_sigma"][0, single_eval_pos:, :],
            })
        return result

    def forward(
        self,
        x: torch.Tensor | dict,
        a: torch.Tensor | dict | None = None, 
        y: torch.Tensor | dict | None = None,
        single_eval_pos: int | None = None,
        *,
        style: torch.Tensor | None = None,
        data_dags: list[Any] | None = None,
        categorical_inds: list[int] | None = None,
        half_layers: bool = False,
    ) -> Any | dict[str, torch.Tensor]:
        """Core forward pass for CATE estimation

        Args:
            x: Covariates. Shape: (seq_len, batch_size, num_features) # seq_len = n_train_rows + n_test_rows
            a: Treatment assignment. Shape: (seq_len, batch_size, 1) 
            y: Factual outcomes. Shape: (seq_len, batch_size, 1)
            single_eval_pos: Position to evaluate at
        Returns:
            Dictionary containing CATE distribution predictions and optionally other outputs
        """

        if isinstance(x, dict):
            assert "main" in set(x.keys()), f"Main must be in input keys: {x.keys()}."
        else:
            x = {"main": x}
        seq_len, batch_size, num_x_features = x["main"].shape # seq_len = n_train_rows + n_test_rows

        if isinstance(a, dict):
            assert "main" in set(a.keys()), f"Main must be in input keys: {a.keys()}."
        else:
            a = {"main": a}
        _, _, num_a_features = a["main"].shape

        if isinstance(y, dict):
            assert "main" in set(y.keys())
        else:
            y = {"main": y}
        _, _, num_y_features = y["main"].shape

        training_targets_provided = y["main"] is not None and y["main"].shape[0] > 0

        # Handle y (can be None during inference)
        if y is None:
            y = torch.zeros(
                0,
                batch_size, 
                device=x["main"].device,
                dtype=x["main"].dtype,
            )
        # Handle a (can be None during inference)
        if a is None:
            a = torch.zeros(
                0,
                batch_size, 
                device=x["main"].device,
                dtype=x["main"].dtype,
            )


        # The model will make predictions from the single_eval_pos'th row onwards. 

        single_eval_pos_ = int(single_eval_pos)

        # Pad features to multiple of features_per_group
        for k in x:
            num_features_ = x[k].shape[2]

            # pad to multiple of features_per_group
            missing_to_next = (
                self.features_per_group - (num_features_ % self.features_per_group)
            ) % self.features_per_group

            if missing_to_next > 0:
                x[k] = torch.cat(
                    (
                        x[k],
                        torch.zeros(
                            seq_len,
                            batch_size,
                            missing_to_next,
                            device=x[k].device,
                            dtype=x[k].dtype,
                        ),
                    ),
                    dim=-1,
                )

        for k in x:
            x[k] = einops.rearrange(
                x[k],
                "s b (f n) -> b s f n",
                n=self.features_per_group,
            ) 

        for k in y:
            if y[k].ndim == 1:
                y[k] = y[k].unsqueeze(-1)
            if y[k].ndim == 2:
                y[k] = y[k].unsqueeze(-1)  # s b -> s b 1

            y[k] = y[k].transpose(0, 1)  # s b 1 -> b s 1

            if y[k].shape[1] < x["main"].shape[1]:
                assert (
                    y[k].shape[1] == single_eval_pos
                    or y[k].shape[1] == x["main"].shape[1]
                )
                assert k != "main" or y[k].shape[1] == single_eval_pos, (
                    "For main y, y must not be given for target"
                    " time steps (Otherwise the solution is leaked)."
                )
                if y[k].shape[1] == single_eval_pos:
                    y[k] = torch.cat(
                        (
                            y[k],
                            torch.nan
                            * torch.zeros(
                                y[k].shape[0],
                                x["main"].shape[1] - y[k].shape[1],
                                y[k].shape[2],
                                device=y[k].device,
                                dtype=y[k].dtype,
                            ),
                        ),
                        dim=1,
                    )

            y[k] = y[k].transpose(0, 1)  # b s 1 -> s b 1

        # making sure no label leakage ever happens
        y["main"][single_eval_pos:] = torch.nan
        a["main"][single_eval_pos:] = torch.nan

        # Encode factual outcome Y
        embedded_y = self.y_encoder(
            y,
            single_eval_pos=single_eval_pos_,
            cache_trainset_representation=self.cache_trainset_representation,
        ).transpose(0, 1)

        del y

        if torch.isnan(embedded_y).any():
            raise ValueError(
                f"{torch.isnan(embedded_y).any()=}, make sure to add nan handlers"
                " to the ys that are not fully provided (test set missing)",
            )

                # Encode treatment A  

        embedded_a = self.a_encoder(
            a,
            single_eval_pos=single_eval_pos_,
            cache_trainset_representation=self.cache_trainset_representation,
        ).transpose(0, 1)

        del a
        if torch.isnan(embedded_a).any():
            raise ValueError(
                f"{torch.isnan(embedded_a).any()=}, make sure to add nan handlers"
                " to the as that are not fully provided (test set missing)",
            )

        # Encode covariates X
        for k in x:
            x[k] = einops.rearrange(x[k], "b s f n -> s (b f) n")
        embedded_x = einops.rearrange(
            self.x_encoder(
                x,
                single_eval_pos=single_eval_pos_,
                cache_trainset_representation=self.cache_trainset_representation,

            ),
            "s (b f) e -> b s f e",
            b=batch_size,
        )
        del x


        embedded_a_grouped = embedded_a.unsqueeze(2)  # (b, s, e) -> (b, s, 1, e)
        embedded_y_grouped = embedded_y.unsqueeze(2)  # (b, s, e) -> (b, s, 1, e)


        embedded_input = torch.cat(
            (embedded_x, embedded_a_grouped, embedded_y_grouped), dim=2
        )  # (b, s, f_x + 1 + 1, e)

        if torch.isnan(embedded_input).any():
            raise ValueError(
                f"There should be no NaNs in the encoded x, a, and y."
                f"Your embedded inputs returned the following:"
                f"{torch.isnan(embedded_x).any()=} | {torch.isnan(embedded_a).any()=} | {torch.isnan(embedded_y).any()=}",
            )

        # Apply positional embeddings
        embedded_input = self.add_embeddings_cate(
            embedded_input,
            data_dags=data_dags,
            num_features=num_x_features + num_a_features,
            seq_len=seq_len,
        )

        # Pass through transformer
        encoder_out = self.transformer_encoder(
            embedded_input,
            single_eval_pos=single_eval_pos_,
            half_layers=half_layers,
            cache_trainset_representation=self.cache_trainset_representation,
        )


        if self.transformer_decoder:
            assert not half_layers
            assert encoder_out.shape[1] == single_eval_pos_

            test_encoder_out = self.transformer_decoder(
                embedded_input[:, single_eval_pos_:],
                single_eval_pos=0,
                att_src=encoder_out,
            )
            encoder_out = torch.cat([encoder_out, test_encoder_out], 1)


        cate_representation = encoder_out.mean(dim=2)  

        gmm_out = None
        if self.use_gmm_head:
            B, S, E = cate_representation.shape
            z_rep = cate_representation.reshape(B * S, E)
            pi, mu, sigma = self.gmm_head(z_rep)
            K = pi.shape[-1]
            pi = pi.reshape(B, S, K)
            mu = mu.reshape(B, S, K)
            sigma = sigma.reshape(B, S, K)
            gmm_out = {"gmm_pi": pi, "gmm_mu": mu, "gmm_sigma": sigma}




        out = {}

        if gmm_out is not None:
            out.update(gmm_out)  # add gmm params if enabled
            out["cate_mean"] = (pi * mu).sum(dim=-1, keepdim=True)
        return out


    def add_embeddings_cate(
        self,
        embedded_input: torch.Tensor,
        *,
        data_dags: Iterable[nx.DiGraph] | None,
        num_features: int,
        seq_len: int,
    ) -> torch.Tensor:
        """embedding addition for causality"""

        with isolate_torch_rng(self.seed, device=embedded_input.device):
            if self.feature_positional_embedding == "normal_rand_vec":
                embs = torch.randn(
                    (embedded_input.shape[2], embedded_input.shape[3]),
                    device=embedded_input.device,
                    dtype=embedded_input.dtype,
                )
                embedded_input += embs[None, None]
            elif self.feature_positional_embedding == "uni_rand_vec":
                embs = (
                    torch.rand(
                        (embedded_input.shape[2], embedded_input.shape[3]),
                        device=embedded_input.device,
                        dtype=embedded_input.dtype,
                    )
                    * 2
                    - 1
                )
                embedded_input += embs[None, None]
            elif self.feature_positional_embedding == "learned":
                w = self.feature_positional_embedding_embeddings.weight
                embs = w[
                    torch.randint(
                        0,
                        w.shape[0],
                        (embedded_input.shape[2],),
                    )
                ]
                embedded_input += embs[None, None]
            elif self.feature_positional_embedding == "subspace":
                embs = torch.randn(
                    (embedded_input.shape[2], embedded_input.shape[3] // 4),
                    device=embedded_input.device,
                    dtype=embedded_input.dtype,
                )
                embs = self.feature_positional_embedding_embeddings(embs)
                embedded_input += embs[None, None]

        return embedded_input



    def reset_save_peak_mem_factor(self, factor: int | None = None) -> None:
        """Sets the save_peak_mem_factor for all layers."""
        for layer in self.transformer_encoder.layers:
            assert hasattr(layer, "save_peak_mem_factor")
            layer.save_peak_mem_factor = factor

    def empty_trainset_representation_cache(self) -> None:
        for layer in (self.transformer_decoder or self.transformer_encoder).layers:
            layer.empty_trainset_representation_cache()

This code defines a complex neural network architecture, `PerFeatureTransformerCATE`, designed for estimating Conditional Average Treatment Effects (CATE) from observational data. It incorporates several advanced techniques, including feature encoding, transformer layers, Gaussian Mixture Models (GMMs), and mechanisms for handling missing data and reducing memory usage.

Here's a breakdown of the key components:

*   **Encoders:** The model uses separate encoders (`x_encoder`, `a_encoder`, `y_encoder`) to process covariates (X), treatment assignments (A), and outcomes (Y). These encoders handle missing values using `NanHandlingEncoderStep` and transform the input features into a suitable embedding space using `LinearInputEncoderStep`.
*   **Transformer Encoder:** A stack of transformer layers (`transformer_encoder`) processes the encoded features to capture complex relationships between them. The number of layers, attention heads, and hidden dimensions are configurable.  The `LayerStack` class provides flexibility in controlling recomputation and dropout within the encoder.
*   **Transformer Decoder (Optional):** An optional transformer decoder (`transformer_decoder`) can be used for more sophisticated modeling.
*   **GMM Head (Optional):** If enabled, a Gaussian Mixture Model (GMM) head (`gmm_head`) is added to model the distribution of CATEs. This allows for capturing heterogeneity in treatment effects across different individuals.
*   **Positional Embeddings:** Positional embeddings are added to the input features to provide information about their order within the sequence. Different types of positional embeddings are supported, including learned embeddings and random vectors.
*   **Forward Pass:** The `forward` method orchestrates the entire computation process, encoding the inputs, passing them through the transformer layers, and generating CATE predictions (and GMM parameters if enabled). It also handles missing data and provides options for controlling recomputation and dropout.
*   **CATE Estimation:** The `estimate_cate` method takes training and test data as input and estimates the CATE on the test set using the trained model.
*   **Helper Functions:** Several helper functions are included to pack evaluation inputs, add positional embeddings, and reset memory usage parameters.

The code also includes several configuration options that allow for customizing the architecture and behavior of the model. These options include the use of GMMs, the number of transformer layers, the type of positional embeddings, and various regularization techniques.

In essence, this `PerFeatureTransformerCATE` class provides a powerful and flexible framework for estimating CATEs from observational data using state-of-the-art deep learning techniques. It is designed to handle complex datasets with missing values and heterogeneous treatment effects.

### Checkpoint adapters

In [ ]:
"""Version-pinned adapters for real released causal foundation models.

No benchmark truth or query outcomes enter these adapters. All normalization is
fitted on the context. We call CausalFM's released core to avoid its wrapper's
eager imports of unrelated training dependencies; the weights are unchanged.
"""


class CausalFMAdapter:
    def __init__(self, device=None, batch_size=CFG.CAUSALFM_BATCH_SIZE, seed=CFG.SEED):
        self.device, self.batch_size, self.seed = device or CFG.DEVICE, batch_size, seed

    def fit(self, X, T, Y):
        X, T, Y = _arrays(X, T, Y)
        self.x_scaler = StandardScaler().fit(X)
        self.y_mean, self.y_scale = float(Y.mean()), max(float(Y.std()), 1e-6)
        torch.manual_seed(self.seed)
        start = time.perf_counter()
        self.model = PerFeatureTransformerCATE().to(self.device)
        checkpoint = CFG.ROOT / CFG.MODEL_ASSETS["CausalFM"]["path"]
        state = torch.load(checkpoint, map_location=self.device, weights_only=True)
        self.model.load_state_dict(state.get("model_state_dict", state), strict=True)
        self.model.eval()
        synchronize_device(self.device)
        self.load_seconds = time.perf_counter() - start
        self.X = torch.as_tensor(self.x_scaler.transform(X), dtype=torch.float32, device=self.device)
        self.T = torch.as_tensor(T[:, None], dtype=torch.float32, device=self.device)
        self.Y = torch.as_tensor(((Y-self.y_mean)/self.y_scale)[:, None], dtype=torch.float32, device=self.device)
        self.context_size = len(X)
        return self

    def predict_cate(self, X):
        X = np.asarray(X, dtype=np.float32)
        values = []
        with torch.inference_mode():
            for start in range(0, len(X), self.batch_size):
                query = torch.as_tensor(self.x_scaler.transform(X[start:start+self.batch_size]),
                                        dtype=torch.float32, device=self.device)
                prediction = self.model.estimate_cate(self.X, self.T, self.Y, query)
                values.append(prediction["cate"].detach().cpu().numpy().reshape(-1))
        return np.concatenate(values) * self.y_scale


class CausalPFNAdapter:
    def __init__(self, device=None, batch_size=CFG.CAUSALPFN_BATCH_SIZE, seed=CFG.SEED, context_limit=None):
        self.device, self.batch_size = device or CFG.DEVICE, batch_size
        self.seed, self.context_limit = seed, context_limit

    def fit(self, X, T, Y):
        X, T, Y = _arrays(X, T, Y)
        self.x_scaler = StandardScaler().fit(X)
        self.y_mean, self.y_scale = float(Y.mean()), max(float(Y.std()), 1e-6)
        checkpoint = CFG.ROOT / CFG.MODEL_ASSETS["CausalPFN"]["path"]
        if not checkpoint.exists():
            raise FileNotFoundError("Run the asset preparation cell to fetch CausalPFN")
        torch.manual_seed(self.seed)
        # The upstream implementation reserves half its capacity per arm. In the
        # matched track capacity=2*n and k=n ensure that ALL rows are eligible,
        # even with imbalanced treatment. Actual context remains n, not 2*n.
        capacity = self.context_limit or 2 * len(X)
        self.model = CATEEstimator(device=self.device, model_path=str(checkpoint),
            cache_dir=str(CFG.ROOT / ".cache/causalpfn"),
            max_context_length=capacity, max_query_length=self.batch_size,
            num_neighbours=capacity//2, calibrate=False, verbose=False)
        self.model.fit(self.x_scaler.transform(X).astype(np.float32), T,
                       ((Y-self.y_mean)/self.y_scale).astype(np.float32))
        self.context_size = sum(min(int(np.sum(T == arm)), capacity//2) for arm in (0, 1))
        self.unique_training_rows = len(X)
        return self

    def predict_cate(self, X):
        query = self.x_scaler.transform(np.asarray(X, dtype=np.float32)).astype(np.float32)
        return np.asarray(self.model.estimate_cate(query)).reshape(-1) * self.y_scale

    def predict_interval(self, X, alpha=0.05, n_samples=1000):
        query = self.x_scaler.transform(np.asarray(X, dtype=np.float32)).astype(np.float32)
        result = self.model.estimate_cate_CI(query, alpha=alpha, n_samples=n_samples)
        return {key: np.asarray(value).reshape(-1)*self.y_scale for key,value in result.items()}

This code defines adapter classes for two pre-trained causal foundation models: CausalFM and CausalPFN. These adapters provide a standardized interface for loading and using these models within the broader experimental framework.

**CausalFMAdapter:**

*   **Initialization:** Loads the pre-trained CausalFM model from disk, scales the input features (X) using `StandardScaler`, and normalizes the outcome variable (Y).
*   **Fit Method:**  Loads the weights of the pre-trained CausalFM model. It transforms the training data (X, T, Y) using the fitted scaler and normalization parameters and stores them for use during prediction. The fit method primarily loads the pretrained weights; it doesn't perform any further training.
*   **Predict\_cate Method:**  Takes new input features (X), transforms them using the same scaling and normalization applied to the training data, and uses the loaded CausalFM model to predict the CATEs.

**CausalPFNAdapter:**

*   **Initialization:** Loads the pre-trained CausalPFN model from disk.
*   **Fit Method:**  Loads the pre-trained CausalPFN model and fits it to the training data (X, T, Y). It scales the input features (X) using `StandardScaler` and normalizes the outcome variable (Y). The fit method trains the CausalPFN model on the provided data.
*   **Predict\_cate Method:**  Takes new input features (X), transforms them using the same scaling and normalization applied to the training data, and uses the fitted CausalPFN model to predict the CATEs.
*   **predict\_interval method**: Estimates confidence intervals for the predicted CATEs using `estimate_cate_CI` from the CausalPFN model.

Both adapters adhere to a common interface: they have `fit` and `predict_cate` methods that take training data (X, T, Y) and new input features (X), respectively, and return CATE predictions. They also handle data preprocessing steps such as scaling and normalization to ensure compatibility with the pre-trained models.

These adapters encapsulate the details of loading and using the foundation models, allowing for easy integration into the broader causal inference workflow without requiring direct interaction with the underlying model implementations. The code emphasizes version pinning and avoids unnecessary dependencies to ensure reproducibility and stability.

In [ ]:
def make_foundation_models(seed=CFG.SEED, device=None):
    return {"CausalFM": CausalFMAdapter(device=device, seed=seed),
            "CausalPFN": CausalPFNAdapter(device=device, seed=seed)}

This function serves as a factory for creating instances of the causal foundation model adapters.

It takes a `seed` and an optional `device` as input. It then returns a dictionary containing two keys: "CausalFM" and "CausalPFN". The values associated with these keys are instances of the corresponding adapter classes (`CausalFMAdapter` and `CausalPFNAdapter`), initialized with the provided seed and device.

In essence, this function provides a convenient way to create and configure the causal foundation models for use in experiments. It encapsulates the instantiation logic and allows for easy access to the models through a dictionary-like structure.

In [ ]:
display_table(pd.DataFrame([
    {'model': name, 'family': 'Vintage / classical', 'device': 'cpu'}
    for name in make_classical_models()
] + [
    {'model': name, 'family': 'Foundation', 'device': model.device}
    for name, model in make_foundation_models().items()
]), 'model_roster', CFG.GRAPH_DIR)

This code snippet generates and displays a table summarizing the models used in the analysis, including both classical methods and foundation models.

It creates a list of dictionaries, each representing a model. The first part of the list contains information about the classical models obtained using `make_classical_models()`. For each classical model, it includes its name, family ("Vintage / classical"), and device (set to "cpu" since these models typically run on the CPU).

The second part of the list contains information about the foundation models obtained using `make_foundation_models()`. For each foundation model, it includes its name, family ("Foundation"), and the device it's running on (obtained from the model object itself).

These two lists are concatenated to create a single list of dictionaries. This list is then converted into a Pandas DataFrame and displayed as an interactive table using `display_table`. The table is saved as a PNG image named "model_roster" in the export directory (`CFG.GRAPH_DIR`).

In essence, this code provides a clear overview of all the models used in the analysis, including their type (classical or foundation) and the device they are running on.

# Our comparisons

| Business or scientific question | Dataset | Evidence |
| --- | --- | --- |
| Which models recover individual effects? | Simulation and IHDP | CATE error against known truth |
| Whom should we email, and is it profitable? | Hillstrom | Independent policy value under a budget |
| How much do identification assumptions matter? | Controlled simulations | Bias, overlap, sensitivity, and interval diagnostics |
| What if customers influence one another? | Simulated customer networks | Direct effects, spillovers, and total policy value |

Model and policy choices are fixed before looking at test outcomes. All reported GPU timings include synchronization.

## 1. Simulation and IHDP: which models recover known effects?

The eight competitors are interaction OLS, boosted S/T/X learners, a cross-fitted DR learner,
EconML's CausalForestDML, and the two released foundation models. Defaults are prespecified;
there is no search using test effects. All models receive identical context rows.

$$\sqrt{PEHE}=\sqrt{n^{-1}\sum_i(\hat\tau(X_i)-\tau(X_i))^2}.$$

The query-average CATE error is measured on the same fixed query covariates for every model.
It is not confused with a noisy sample average of individual potential-outcome differences.
IHDP replications reuse covariates and are not independent populations. Timing includes model
loading/preparation and the CausalPFN retrieval-model fit; pretraining is already amortized.

In [ ]:
accuracy = (load_benchmark_results('accuracy', CFG.OUT) if CFG.REUSE_RESULTS else
            run_accuracy_benchmark(mode=CFG.MODE, output_dir=CFG.OUT, progress=True))
assert accuracy['manifest']['config']['mode'] == CFG.MODE
accuracy_metrics = accuracy['metrics']
display_table(accuracy_metrics.head(12), 'accuracy_preview', CFG.GRAPH_DIR)

This code snippet loads or runs the accuracy benchmark and then displays a preview of the results.

First, it checks if `CFG.REUSE_RESULTS` is True. If so, it attempts to load previously generated benchmark results from the output directory (`CFG.OUT`) using the `load_benchmark_results` function. This avoids re-running the potentially time-consuming benchmark if results are already available.

If `CFG.REUSE_RESULTS` is False (or if loading fails), it runs the accuracy benchmark using the `run_accuracy_benchmark` function, specifying the mode and output directory from the configuration (`CFG`). The `progress=True` argument enables progress updates during the benchmark execution.

The result of either loading or running the benchmark is stored in the `accuracy` variable. An assertion verifies that the mode used in the loaded or run benchmark matches the current configuration mode (`CFG.MODE`), ensuring consistency.

Finally, it extracts the metrics DataFrame from the `accuracy` dictionary and displays the first 12 rows of this DataFrame as an interactive table using `display_table`. The table is saved as a PNG image named "accuracy_preview" in the export directory (`CFG.GRAPH_DIR`). This provides a quick overview of the benchmark results.

In [ ]:
accuracy_summary = accuracy_metrics.groupby(['dataset','scenario','n_train','method']).agg(
    cate_rmse=('cate_rmse','mean'), mean_abs_query_effect_error=('ate_abs_error','mean'),
    mean_policy_regret=('policy_regret','mean'), median_seconds=('total_seconds','median'),
    replications=('cate_rmse','size')).reset_index()
display_table(accuracy_summary, 'accuracy_summary', CFG.GRAPH_DIR)
accuracy_summary.to_csv(CFG.OUT/'accuracy_summary.csv', index=False)
sim = accuracy_metrics.query("dataset == 'Simulation'")
for scenario, frame in sim.groupby('scenario'):
    fig, ax = plt.subplots(figsize=(10,5))
    for method, values in frame.groupby('method'):
        means = values.groupby('n_train')['cate_rmse'].mean()
        ax.plot(means.index,means.values,marker='o',label=method,color=CFG.COLORS.get(method))
    ax.set(title=scenario, xlabel='Identical context rows per method', ylabel='CATE RMSE (lower is better)')
    ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=9)
    finish_figure(f'sample_efficiency_{scenario}', CFG.GRAPH_DIR)

This code snippet summarizes and visualizes the accuracy benchmark results.

First, it groups the `accuracy_metrics` DataFrame by dataset, scenario, training sample size (`n_train`), and method. It then calculates several aggregate statistics for each group:
*   `cate_rmse`: The mean CATE Root Mean Squared Error.
*   `mean_abs_query_effect_error`: The mean absolute error in estimating the Average Treatment Effect (ATE).
*   `mean_policy_regret`: The mean policy regret, representing the difference between the performance of the chosen policy and an optimal policy.
*   `median_seconds`: The median total execution time.
*   `replications`: The number of replications used for each group.

The resulting aggregated DataFrame is stored in `accuracy_summary`, which is then displayed as an interactive table using `display_table` and saved to a CSV file named "accuracy_summary.csv" in the output directory (`CFG.OUT`).

Next, it filters the `accuracy_metrics` DataFrame to include only results from the simulation experiments (where `dataset == 'Simulation'`). It then iterates through each scenario within the simulation data:
*   For each scenario, it creates a plot showing the relationship between training sample size (`n_train`) and CATE RMSE for each method. The x-axis represents the number of training samples, and the y-axis represents the mean CATE RMSE. Each line on the plot corresponds to a different method, with colors defined in `CFG.COLORS`.
*   The plot is titled with the scenario name, and labels are added to the axes. A legend is included to identify each method.
*   Finally, the plot is saved as a PNG image named "sample_efficiency_{scenario}" in the export directory using `finish_figure`.

In essence, this code provides a comprehensive summary of the accuracy benchmark results, including aggregated statistics and visualizations that illustrate the sample efficiency of different methods under various scenarios. The plots help to assess how the performance of each method improves as the training sample size increases.

In [ ]:
ihdp_results = accuracy_metrics.query("dataset == 'IHDP'")
ihdp_summary = ihdp_results.groupby('method').agg(cate_rmse=('cate_rmse','mean'),
    rmse_sd_across_replications=('cate_rmse','std'), total_seconds=('total_seconds','median'))
display_table(ihdp_summary.sort_values('cate_rmse'), 'ihdp_summary', CFG.GRAPH_DIR)
for column, label in zip(['cate_rmse','total_seconds'],['CATE RMSE','Median local seconds']):
    fig, ax = plt.subplots(figsize=(10,5))
    ordered = ihdp_summary.sort_values(column)
    ax.barh(ordered.index,ordered[column],color=[CFG.COLORS.get(x,'#607d8b') for x in ordered.index])
    ax.set_xlabel(label); ax.set_title('IHDP: ' + label)
    finish_figure(f'ihdp_{column}', CFG.GRAPH_DIR)
display(Markdown('**Interpretation:** read each scenario separately. These small replication counts '
    'support demonstration and debugging, not a universal winner or a precise speedup claim. '
    'Pretraining cost is excluded; local checkpoint loading and context preparation are included.'))

This code snippet analyzes and visualizes the accuracy benchmark results specifically for the IHDP dataset.

First, it filters the `accuracy_metrics` DataFrame to include only results from the IHDP experiments (where `dataset == 'IHDP'`). It then groups these results by method and calculates several aggregate statistics:
*   `cate_rmse`: The mean CATE Root Mean Squared Error.
*   `rmse_sd_across_replications`: The standard deviation of the CATE RMSE across replications, providing a measure of variability in performance.
*   `total_seconds`: The median total execution time.

The resulting aggregated DataFrame is stored in `ihdp_summary`, which is then displayed as an interactive table using `display_table`. The table is sorted by CATE RMSE to easily identify the best-performing methods and saved as a PNG image named "ihdp_summary" in the export directory (`CFG.GRAPH_DIR`).

Next, it generates two bar plots visualizing the performance of different methods on the IHDP dataset:
*   The first plot shows the CATE RMSE for each method, with bars sorted by RMSE value.
*   The second plot shows the median total execution time for each method, also with bars sorted by time.

For both plots, the color of each bar is determined using `CFG.COLORS`, and appropriate labels and titles are added. The plots are saved as PNG images named "ihdp_cate_rmse" and "ihdp_total_seconds" in the export directory using `finish_figure`.

Finally, a Markdown string is displayed to provide context for interpreting the results. It emphasizes that the small replication counts used in the benchmark support demonstration and debugging but do not guarantee a universal winner or precise speedup claims. It also clarifies that pretraining costs are excluded from the timing measurements.

In essence, this code provides a focused analysis of the accuracy benchmark results on the IHDP dataset, summarizing performance metrics and visualizing them through informative bar plots. The accompanying Markdown text helps to interpret the results appropriately given the limitations of the experimental setup.

## 2. Hillstrom: which customers should receive an email?

We now fit every competitor on the same bounded context from randomized or deliberately confounded
training. Validation selects a model once at the predeclared **20% budget**. A separate randomized
test subset evaluates the locked choices. Random targeting and response targeting are decision baselines.
Two full-training classical fits form a separately labeled practical comparison.
Matched models receive the same 512 labeled rows in demo mode. Their common encoder/scaler is fitted
on the full training pool's baseline covariates; this is shared preprocessing, not access to just
512 raw covariate rows.

Every policy receives the **same independent AIPW evaluator** with known propensity 0.5 and out-of-fold
outcome predictions. IPW provides an additional design-based check. Evaluator models do not determine
the candidate policies: their nuisance fits are separate from candidate predictions, and all candidates
use the same evaluation scores. The real data have no observed individual treatment-effect truth.

Primary economic scenario: **40% contribution margin and $0.10 cost per assigned email**. These are
assumptions, not fields in Hillstrom. All revenue effects remain in original dollars.

In [ ]:
marketing = (load_benchmark_results('hillstrom', CFG.OUT) if CFG.REUSE_RESULTS else
             run_hillstrom_benchmark(mode=CFG.MODE, output_dir=CFG.OUT, seed=CFG.SEED, progress=True))
assert marketing['manifest']['config']['mode'] == CFG.MODE
assert marketing['manifest']['config']['seed'] == CFG.SEED
display_table(marketing['selection'], 'validation_selection', CFG.GRAPH_DIR)
display_table(marketing['metrics'], 'marketing_model_fits', CFG.GRAPH_DIR)

This code snippet loads or runs the Hillstrom benchmark and then displays key results related to policy selection and model performance.

First, it checks if `CFG.REUSE_RESULTS` is True. If so, it attempts to load previously generated benchmark results from the output directory (`CFG.OUT`) using the `load_benchmark_results` function. This avoids re-running the potentially time-consuming benchmark if results are already available.

If `CFG.REUSE_RESULTS` is False (or if loading fails), it runs the Hillstrom benchmark using the `run_hillstrom_benchmark` function, specifying the mode, output directory, seed, and progress indicator from the configuration (`CFG`).

The result of either loading or running the benchmark is stored in the `marketing` variable. Two assertions verify that the loaded or run benchmark matches the current configuration:
*   The mode used in the benchmark must match `CFG.MODE`.
*   The seed used in the benchmark must match `CFG.SEED`.

Finally, it displays two DataFrames as interactive tables using `display_table`:
*   `marketing['selection']`: This DataFrame contains information about the selected policy for each regime (observational vs. randomized), including the chosen method and validation performance. The table is saved as a PNG image named "validation_selection" in the export directory (`CFG.GRAPH_DIR`).
*   `marketing['metrics']`: This DataFrame contains detailed metrics on model performance, including estimates of treatment effects, standard errors, and confidence intervals. The table is saved as a PNG image named "marketing_model_fits" in the export directory (`CFG.GRAPH_DIR`).

In essence, this code provides a summary of the Hillstrom benchmark results, focusing on policy selection and model performance metrics. These tables offer insights into which models perform best under different conditions and how well they generalize to unseen data.

In [ ]:
primary = marketing['policies'].loc[np.isclose(marketing['policies']['budget_fraction'], CFG.PRIMARY_BUDGET_FRACTION)].copy()
display_table(primary[['regime','method','track','treated','revenue_total','cost_total','profit_total',
                 'profit_total_ci_low','profit_total_ci_high','roi','selected_on_validation']], 'primary_budget_policies', CFG.GRAPH_DIR)
display_table(marketing['paired'], 'paired_policy_comparisons', CFG.GRAPH_DIR)
display(Markdown('Intervals concern the frozen policies evaluated on this holdout. Paired comparisons '
    'use the same people. They do not quantify all model-selection or training variation. '
    'Negative value and uncertainty crossing zero remain visible.'))

This code snippet focuses on analyzing the performance of policies selected based on a primary budget fraction in the Hillstrom benchmark, along with paired policy comparisons.

First, it filters the `marketing['policies']` DataFrame to select only those rows where the `budget_fraction` is close to the value specified in `CFG.PRIMARY_BUDGET_FRACTION`. The `.copy()` method creates a new DataFrame to avoid modifying the original data. This filtered DataFrame is stored in the `primary` variable.

Next, it displays a subset of columns from the `primary` DataFrame as an interactive table using `display_table`. The selected columns include:
*   `regime`: The training regime (e.g., randomized or observational).
*   `method`: The causal inference method used.
*   `track`: Indicates whether the model was trained on full data or a subset.
*   `treated`: The number of individuals treated by the policy.
*   `revenue_total`: The total revenue generated by the policy.
*   `cost_total`: The total cost of implementing the policy.
*   `profit_total`: The total profit generated by the policy.
*   `profit_total_ci_low`: The lower bound of the confidence interval for the total profit.
*   `profit_total_ci_high`: The upper bound of the confidence interval for the total profit.
*   `roi`: The return on investment (ROI) of the policy.
*   `selected_on_validation`: A boolean indicating whether this policy was selected based on validation performance.

The table is saved as a PNG image named "primary_budget_policies" in the export directory (`CFG.GRAPH_DIR`).

Then, it displays the `marketing['paired']` DataFrame as an interactive table using `display_table`. This DataFrame contains results from paired policy comparisons, showing the difference in profit between different policies applied to the same individuals. The table is saved as a PNG image named "paired_policy_comparisons" in the export directory (`CFG.GRAPH_DIR`).

Finally, it displays a Markdown string providing context for interpreting the results. It emphasizes that the confidence intervals apply only to the frozen policies evaluated on the holdout set and that paired comparisons use the same individuals. It also notes that negative values and uncertainty crossing zero remain visible, indicating potential non-significant effects.

In essence, this code snippet provides a detailed analysis of the selected policies based on the primary budget fraction and compares their performance using paired comparisons. The accompanying Markdown text helps to interpret the results appropriately given the limitations of the experimental setup.

In [ ]:
for regime, frame in marketing['policies'].groupby('regime',sort=False):
    fig, ax = plt.subplots(figsize=(11,5.5))
    for method, values in frame.groupby('method',sort=False):
        if values['track'].iloc[0] == 'Practical full data': continue
        ax.plot(values['budget_fraction'],values['profit_per_person'],marker='.',
                label=method,color=CFG.COLORS.get(method),alpha=.9)
    ax.axhline(0,color='black',lw=.8)
    ax.set(title=regime,xlabel='Budget as fraction of full contact cost',ylabel='Incremental profit per eligible customer ($)')
    ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=8)
    finish_figure(f'profit_by_budget_{regime}', CFG.GRAPH_DIR)

This code snippet generates a series of plots visualizing the relationship between budget fraction and incremental profit per eligible customer for different causal inference methods within each training regime in the Hillstrom benchmark.

It iterates through each unique `regime` (e.g., randomized, observational) in the `marketing['policies']` DataFrame:
*   For each regime, it creates a new plot with specified dimensions.
*   Then, it iterates through each unique `method` within that regime. It skips methods labeled as 'Practical full data'.
*   For each method, it plots the incremental profit per eligible customer (y-axis) against the budget fraction (x-axis). The points are connected by lines, and each line is styled with a marker, color from `CFG.COLORS`, and transparency.
*   A horizontal line at y=0 is added to indicate zero profit.
*   The plot is titled with the regime name, and labels are added to the axes. A legend is included to identify each method.
*   Finally, the plot is saved as a PNG image named "profit_by_budget_{regime}" in the export directory using `finish_figure`.

In essence, this code generates a series of plots that illustrate how the profitability of different causal inference methods varies with the budget allocated to treatment. These plots provide insights into which methods are most effective at maximizing profit under different budgetary constraints and within different training regimes. The exclusion of 'Practical full data' methods allows for focusing on the performance of models trained specifically within the context of limited data.

In [ ]:
for regime,frame in marketing['gains'].groupby('regime',sort=False):
    fig, ax = plt.subplots(figsize=(10,5.5))
    for method,values in frame.groupby('method',sort=False):
        if '[full data]' in method: continue
        ax.plot(values['q'],values['centered_gain'],label=method,color=CFG.COLORS.get(method))
    ax.axhline(0,color='black',lw=.8)
    ax.set(title=regime,xlabel='Top fraction of ranking',ylabel='Centered revenue gain ($ per eligible customer)')
    ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=8)
    finish_figure(f'centered_gain_{regime}', CFG.GRAPH_DIR)
chosen = marketing['selection'].iloc[0]
chosen_regime, chosen_model = chosen['regime'], chosen['best_candidate_model']
calibration = marketing['calibration'].query('regime == @chosen_regime and method == @chosen_model')
display_table(calibration, 'uplift_calibration', CFG.GRAPH_DIR)

This code snippet analyzes and visualizes the gain curves for different causal inference methods within each training regime in the Hillstrom benchmark, and then displays calibration results for the selected policy.

First, it iterates through each unique `regime` (e.g., randomized, observational) in the `marketing['gains']` DataFrame:
*   For each regime, it creates a new plot with specified dimensions.
*   Then, it iterates through each unique `method` within that regime. It skips methods containing "[full data]" in their name to focus on models trained specifically for this analysis.
*   For each method, it plots the centered gain (y-axis) against the fraction of individuals ranked by predicted treatment effect (x-axis). The lines are styled with colors from `CFG.COLORS`.
*   A horizontal line at y=0 is added to indicate zero gain.
*   The plot is titled with the regime name, and labels are added to the axes. A legend is included to identify each method.
*   Finally, the plot is saved as a PNG image named "centered_gain_{regime}" in the export directory using `finish_figure`.

These plots visualize how much incremental revenue can be achieved by targeting individuals with higher predicted treatment effects, and how this gain changes as more of the population is targeted. The centered gain metric helps to compare performance across different methods by subtracting the average gain from each point on the curve.

Next, it retrieves information about the selected policy (the best-performing method within a specific regime) from the `marketing['selection']` DataFrame. It then filters the `marketing['calibration']` DataFrame to select calibration results for that specific regime and method. Finally, it displays this calibration table as an interactive table using `display_table`, saved as "uplift_calibration" in the export directory (`CFG.GRAPH_DIR`). This table shows how well the predicted treatment effects align with observed outcomes across different bins of predicted values.

In essence, this code snippet provides insights into both the overall profitability and calibration of the selected policies, helping to assess their reliability and potential for real-world application.

In [ ]:
display_table(marketing['ate'], 'population_ate', CFG.GRAPH_DIR)
display_table(marketing['diagnostics'], 'marketing_diagnostics', CFG.GRAPH_DIR)
display(Markdown('These ATE rows use their stated training regime and propensity specification. '
    'They are population-effect diagnostics, not individual-uplift estimates. A real RCT estimate '
    'has sampling uncertainty and is not exact ground truth.'))

This code snippet displays two summary tables related to the Hillstrom benchmark: one showing Average Treatment Effect (ATE) estimates and another providing diagnostic statistics.

First, it displays the `marketing['ate']` DataFrame as an interactive table using `display_table`. This DataFrame contains ATE estimates for different methods and regimes, along with associated metrics like bias and coverage. The table is saved as a PNG image named "population_ate" in the export directory (`CFG.GRAPH_DIR`).

Next, it displays the `marketing['diagnostics']` DataFrame as an interactive table using `display_table`. This DataFrame contains diagnostic statistics related to propensity score estimation and observational selection, such as effective sample size, maximum weight, and propensity score ranges. The table is saved as a PNG image named "marketing_diagnostics" in the export directory (`CFG.GRAPH_DIR`).

Finally, it displays a Markdown string providing context for interpreting the ATE estimates. It emphasizes that these are population-level effects based on the specified training regime and propensity model, not individual uplift estimates. It also clarifies that even a "real RCT estimate" is subject to sampling uncertainty and should not be considered exact ground truth.

In essence, this code snippet provides a comprehensive overview of the ATE estimates and diagnostic statistics from the Hillstrom benchmark, helping to assess the validity and reliability of the results. The accompanying Markdown text helps to interpret these results appropriately given the limitations of observational data and causal inference methods.

## 3. Hillstrom: when is targeting profitable under a budget?

In [ ]:
# A small exact example: these values are illustrative, not Hillstrom estimates.
example_revenue = np.array([70.,120.,150.])
example_cost = np.array([10.,20.,30.])
allocation = budget_policy(example_revenue, example_cost, budget=50, margin=1.)
greedy = np.array([True,True,False])
display_table(pd.DataFrame({'incremental_revenue':example_revenue, 'cost':example_cost,
                     'exact_selection':allocation,'greedy_selection':greedy}), 'knapsack_example', CFG.GRAPH_DIR)
print('Exact net profit:', np.sum(allocation*(example_revenue-example_cost)))
print('Ratio-greedy net profit:', np.sum(greedy*(example_revenue-example_cost)))
assert np.sum(allocation*example_cost) <= 50

This code snippet provides a small, illustrative example of how the `budget_policy` function works to optimize treatment allocation under a budget constraint.

First, it defines two NumPy arrays: `example_revenue` and `example_cost`, representing the potential revenue and cost associated with treating each individual.

Then, it calls the `budget_policy` function with these values, specifying a total budget of 50 and a margin of 1. The `budget_policy` function determines which individuals should be treated to maximize profit within the given budget constraint. The result is stored in the `allocation` variable, which is a boolean array indicating whether each individual should be treated (True) or not (False).

Next, it creates a NumPy array called `greedy`, representing a simple greedy allocation strategy where the two individuals with the highest revenue are selected.

It then displays a Pandas DataFrame as an interactive table using `display_table`. The DataFrame contains the incremental revenue, cost, exact selection (from `budget_policy`), and greedy selection for each individual. The table is saved as a PNG image named "knapsack_example" in the export directory (`CFG.GRAPH_DIR`).

Finally, it calculates and prints the net profit achieved by both the optimal allocation (determined by `budget_policy`) and the greedy allocation strategy. An assertion verifies that the total cost of the optimal allocation does not exceed the budget constraint.

In essence, this code snippet demonstrates how the `budget_policy` function can be used to optimize treatment allocation under a budget constraint, and it compares its performance to a simple greedy approach. The example provides a clear illustration of the benefits of using an optimization algorithm like `budget_policy` over a naive strategy.

In [ ]:
# Revalue the SAME locked policies under alternative economics; do not retarget from test outcomes.
fixed = primary.query("regime == 'Randomized' and track == 'Matched data'")
cost_grid = np.linspace(*CFG.CONTACT_COST_GRID)
plt.figure(figsize=(10,5))
for _, row in fixed.iterrows():
    profits = (CFG.MARGIN*row.revenue_total - cost_grid*row.treated)/row['n']
    plt.plot(cost_grid,profits,label=row['method'],color=CFG.COLORS.get(row['method']))
plt.axhline(0,color='black',lw=.8); plt.xlabel('Assumed cost per contact ($)')
plt.ylabel('Revalued incremental profit per eligible customer ($)')
plt.title(f'Economic sensitivity of fixed policies at {CFG.PRIMARY_BUDGET_FRACTION:.0%} capacity')
plt.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=9)
finish_figure('contact_cost_sensitivity', CFG.GRAPH_DIR)
display(Markdown('Contact counts are held fixed in this sensitivity plot, so the dollar budget changes '
    'with the assumed unit cost. It is not a reoptimized fixed-dollar-budget experiment.'))

This code snippet performs a sensitivity analysis by evaluating the profitability of previously selected policies under different assumptions about the cost per contact.

First, it filters the `primary` DataFrame to select only those rows representing policies that were trained on randomized data and used matched data for fitting. This ensures that the policies are based on a reliable causal estimate. The result is stored in the `fixed` variable.

Then, it creates a plot showing how the incremental profit per eligible customer changes as the cost per contact varies across a range of values defined by `CFG.CONTACT_COST_GRID`. For each policy in the `fixed` DataFrame:
*   It calculates the incremental profit for each cost value using the formula: `(CFG.MARGIN * row.revenue_total - cost_grid * row.treated) / row['n']`, where `row.revenue_total` is the total revenue generated by the policy, `row.treated` is the number of individuals treated, and `row['n']` is the total number of eligible customers.
*   It plots the incremental profit against the cost per contact, using a color from `CFG.COLORS` to identify each method.

The plot includes a horizontal line at y=0 to indicate zero profit, labels for the axes, and a title indicating the capacity level used for policy selection. A legend is added to identify each method. The plot is saved as a PNG image named "contact_cost_sensitivity" in the export directory using `finish_figure`.

Finally, it displays a Markdown string clarifying that the contact counts are held fixed in this sensitivity analysis, meaning that the dollar budget changes with the assumed unit cost. It emphasizes that this is not a reoptimized fixed-dollar-budget experiment but rather an evaluation of how sensitive the profitability of existing policies is to changes in the cost per contact.

In essence, this code snippet provides insights into the robustness of the selected policies to variations in the cost per contact. It helps to understand how much the profitability of these policies might change if the cost of reaching customers were to increase or decrease.

### Hillstrom treatment versions after policy selection

In [ ]:
display_table(raw_hillstrom.groupby('segment').agg(customers=('spend','size'),mean_spend=('spend','mean'),
    visit_rate=('visit','mean'),conversion_rate=('conversion','mean')), 'treatment_version_summary', CFG.GRAPH_DIR)
display(Markdown('These full-source descriptive treatment-version results are shown after the locked '
    'primary analysis. They did not select the primary arm, model, budget, or policy.'))

This code snippet displays a summary of key characteristics for each customer segment in the raw Hillstrom dataset *after* the main causal inference analyses have been completed.

It groups the `raw_hillstrom` DataFrame by 'segment' (Mens E-Mail, Womens E-Mail, No E-Mail) and calculates several aggregate statistics:
*   `customers`: The number of customers in each segment (using `.size`).
*   `mean_spend`: The average spend per customer in each segment.
*   `visit_rate`: The average visit rate per customer in each segment.
*   `conversion_rate`: The average conversion rate per customer in each segment.

The resulting aggregated DataFrame is displayed as an interactive table using `display_table`. The table is saved as a PNG image named "treatment_version_summary" in the export directory (`CFG.GRAPH_DIR`).

Finally, it displays a Markdown string emphasizing that these descriptive results are presented *after* the primary causal analysis and were not used to select the primary treatment arm, model, budget, or policy. This clarifies that this is purely exploratory information about the characteristics of the different customer segments in the original dataset.

In essence, this code snippet provides a final overview of the raw data distribution across customer segments, offering context for interpreting the results of the causal inference analyses without influencing those results directly.

## 4. Simulation: can adjustment recover effects when one nuisance model is wrong?


In [ ]:
dr_demo = run_double_robustness_experiment(n=CFG.DR_N, repetitions=CFG.WORKLOADS[CFG.MODE]['dr_repetitions'], seed=CFG.SEED)
save_tables(dr_demo, 'double_robustness', CFG.OUT)
display_table(dr_demo['summary'][['outcome_spec','propensity_spec','method','bias','rmse','repetitions','bias_mcse']], 'double_robustness_summary', CFG.GRAPH_DIR)
for method in ['Outcome regression','IPW','AIPW']:
    fig, ax = plt.subplots(figsize=(6,4.5))
    table = dr_demo['summary'].query('method == @method').pivot(index='outcome_spec',columns='propensity_spec',values='bias')
    sns.heatmap(table,annot=True,fmt='.3f',center=0,cmap='vlag',ax=ax,cbar=False)
    ax.set_title(method+' bias'); ax.set_xlabel('Propensity model'); ax.set_ylabel('Outcome model')
    finish_figure(f'double_robustness_{method}', CFG.GRAPH_DIR)

This code snippet runs a double robustness experiment to assess the sensitivity of causal effect estimates to misspecification of both the outcome and propensity models, then visualizes the results.

First, it calls `run_double_robustness_experiment` with parameters specified in the configuration (`CFG`). This function simulates data under different scenarios (correctly specified vs. incorrectly specified outcome and propensity models) and estimates causal effects using various methods. The number of samples (`n`) and repetitions are determined by the current mode (`CFG.MODE`).

The results of the experiment, including detailed metrics for each scenario and method, are stored in the `dr_demo` variable.

Next, it saves the tables contained within `dr_demo` to CSV files in the output directory (`CFG.OUT`) using the `save_tables` function.

Then, it displays a summary of the experiment results as an interactive table using `display_table`. The displayed columns include:
*   `outcome_spec`: Whether the outcome model was correctly specified.
*   `propensity_spec`: Whether the propensity model was correctly specified.
*   `method`: The estimation method used (e.g., Outcome regression, IPW, AIPW).
*   `bias`: The bias of the estimated causal effect.
*   `rmse`: The Root Mean Squared Error of the estimated causal effect.
*   `repetitions`: The number of replications used for each scenario.
*   `bias_mcse`: The Monte Carlo standard error of the bias estimate.

The table is saved as a PNG image named "double_robustness_summary" in the export directory (`CFG.GRAPH_DIR`).

Finally, it generates heatmaps visualizing the bias of different estimation methods under various combinations of outcome and propensity model specifications:
*   For each method (Outcome regression, IPW, AIPW), it creates a heatmap showing the bias as a function of both `outcome_spec` and `propensity_spec`. The heatmap uses color to represent the magnitude of the bias, with 0 representing no bias.
*   The heatmaps are saved as PNG images named "double_robustness_{method}" in the export directory using `finish_figure`.

In essence, this code snippet provides a comprehensive assessment of the double robustness property of different causal inference methods. The results demonstrate how well each method performs when either the outcome or propensity model is misspecified, and the heatmaps provide a clear visualization of these effects.

In [ ]:
coverage = dr_demo['summary'].query("method == 'AIPW'")[['outcome_spec','propensity_spec','coverage','coverage_mcse','repetitions']]
display_table(coverage, 'aipw_coverage', CFG.GRAPH_DIR)
display(Markdown('Coverage is empirical over repeated generated cohorts. With one nuisance misspecified, '
    'point consistency can persist while a simple influence-score variance may omit nuisance-estimation terms. '
    'The few-replication demo is deliberately not a calibration guarantee.'))

This code snippet focuses on analyzing the coverage of confidence intervals for the AIPW (Augmented Inverse Probability Weighting) estimator in the double robustness experiment.

First, it filters the `dr_demo['summary']` DataFrame to select only rows where the method is 'AIPW'. It then selects a subset of columns:
*   `outcome_spec`: Whether the outcome model was correctly specified.
*   `propensity_spec`: Whether the propensity model was correctly specified.
*   `coverage`: The empirical coverage probability of the confidence interval (the proportion of times the true causal effect falls within the estimated confidence interval).
*   `coverage_mcse`: The Monte Carlo standard error of the coverage estimate.
*   `repetitions`: The number of replications used for each scenario.

The resulting DataFrame is displayed as an interactive table using `display_table`. The table is saved as a PNG image named "aipw_coverage" in the export directory (`CFG.GRAPH_DIR`).

Finally, it displays a Markdown string providing important caveats about interpreting the coverage results. It emphasizes that:
*   The coverage estimates are empirical and based on repeated simulations.
*   Even when one of the nuisance models (outcome or propensity) is misspecified, point consistency can still hold (meaning the estimator converges to the true value as the sample size increases). However, a simple influence-score variance calculation may not capture all sources of uncertainty due to nuisance estimation.
*   The small number of replications used in this demonstration is deliberately not intended to provide a rigorous calibration guarantee.

In essence, this code snippet provides insights into the coverage properties of AIPW under different model specifications and highlights the importance of considering potential limitations when interpreting confidence intervals based on limited simulations. The accompanying Markdown text emphasizes that these results should be viewed as illustrative rather than definitive statements about the reliability of the estimator.

## 5. Simulation: how fragile are the estimates to confounding and poor overlap?

In [ ]:
sweeps = run_sensitivity_sweeps(n=CFG.SENSITIVITY_N, repetitions=CFG.WORKLOADS[CFG.MODE]['sensitivity_repetitions'], seed=CFG.SENSITIVITY_SEED)
save_tables(sweeps,'sensitivity',CFG.OUT)
for experiment in ['Overlap','Hidden confounding']:
    fig, ax = plt.subplots(figsize=(10,5))
    frame = sweeps['summary'].query('experiment == @experiment')
    for method, values in frame.groupby('method',sort=False):
        ax.plot(values['strength'],values['bias'],marker='o',label=method)
    ax.axhline(0,color='black',ls='--',lw=.8)
    ax.set(title=experiment,xlabel='Stress strength',ylabel='ATE bias (true population ATE = 1)')
    ax.legend(fontsize=8)
    finish_figure(f'identification_sensitivity_{experiment}',CFG.GRAPH_DIR)
display_table(sweeps['diagnostics'].groupby(['experiment','strength'])[['ess_control','ess_treated','max_weight']].mean(), 'overlap_diagnostics', CFG.GRAPH_DIR)
display_table(sweeps['summary'].query("experiment == 'Clipping'")[['strength','method','bias','rmse']], 'clipping_sensitivity', CFG.GRAPH_DIR)

This code snippet runs a series of sensitivity analyses to assess the robustness of causal effect estimates to different types of confounding and data quality issues, then visualizes the results.

First, it calls `run_sensitivity_sweeps` with parameters specified in the configuration (`CFG`). This function simulates data under various scenarios, including varying degrees of overlap between treatment groups, hidden confounding, and propensity score clipping. It estimates causal effects using different methods for each scenario. The number of samples (`n`) and repetitions are determined by the current mode (`CFG.MODE`).

The results of the experiment, including detailed metrics and diagnostic statistics, are stored in the `sweeps` variable.

Next, it saves the tables contained within `sweeps` to CSV files in the output directory (`CFG.OUT`) using the `save_tables` function.

Then, it generates plots visualizing the bias of different estimation methods under varying levels of overlap and hidden confounding:
*   For each experiment ('Overlap' and 'Hidden confounding'), it creates a plot showing the bias as a function of the stress strength (the level of confounding). Each line on the plot represents a different estimation method.
*   The plots include a horizontal line at y=0 to indicate zero bias, labels for the axes, and a title indicating the experiment being visualized. A legend is added to identify each method.
*   The plots are saved as PNG images named "identification_sensitivity_{experiment}" in the export directory using `finish_figure`.

Next, it displays a summary of diagnostic statistics related to propensity score weighting (effective sample size for control and treated groups, maximum weight) grouped by experiment and stress strength. The table is saved as a PNG image named "overlap_diagnostics" in the export directory (`CFG.GRAPH_DIR`).

Finally, it displays a table summarizing the results of the clipping sensitivity analysis, showing the bias and RMSE for different methods under varying levels of propensity score clipping. The table is saved as a PNG image named "clipping_sensitivity" in the export directory (`CFG.GRAPH_DIR`).

In essence, this code snippet provides a comprehensive assessment of the robustness of causal effect estimates to various types of sensitivity threats. The plots and tables help to identify which methods are most robust under different conditions and provide insights into the potential impact of confounding and data quality issues on causal inference results.

In [ ]:
# Constant-effect additive model: formal OLS sensitivity has a causal ATE interpretation
# here under the stated identification assumptions. This is a synthetic tutorial example.
rng = np.random.default_rng(CFG.FORMAL_SEED)
sx = rng.normal(size=(CFG.FORMAL_N,3))
st = rng.binomial(1,expit(.6*sx[:,0]-.4*sx[:,1]))
sy = 1 + sx @ np.array([1.,.7,-.4]) + st + rng.normal(size=len(st))
formal = linear_sensitivity_analysis(sx,st,sy,feature_names=['history','engagement','tenure'])
display_table(pd.Series({k:v for k,v in formal.items() if not isinstance(v,pd.DataFrame)}), 'formal_sensitivity_summary', CFG.GRAPH_DIR)
display_table(formal['benchmarks'], 'formal_sensitivity_benchmarks', CFG.GRAPH_DIR)
save_tables(formal,'formal_sensitivity',CFG.OUT)
grid = formal['contours'].pivot(index='r2_outcome',columns='r2_treatment',values='adjusted_estimate')
fig,ax = plt.subplots(figsize=(8,5))
surface = ax.contourf(grid.columns,grid.index,grid.values,levels=15,cmap='vlag')
ax.contour(grid.columns,grid.index,grid.values,levels=[0],colors='black',linewidths=1.5)
bench = formal['benchmarks']
ax.scatter(bench['r2_treatment'],bench['r2_outcome'],color='black',s=25)
for _, row in bench.iterrows(): ax.annotate(row['feature'],(row.r2_treatment,row.r2_outcome),xytext=(5,4),textcoords='offset points')
ax.set(xlabel='Partial R²: omitted U with treatment given X',ylabel='Partial R²: omitted U with outcome given T,X',
       title='Formal sensitivity of the additive OLS treatment coefficient')
fig.colorbar(surface,ax=ax,label='Adversarially adjusted coefficient')
finish_figure('formal_sensitivity',CFG.GRAPH_DIR)

This code snippet performs a formal sensitivity analysis to assess the robustness of an ordinary least squares (OLS) estimate of the treatment effect in a simplified causal model.

First, it generates synthetic data with a known causal structure: `sy = 1 + sx @ np.array([1.,.7,-.4]) + st + rng.normal(size=len(st))`. This equation represents an additive model where the outcome (sy) is determined by covariates (sx), treatment (st), and random noise.

Then, it calls the `linear_sensitivity_analysis` function with the synthetic data to estimate the sensitivity of the OLS coefficient for the treatment effect to omitted variable bias. The function calculates the maximum potential bias due to unobserved confounders that are correlated with both the treatment and outcome. It also provides benchmark values based on observed covariate partial R²s.

Next, it displays a summary of the results as an interactive table using `display_table`. This table includes key metrics such as the estimated coefficient, standard error, robustness value (the amount of shared variance needed to reduce the coefficient to zero), and other relevant statistics. The table is saved as a PNG image named "formal_sensitivity_summary" in the export directory (`CFG.GRAPH_DIR`).

It also displays the benchmark values as an interactive table using `display_table`. This table shows the partial R²s for each covariate, providing context for interpreting the sensitivity analysis results. The table is saved as a PNG image named "formal_sensitivity_benchmarks" in the export directory (`CFG.GRAPH_DIR`).

The function's full output is also saved to CSV files using `save_tables`.

Finally, it generates a contour plot visualizing the adjusted coefficient under different levels of omitted variable bias. The x-axis represents the partial R² between an unobserved confounder and the treatment, while the y-axis represents the partial R² between the confounder and the outcome. The color of each point on the plot indicates the adjusted coefficient value. A contour line at zero is added to show where the coefficient would be reduced to zero by omitted variable bias. Benchmark values are plotted as scatter points with annotations indicating which covariate they represent. This plot provides a visual representation of how sensitive the OLS estimate is to unobserved confounding. The plot is saved as a PNG image named "formal_sensitivity" in the export directory using `finish_figure`.

In essence, this code snippet provides a rigorous assessment of the sensitivity of an OLS estimate to omitted variable bias under specific assumptions about the causal model. The results highlight the importance of considering potential confounding when interpreting observational data and provide insights into how much unobserved confounding would be needed to invalidate the estimated treatment effect.

In [ ]:
if CFG.REUSE_RESULTS:
    stress_manifest = json.loads((CFG.OUT/'stress_manifest.json').read_text())
    assert stress_manifest['status'] == 'complete'
    assert stress_manifest['config']['mode'] == CFG.MODE and stress_manifest['config']['seed'] == CFG.SEED
    assert stress_manifest['config'].get('foundation_device') == CFG.DEVICE, 'Rerun with CFG.REUSE_RESULTS = False for this device'
    model_stress = {key: pd.read_csv(CFG.OUT/f'stress_{key}.csv')
                    for key in ['metrics','intervals','interference_models']}
    model_stress['manifest'] = stress_manifest
else:
    model_stress = run_model_stress(mode=CFG.MODE, seed=CFG.SEED, output_dir=CFG.OUT)
display_table(model_stress['metrics'].groupby(['scenario','model']).agg(
    cate_rmse=('cate_rmse','mean'),ate_bias=('ate_bias','mean')), 'foundation_model_stress_summary', CFG.GRAPH_DIR)
for metric in ['cate_rmse','ate_bias']:
    fig, ax = plt.subplots(figsize=(10,5.5))
    sns.barplot(data=model_stress['metrics'],x='scenario',y=metric,hue='model',palette=CFG.COLORS,errorbar=None,ax=ax)
    label = {'cate_rmse':'CATE RMSE','ate_bias':'ATE bias'}[metric]
    ax.set_title(label); ax.set_ylabel(label); ax.tick_params(axis='x',rotation=15)
    ax.axhline(0,color='black',lw=.8)
    finish_figure(f'foundation_model_stress_{metric}',CFG.GRAPH_DIR)

This code snippet loads or runs the foundation model stress test and then visualizes the results.

First, it checks if `CFG.REUSE_RESULTS` is True. If so, it attempts to load previously generated results from files in the output directory (`CFG.OUT`). It verifies that the loaded manifest file indicates a complete run with the correct mode, seed, and device. The metrics, intervals, and interference models are then loaded from their respective CSV files.

If `CFG.REUSE_RESULTS` is False (or if loading fails), it runs the foundation model stress test using the `run_model_stress` function, specifying the mode, seed, and output directory from the configuration (`CFG`).

The results of either loading or running the stress test are stored in the `model_stress` variable.

Next, it displays a summary table showing the mean CATE RMSE and ATE bias for each model within each scenario. The table is saved as a PNG image named "foundation_model_stress_summary" in the export directory (`CFG.GRAPH_DIR`).

Then, it generates bar plots visualizing the performance of different models for each metric (CATE RMSE and ATE bias). For each metric:
*   It creates a bar plot showing the mean value of the metric for each model within each scenario. The bars are colored using `CFG.COLORS`.
*   A horizontal line at y=0 is added to indicate zero bias or error.
*   The plot is titled with the metric name, and labels are added to the axes.
*   The plot is saved as a PNG image named "foundation_model_stress_{metric}" in the export directory using `finish_figure`.

In essence, this code snippet provides a comprehensive overview of the performance of different foundation models under various stress test scenarios. The summary table and bar plots help to compare the robustness and accuracy of these models and identify potential areas for improvement.

## 6. Simulation: do native uncertainty intervals contain the true effects?

In [ ]:
display_table(model_stress['intervals'], 'native_interval_fits', CFG.GRAPH_DIR)
interval_summary = model_stress['intervals'].groupby(['scenario','model']).agg(
    empirical_containment=('coverage','mean'),mean_width=('mean_width','mean'),fits=('coverage','size'))
display_table(interval_summary, 'native_interval_summary', CFG.GRAPH_DIR)
interval_summary.to_csv(CFG.OUT/'native_interval_summary.csv')

## 7. Customer networks: what changes when customers affect one another?

In [ ]:
interference = run_interference_experiment(n_clusters=CFG.WORKLOADS[CFG.MODE]['interference_clusters'], cluster_size=CFG.INTERFERENCE_CLUSTER_SIZE,
    spillovers=CFG.INTERFERENCE_SPILLOVERS, interaction=CFG.INTERFERENCE_INTERACTION,seed=CFG.SEED)
save_tables(interference,'interference',CFG.OUT)
display_table(interference['exposure_diagnostics'], 'interference_exposure_diagnostics', CFG.GRAPH_DIR)
display_table(interference['contrasts'], 'interference_contrasts', CFG.GRAPH_DIR)

This code snippet runs an experiment to evaluate the performance of causal inference methods in a setting with partial interference, where the treatment effect on one individual can be influenced by the treatment status of their peers within a cluster.

First, it calls `run_interference_experiment` with parameters specified in the configuration (`CFG`). This function simulates data with varying degrees of spillover effects and interaction effects, estimates causal contrasts using different methods, and collects diagnostic statistics. The number of clusters, cluster size, spillovers, interaction strength, and seed are determined by the current mode (`CFG.MODE`) and other configuration settings.

The results of the experiment, including exposure diagnostics, causal contrast estimates, and data summaries, are stored in the `interference` variable.

Next, it saves the tables contained within `interference` to CSV files in the output directory (`CFG.OUT`) using the `save_tables` function.

Then, it displays two summary tables:
*   `interference['exposure_diagnostics']`: This DataFrame contains diagnostic statistics related to exposure probabilities and cluster characteristics. The table is saved as a PNG image named "interference_exposure_diagnostics" in the export directory (`CFG.GRAPH_DIR`).
*   `interference['contrasts']`: This DataFrame contains estimates of direct, spillover, and total effects under different scenarios. The table is saved as a PNG image named "interference_contrasts" in the export directory (`CFG.GRAPH_DIR`).

In essence, this code snippet provides an assessment of how well causal inference methods perform in settings with partial interference. The exposure diagnostics help to understand the characteristics of the simulated data, while the contrast estimates provide insights into the magnitude and direction of direct and spillover effects.

In [ ]:
contrast_rows = interference['contrasts'].query('spillover == 1.5').copy()
plot_intervals(contrast_rows,'contrast','hajek_estimate','hajek_ci_low','hajek_ci_high',
               'Positive spillovers: exposure-aware estimates and simulator truth',truth='truth')
finish_figure('interference_effects',CFG.GRAPH_DIR)
for spillover,frame in interference['policy_values'].groupby('spillover',sort=False):
    fig, ax = plt.subplots(figsize=(10,5))
    ax.plot(frame.saturation,frame.truth_increment,label='Simulator total truth',color='black',lw=2)
    ax.plot(frame.saturation,frame.naive_additive_increment,label='Naive additive forecast',ls='--',color='#b85a18')
    ax.plot(frame.saturation,frame.hajek_estimate,label='Exposure-aware estimate',color='#2b598b')
    ax.fill_between(frame.saturation,frame.hajek_ci_low,frame.hajek_ci_high,color='#2b598b',alpha=.15)
    ax.axhline(0,color='black',lw=.5)
    ax.set(title=f'Spillover parameter {spillover:g}',xlabel='Target saturation',ylabel='Incremental outcome per member')
    ax.legend(fontsize=8)
    spillover_label = f'{spillover:g}'.replace('-', 'minus_')
    finish_figure(f'interference_policy_value_spillover_{spillover_label}',CFG.GRAPH_DIR)

In [ ]:
rollout = model_stress['interference_models']
display_table(rollout.groupby(['spillover','model'])[['predicted_full_rollout_increment','truth_full_rollout_increment','error']].mean(), 'rollout_predictions', CFG.GRAPH_DIR)
fig,ax = plt.subplots(figsize=(11,4.8))
sns.barplot(data=rollout,x='spillover',y='predicted_full_rollout_increment',hue='model',palette=CFG.COLORS,errorbar=None,ax=ax)
truths = rollout.groupby('spillover').truth_full_rollout_increment.mean().sort_index()
ax.scatter(np.arange(len(truths)),truths.values,marker='D',s=75,color='black',label='True total rollout effect',zorder=5)
ax.axhline(0,color='black',lw=.7)
ax.set_title('Both foundation models used as deliberate wrong-estimand baselines')
ax.set_ylabel('Incremental outcome per member');ax.legend(fontsize=8,bbox_to_anchor=(1.02,1),loc='upper left')
finish_figure('foundation_models_under_interference',CFG.GRAPH_DIR)

## 8. Hillstrom: which policy would we take to an online experiment?

In [ ]:
display_table(marketing['selection'], 'locked_policy_summary', CFG.GRAPH_DIR)
best_ihdp = ihdp_summary['cate_rmse'].idxmin()
hidden = model_stress['metrics'].groupby(['scenario','model'])['ate_bias'].mean()
display(Markdown(f'**Measured local result:** the lowest mean IHDP CATE RMSE in this run belongs to '
    f'**{best_ihdp}**, across {len(ihdp_results) // ihdp_results.method.nunique()} replications. '
    'This is a scenario-specific descriptive result. The validation selections above determine the '
    'marketing candidates; test outcomes do not select a replacement winner.'))
display(Markdown('**What this demonstration establishes:** real checkpoints run locally; classical and '
    'foundation-model predictions can be compared with consistent effect targets; targeting can be '
    'evaluated independently under a budget; and hidden confounding or interference can invalidate '
    'apparently attractive recommendations. The demo does not establish universal superiority or realized online ROI.'))
assert accuracy_metrics['status'].eq('success').all()
assert {'CausalFM','CausalPFN'}.issubset(set(accuracy_metrics.method))
assert {'CausalFM','CausalPFN'}.issubset(set(marketing['metrics'].method))
assert (marketing['policies'].cost_total <= marketing['policies'].budget + 1e-7).all()
print('All required model runs succeeded; all evaluated policies respected their budgets.')

This code segment focuses on displaying results and performing assertions to verify the integrity of a simulation or experiment.

First, it calls `display_table` with data from the ‘selection’ section of a DataFrame called `marketing`, assigning the filename 'locked_policy_summary' and storing the output in the directory specified by `CFG.GRAPH_DIR`. This likely generates and saves a table summarizing the selected marketing policies.

Next, it identifies the model that achieved the lowest root mean squared error (RMSE) for CATE (Conditional Average Treatment Effect) within the `ihdp_summary` DataFrame, storing the name of this best-performing model in the variable `best_ihdp`. 

The code then calculates the average ATE (Average Treatment Effect) bias across different scenarios and models using data from the `model_stress` DataFrame. The result is stored in a grouped Series called `hidden`.

Following this, two Markdown blocks are displayed. The first communicates the best IHDP model found, along with information about the number of replications used in the analysis. It emphasizes that these results are specific to the scenario and do not determine which marketing candidates are ultimately selected. The second Markdown block outlines the broader goals and limitations of the demonstration: verifying local execution, enabling comparison between different prediction models, evaluating targeting under budget constraints, and highlighting the potential for confounding or interference to affect recommendations. It clarifies that the demo doesn’t prove universal superiority or real-world ROI.

The remaining lines consist of assertions—checks to ensure certain conditions are met. The first assertion verifies that all runs in `accuracy_metrics` completed successfully (status is 'success').  The next three assertions confirm that specific models ('CausalFM' and 'CausalPFN') are present within both the `accuracy_metrics` and `marketing` DataFrames, indicating they were included in the analysis. Finally, an assertion checks that the total cost of all policies in the ‘policies’ section of the `marketing` DataFrame does not exceed their respective budgets (allowing for a small tolerance).

If all assertions pass, the code prints a message confirming successful model runs and budget adherence.